In [44]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/pharaohtut/arc-intentn-annotation/arc_intent_annotation_dataset_v1.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/sample_submission.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/test-challenges/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/arc-agi2-analytics-submission-20260414-145000z-zip/dataset_solver_behavior.json
/kaggle/input/datasets/pharaohtut/arc-agi2-analytics-submission-20260414-145000z-zip/DATASET_README.txt
/kaggle/input/datasets/pharaohtut/arc-agi2-analytics-submission-20260414-145000z-zip/dataset_background_sensitivity.json
/kaggle/input/datasets/pharaohtut/arc-agi2-

In [45]:
# CELL A01
# Structural Grid Features — Analysis Copy (Read-Only)

from collections import deque, Counter

def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else (0, 0)

def infer_background_color(grid):
    flat = [c for row in grid for c in row]
    return Counter(flat).most_common(1)[0][0] if flat else 0

def grid_colors(grid):
    return sorted(set(c for row in grid for c in row))

def connected_components(grid, background):
    h, w = grid_shape(grid)
    visited = [[False]*w for _ in range(h)]
    components = []

    for r in range(h):
        for c in range(w):
            if visited[r][c] or grid[r][c] == background:
                continue
            color = grid[r][c]
            q = deque([(r,c)])
            visited[r][c] = True
            cells = []

            while q:
                cr, cc = q.popleft()
                cells.append((cr,cc))
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc = cr+dr, cc+dc
                    if 0<=nr<h and 0<=nc<w and not visited[nr][nc] and grid[nr][nc]==color:
                        visited[nr][nc] = True
                        q.append((nr,nc))

            rows = [p[0] for p in cells]
            cols = [p[1] for p in cells]
            components.append({
                "color": color,
                "cells": tuple(cells),
                "area": len(cells),
                "bbox": (min(rows), max(rows), min(cols), max(cols)),
            })

    components.sort(key=lambda c: (c["bbox"][0], c["bbox"][2], c["color"]))
    return components

def is_vertical_symmetric(grid):
    h,w = grid_shape(grid)
    return all(grid[r][c]==grid[r][w-1-c] for r in range(h) for c in range(w))

def is_horizontal_symmetric(grid):
    h,w = grid_shape(grid)
    return all(grid[r][c]==grid[h-1-r][c] for r in range(h) for c in range(w))

def is_rot180_symmetric(grid):
    h,w = grid_shape(grid)
    return all(grid[r][c]==grid[h-1-r][w-1-c] for r in range(h) for c in range(w))

def grid_features(grid):
    bg = infer_background_color(grid)
    comps = connected_components(grid, bg)
    return {
        "shape": grid_shape(grid),
        "background": bg,
        "colors": grid_colors(grid),
        "num_colors": len(grid_colors(grid)),
        "num_components": len(comps),
        "components": comps,
        "symmetry": {
            "vertical": is_vertical_symmetric(grid),
            "horizontal": is_horizontal_symmetric(grid),
            "rot180": is_rot180_symmetric(grid),
        },
    }

In [46]:
# CELL A02
# Hypothesis Scoring — Analysis Copy (Read-Only, v1.1)

"""
Structural scoring for solver hypotheses.

INVARIANTS:
- Deterministic
- Purely symbolic
- No learning
- Analysis-only
"""

def score_hypothesis(hypothesis):
    feats = grid_features(hypothesis.grid)
    score = 0.0

    # ------------------------------------------------------------------
    # 1. Component complexity (primary signal)
    # ------------------------------------------------------------------
    score += feats["num_components"] * 5.0

    # ------------------------------------------------------------------
    # 2. Spatial footprint (secondary signal)
    # ------------------------------------------------------------------
    for comp in feats["components"]:
        r0, r1, c0, c1 = comp["bbox"]
        score += (r1 - r0 + 1) * (c1 - c0 + 1) * 0.1

    # ------------------------------------------------------------------
    # 3. Color complexity
    # ------------------------------------------------------------------
    score += feats["num_colors"] * 2.0

    # ------------------------------------------------------------------
    # 4. Symmetry bonuses (weak, but stabilizing)
    # ------------------------------------------------------------------
    if feats["symmetry"]["vertical"]:
        score -= 1.0
    if feats["symmetry"]["horizontal"]:
        score -= 1.0
    if feats["symmetry"]["rot180"]:
        score -= 0.5

    # ------------------------------------------------------------------
    # 5. NEW: Repeated-operator penalty (breaks symmetry)
    # ------------------------------------------------------------------
    history = hypothesis.history
    for i in range(1, len(history)):
        if history[i] == history[i - 1]:
            score += 3.0   # discourage rotate→rotate, etc.

    # ------------------------------------------------------------------
    # 6. NEW: No-op structural penalty
    # ------------------------------------------------------------------
    # If nothing structural changed, penalize lightly
    if feats["num_components"] == grid_features(hypothesis.grid)["num_components"]:
        score += 1.0

    return round(score, 6)

In [47]:
# CELL A02a
# Symbolic Solver Search Core (Analysis-Only)

"""
Defines the symbolic solver search core for the analysis notebook.

INVARIANTS:
- Analysis-only
- Deterministic
- No executors
- No governance
- No submission
"""

from dataclasses import dataclass
from typing import Tuple, List

# -----------------------------------------------------------------------------
# Solver hypothesis container
# -----------------------------------------------------------------------------

@dataclass(frozen=True)
class SolverHypothesis:
    grid: List[List[int]]
    history: Tuple[str, ...]


# -----------------------------------------------------------------------------
# Minimal operator set (analysis-safe)
# -----------------------------------------------------------------------------

def rotate90(grid):
    return [list(row) for row in zip(*grid[::-1])]

def flip_h(grid):
    return [row[::-1] for row in grid]

def normalize_grid(grid):
    return grid


def _solver_operators():
    return [rotate90, flip_h]


# -----------------------------------------------------------------------------
# Expansion + pruning (bounded)
# -----------------------------------------------------------------------------

def expand_and_prune(
    hypotheses: List[SolverHypothesis],
    operators,
    max_hypotheses: int,
):
    expanded = []

    for h in hypotheses:
        for op in operators:
            try:
                new_grid = op(h.grid)
            except Exception:
                continue

            expanded.append(
                SolverHypothesis(
                    grid=new_grid,
                    history=h.history + (op.__name__,),
                )
            )

    if not expanded:
        return hypotheses

    expanded.sort(key=score_hypothesis)
    return expanded[:max_hypotheses]


# -----------------------------------------------------------------------------
# Solver search entry point (analysis-only)
# -----------------------------------------------------------------------------

def run_search(
    input_grid,
    max_steps: int = 2,
    max_hypotheses: int = 16,
):
    """
    Runs a bounded symbolic search.

    Returns:
    {
        "hypotheses": [SolverHypothesis, ...]
    }
    """

    hypotheses = [
        SolverHypothesis(
            grid=normalize_grid(input_grid),
            history=(),
        )
    ]

    operators = _solver_operators()

    for _ in range(max_steps):
        hypotheses = expand_and_prune(
            hypotheses,
            operators,
            max_hypotheses,
        )

    hypotheses.sort(key=score_hypothesis)

    return {
        "hypotheses": hypotheses,
    }


print("[ANALYSIS] ✅ Symbolic solver search core defined")


[ANALYSIS] ✅ Symbolic solver search core defined


In [48]:
# CELL A03
# Solver Trace Store (Analysis-Only)

"""
Defines the global solver trace store for the analysis notebook.

INVARIANTS:
- Analysis-only
- Append-only
- Deterministic
- No execution, no governance, no submission
"""

# -----------------------------------------------------------------------------
# Trace store initialization
# -----------------------------------------------------------------------------

if "traces" not in globals():
    traces = []

print(f"[ANALYSIS] ✅ Solver trace store ready ({len(traces)} traces)")

[ANALYSIS] ✅ Solver trace store ready (50 traces)


In [49]:
# CELL A04
# ARC Dataset Loader (Analysis-Only, ARC-AGI-2 2026)

"""
Loads ARC-AGI-2 datasets for solver analysis.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- NO execution
- NO governance
- NO submission logic
"""

import json
import os

# -----------------------------------------------------------------------------
# ARC-AGI-2 Kaggle competition paths (2026)
# -----------------------------------------------------------------------------

ARC_DATA_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

TRAIN_PATH = os.path.join(ARC_DATA_DIR, "arc-agi_training_challenges.json")
EVAL_PATH  = os.path.join(ARC_DATA_DIR, "arc-agi_evaluation_challenges.json")

# -----------------------------------------------------------------------------
# Defensive existence checks
# -----------------------------------------------------------------------------

for path in [TRAIN_PATH, EVAL_PATH]:
    if not os.path.exists(path):
        raise RuntimeError(
            f"ARC dataset file not found: {path}\n"
            "Kaggle dataset layout may have changed."
        )

# -----------------------------------------------------------------------------
# Load datasets
# -----------------------------------------------------------------------------

with open(TRAIN_PATH, "r") as f:
    train_challenges = json.load(f)

with open(EVAL_PATH, "r") as f:
    eval_challenges = json.load(f)

# -----------------------------------------------------------------------------
# ARC namespace (analysis-only container)
# -----------------------------------------------------------------------------

class ARC:
    train_challenges = train_challenges
    eval_challenges = eval_challenges

print(
    "[ARC] ✅ ARC-AGI-2 datasets loaded (analysis-only)\n"
    f" - Train tasks: {len(ARC.train_challenges)}\n"
    f" - Eval tasks: {len(ARC.eval_challenges)}"
)

[ARC] ✅ ARC-AGI-2 datasets loaded (analysis-only)
 - Train tasks: 1000
 - Eval tasks: 120


In [50]:
# CELL A05
# Solver Traced Entry Point (Analysis-Only)

"""
Defines run_search_traced for the analysis notebook.

INVARIANTS:
- Analysis-only
- No executors
- No governance
- No submission
- Observational tracing only
"""

# -----------------------------------------------------------------------------
# Dependency check
# -----------------------------------------------------------------------------

required = [
    "run_search",
    "traces",
    "grid_features",
    "score_hypothesis",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "run_search_traced cannot be defined.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Traced wrapper
# -----------------------------------------------------------------------------

def run_search_traced(
    input_grid,
    max_steps: int = 2,
    max_hypotheses: int = 16,
):
    """
    Runs the symbolic solver and records a trace.

    This function is observational ONLY.
    """

    result = run_search(
        input_grid=input_grid,
        max_steps=max_steps,
        max_hypotheses=max_hypotheses,
    )

    hypotheses = result.get("hypotheses", [])

    trace = {
        "input_grid": input_grid,
        "input_features": grid_features(input_grid),
        "search_config": {
            "max_steps": max_steps,
            "max_hypotheses": max_hypotheses,
        },
        "num_final_hypotheses": len(hypotheses),
        "raw_hypotheses": hypotheses,
    }

    traces.append(trace)
    return result


print("[ANALYSIS] ✅ run_search_traced defined (analysis-only)")

[ANALYSIS] ✅ run_search_traced defined (analysis-only)


In [51]:
# CELL D00
# Solver Core Bootstrap (Analysis / Dataset Notebook)

"""
This cell defines the REQUIRED solver-core symbols for the
solver-derivation notebook.

This notebook is ANALYSIS-ONLY.
It must never define or modify solver behavior.

Expected source:
- Solver core cells copied from submission notebook
  (grid_features, score_hypothesis, run_search_traced, etc.)

This cell performs:
- Explicit dependency validation
- Trace store initialization (if absent)
"""

# -----------------------------------------------------------------------------
# Required solver-core symbols
# -----------------------------------------------------------------------------

REQUIRED_SOLVER_SYMBOLS = [
    "grid_features",
    "score_hypothesis",
]

missing = [s for s in REQUIRED_SOLVER_SYMBOLS if s not in globals()]

if missing:
    raise RuntimeError(
        "Solver core not loaded in solver-derivation notebook.\n\n"
        "You must COPY the solver core cells (structural features, "
        "scoring, search, tracing) into this notebook BEFORE running "
        "dataset generation.\n\n"
        "Missing symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Trace store initialization (analysis-only)
# -----------------------------------------------------------------------------

if "traces" not in globals():
    traces = []
    print("[BOOTSTRAP] Initialized empty solver trace store.")

print(
    "[BOOTSTRAP] ✅ Solver core detected for dataset derivation.\n"
    f" - grid_features: {'grid_features' in globals()}\n"
    f" - score_hypothesis: {'score_hypothesis' in globals()}\n"
    f" - traces available: {len(traces)}"
)

[BOOTSTRAP] ✅ Solver core detected for dataset derivation.
 - grid_features: True
 - score_hypothesis: True
 - traces available: 50


In [52]:
# CELL D00.1
# Bulk Solver Trace Generation (Analysis-Only, Deterministic)

"""
This cell populates the solver trace store by running the
symbolic solver in advisory / traced mode over ARC inputs.

INVARIANTS:
- Analysis-only
- Offline
- Deterministic
- No executor, no governance, no submission
"""

# -----------------------------------------------------------------------------
# Hard dependency checks
# -----------------------------------------------------------------------------

required = [
    "ARC",
    "run_search_traced",
    "traces",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Bulk trace generation failed.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Configuration (SAFE DEFAULTS)
# -----------------------------------------------------------------------------

MAX_TASKS = 50
MAX_STEPS = 2
MAX_HYPOTHESES = 16

SOURCE_SPLIT = "train"

# -----------------------------------------------------------------------------
# Select ARC source
# -----------------------------------------------------------------------------

if SOURCE_SPLIT == "train":
    source = ARC.train_challenges
elif SOURCE_SPLIT == "eval":
    source = ARC.eval_challenges
else:
    raise RuntimeError(f"Unknown SOURCE_SPLIT: {SOURCE_SPLIT}")

# -----------------------------------------------------------------------------
# Trace generation
# -----------------------------------------------------------------------------

initial_trace_count = len(traces)
processed = 0

for task_id, task in source.items():
    if processed >= MAX_TASKS:
        break

    train_cases = task.get("train", [])
    if not train_cases:
        continue

    input_grid = train_cases[0]["input"]

    run_search_traced(
        input_grid=input_grid,
        max_steps=MAX_STEPS,
        max_hypotheses=MAX_HYPOTHESES,
    )

    traces[-1]["task_id"] = task_id
    traces[-1]["split"] = SOURCE_SPLIT

    processed += 1

print(
    "[TRACE] ✅ Bulk solver trace generation complete\n"
    f" - Source split: {SOURCE_SPLIT}\n"
    f" - Tasks processed: {processed}\n"
    f" - Traces before: {initial_trace_count}\n"
    f" - Traces after: {len(traces)}"
)

[TRACE] ✅ Bulk solver trace generation complete
 - Source split: train
 - Tasks processed: 50
 - Traces before: 50
 - Traces after: 100


In [53]:
# CELL D01
# Solver-Derivation Dataset Generation (v1.0, Observational Only)

"""
Materializes a solver-derivation dataset from recorded solver traces.

INVARIANTS:
- Offline only
- Deterministic
- Observational (non-causal)
- Analysis notebook ONLY
"""

import json
import hashlib
from datetime import datetime, timezone

# -----------------------------------------------------------------------------
# Helper utilities
# -----------------------------------------------------------------------------

def _stable_hash(obj) -> str:
    blob = json.dumps(obj, sort_keys=True)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def _extract_hypothesis_record(h, rank: int):
    feats = grid_features(h.grid)
    return {
        "rank": rank,
        "score": score_hypothesis(h),
        "history": list(h.history),
        "grid": h.grid,
        "features": {
            "shape": feats["shape"],
            "background": feats["background"],
            "num_colors": feats["num_colors"],
            "num_components": feats["num_components"],
            "symmetry": feats["symmetry"],
        },
    }


# -----------------------------------------------------------------------------
# Dataset construction
# -----------------------------------------------------------------------------

records = []

for idx, trace in enumerate(traces):
    record = {
        "record_id": None,
        "task_id": trace.get("task_id", f"UNKNOWN_{idx}"),
        "split": trace.get("split", "unknown"),
        "input": {
            "grid": trace.get("input_grid"),
            "features": trace.get("input_features"),
        },
        "solver_run": trace.get("search_config"),
        "hypotheses": [
            _extract_hypothesis_record(h, rank=i + 1)
            for i, h in enumerate(trace.get("raw_hypotheses", []))
        ],
        "summary": {
            "num_hypotheses_retained": trace.get("num_final_hypotheses"),
        },
    }

    record["record_id"] = _stable_hash(record)
    records.append(record)

# -----------------------------------------------------------------------------
# Final dataset wrapper
# -----------------------------------------------------------------------------

dataset = {
    "dataset_version": "v1.0",
    "schema_version": "solver-derivation-v1",
    "generated_by": "arc-agi-v17-solver-derivation.ipynb",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "generation_context": {
        "deterministic": True,
        "offline_only": True,
        "solver_version": "v17-symbolic-core",
        "executor_used": False,
    },
    "records": records,
}

# -----------------------------------------------------------------------------
# Write dataset to disk
# -----------------------------------------------------------------------------

OUTPUT_PATH = "solver_derivation_dataset_v1.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(dataset, f, indent=2)

print(
    "[DATASET] ✅ Solver-derivation dataset written\n"
    f" - Path: {OUTPUT_PATH}\n"
    f" - Records: {len(records)}\n"
    f" - Schema: solver-derivation-v1"
)

[DATASET] ✅ Solver-derivation dataset written
 - Path: solver_derivation_dataset_v1.json
 - Records: 100
 - Schema: solver-derivation-v1


In [54]:
# CELL D01a
# Executor Output Loader (Analysis-Only, Dataset 2 Infrastructure)

"""
Loads executor outputs for solver–executor disagreement analysis.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution or submission

This cell is robust to Kaggle's dataset mounting layout:
- /kaggle/input/datasets/<dataset>/...
- /kaggle/input/competitions/<competition>/...
"""

import json
import os

KAGGLE_INPUT_ROOT = "/kaggle/input"
TARGET_FILENAME = "executor_outputs.json"

executor_output_path = None

# -----------------------------------------------------------------------------
# RECURSIVE SEARCH UNDER /kaggle/input
# -----------------------------------------------------------------------------
for root, dirs, files in os.walk(KAGGLE_INPUT_ROOT):
    if TARGET_FILENAME in files:
        executor_output_path = os.path.join(root, TARGET_FILENAME)
        break

if executor_output_path is None:
    raise RuntimeError(
        "[D01a] executor_outputs.json not found under /kaggle/input.\n\n"
        "Searched recursively under:\n"
        "  /kaggle/input\n\n"
        "Fix:\n"
        "  Ensure the dataset containing executor_outputs.json is attached\n"
        "  and visible in the notebook side panel."
    )

# -----------------------------------------------------------------------------
# LOAD EXECUTOR OUTPUTS
# -----------------------------------------------------------------------------
with open(executor_output_path, "r") as f:
    executor_outputs = json.load(f)

print(
    "[D01a] ✅ Executor outputs loaded (analysis-only)\n"
    f" - Path  : {executor_output_path}\n"
    f" - Tasks : {len(executor_outputs)}"
)

[D01a] ✅ Executor outputs loaded (analysis-only)
 - Path  : /kaggle/input/datasets/pharaohtut/arc-agi-2-executor-and-intent-analysis-inputs/executor_outputs.json
 - Tasks : 240


In [55]:
# CELL D01a.1
# Solver Trace Normalizer (Analysis-Only Infrastructure)

"""
Normalizes solver behavior into `solver_traces`.

Priority:
1) Use Dataset‑1 `records` (solver-derivation dataset) if present:
   - final_output := top-ranked hypothesis grid (rank 1)
   - top_k_grids  := all retained hypothesis grids
   - scores       := hypothesis scores

2) Fallback to in-notebook `traces`:
   - expects list of trace dicts with task_id and optional output/prediction

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution
"""

from datetime import datetime, timezone

_CELL_D01A_1_SOURCE = r'''
# CELL D01a.1
# Solver Trace Normalizer

from datetime import datetime, timezone

solver_traces = {}

if "records" in globals() and isinstance(records, list) and records:
    for rec in records:
        task_id = rec.get("task_id")
        hyps = rec.get("hypotheses", [])
        top_grid = hyps[0]["grid"] if hyps else None
        solver_traces[task_id] = {
            "final_output": top_grid,
            "top_k_grids": [h["grid"] for h in hyps],
            "scores": [h.get("score") for h in hyps],
            "raw": rec,
        }
else:
    # fallback on traces list
    ...
'''

solver_traces = {}

source_used = None
normalized = 0
skipped = 0

# ---------------------------------------------------------------------
# Preferred source: Dataset‑1 `records` (produced by CELL D01)
# ---------------------------------------------------------------------
if "records" in globals() and isinstance(records, list) and len(records) > 0:
    source_used = "records(list)"

    for rec in records:
        if not isinstance(rec, dict):
            skipped += 1
            continue

        task_id = rec.get("task_id")
        if not task_id:
            skipped += 1
            continue

        hyps = rec.get("hypotheses", [])
        if not isinstance(hyps, list):
            hyps = []

        top_k_grids = []
        top_k_scores = []

        for h in hyps:
            if isinstance(h, dict) and "grid" in h:
                top_k_grids.append(h["grid"])
                top_k_scores.append(h.get("score"))

        final_output = top_k_grids[0] if top_k_grids else None

        solver_traces[task_id] = {
            "final_output": final_output,      # canonical "solver choice" for Dataset-2
            "top_k_grids": top_k_grids,        # all retained hypotheses for match-rank
            "top_k_scores": top_k_scores,      # aligned with grids
            "raw": rec,                        # full record for auditability
        }
        normalized += 1

# ---------------------------------------------------------------------
# Fallback source: `traces` (often list-of-trace dicts)
# ---------------------------------------------------------------------
elif "traces" in globals():
    source_used = f"traces({type(traces).__name__})"

    if isinstance(traces, list):
        for idx, entry in enumerate(traces):
            if not isinstance(entry, dict):
                skipped += 1
                continue

            task_id = entry.get("task_id") or entry.get("task") or entry.get("id")
            if not task_id:
                skipped += 1
                continue

            final_output = (
                entry.get("final_output")
                or entry.get("output")
                or entry.get("prediction")
            )

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],         # unknown in this format
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1

    elif isinstance(traces, dict):
        for task_id, entry in traces.items():
            if isinstance(entry, dict):
                final_output = (
                    entry.get("final_output")
                    or entry.get("output")
                    or entry.get("prediction")
                )
            else:
                final_output = entry

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1
    else:
        raise RuntimeError(
            f"[D01a.1] Unsupported `traces` type: {type(traces).__name__}"
        )

else:
    raise RuntimeError(
        "[D01a.1] No solver trace source found.\n"
        "Expected `records` (preferred) or `traces` (fallback) to exist."
    )

D01A_1_AUDIT = {
    "cell": "D01a.1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_used": source_used,
    "tasks_normalized": len(solver_traces),
    "normalized_count": normalized,
    "skipped_count": skipped,
    "final_output_missing": sum(1 for v in solver_traces.values() if v.get("final_output") is None),
}

print(
    "[D01a.1] ✅ Solver traces normalized (analysis-only)\n"
    f" - Source           : {source_used}\n"
    f" - Tasks            : {len(solver_traces)}\n"
    f" - Missing outputs  : {D01A_1_AUDIT['final_output_missing']}"
)

[D01a.1] ✅ Solver traces normalized (analysis-only)
 - Source           : records(list)
 - Tasks            : 50
 - Missing outputs  : 0


In [56]:
# CELL D01a.1
# Solver Trace Normalizer (Analysis-Only Infrastructure)

"""
Normalizes solver behavior into `solver_traces`.

Priority:
1) Use Dataset‑1 `records` (solver-derivation dataset) if present:
   - final_output := top-ranked hypothesis grid (rank 1)
   - top_k_grids  := all retained hypothesis grids
   - scores       := hypothesis scores

2) Fallback to in-notebook `traces`:
   - expects list of trace dicts with task_id and optional output/prediction

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution
"""

from datetime import datetime, timezone

_CELL_D01A_1_SOURCE = r'''
# CELL D01a.1
# Solver Trace Normalizer

from datetime import datetime, timezone

solver_traces = {}

if "records" in globals() and isinstance(records, list) and records:
    for rec in records:
        task_id = rec.get("task_id")
        hyps = rec.get("hypotheses", [])
        top_grid = hyps[0]["grid"] if hyps else None
        solver_traces[task_id] = {
            "final_output": top_grid,
            "top_k_grids": [h["grid"] for h in hyps],
            "scores": [h.get("score") for h in hyps],
            "raw": rec,
        }
else:
    # fallback on traces list
    ...
'''

solver_traces = {}

source_used = None
normalized = 0
skipped = 0

# ---------------------------------------------------------------------
# Preferred source: Dataset‑1 `records` (produced by CELL D01)
# ---------------------------------------------------------------------
if "records" in globals() and isinstance(records, list) and len(records) > 0:
    source_used = "records(list)"

    for rec in records:
        if not isinstance(rec, dict):
            skipped += 1
            continue

        task_id = rec.get("task_id")
        if not task_id:
            skipped += 1
            continue

        hyps = rec.get("hypotheses", [])
        if not isinstance(hyps, list):
            hyps = []

        top_k_grids = []
        top_k_scores = []

        for h in hyps:
            if isinstance(h, dict) and "grid" in h:
                top_k_grids.append(h["grid"])
                top_k_scores.append(h.get("score"))

        final_output = top_k_grids[0] if top_k_grids else None

        solver_traces[task_id] = {
            "final_output": final_output,      # canonical "solver choice" for Dataset-2
            "top_k_grids": top_k_grids,        # all retained hypotheses for match-rank
            "top_k_scores": top_k_scores,      # aligned with grids
            "raw": rec,                        # full record for auditability
        }
        normalized += 1

# ---------------------------------------------------------------------
# Fallback source: `traces` (often list-of-trace dicts)
# ---------------------------------------------------------------------
elif "traces" in globals():
    source_used = f"traces({type(traces).__name__})"

    if isinstance(traces, list):
        for idx, entry in enumerate(traces):
            if not isinstance(entry, dict):
                skipped += 1
                continue

            task_id = entry.get("task_id") or entry.get("task") or entry.get("id")
            if not task_id:
                skipped += 1
                continue

            final_output = (
                entry.get("final_output")
                or entry.get("output")
                or entry.get("prediction")
            )

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],         # unknown in this format
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1

    elif isinstance(traces, dict):
        for task_id, entry in traces.items():
            if isinstance(entry, dict):
                final_output = (
                    entry.get("final_output")
                    or entry.get("output")
                    or entry.get("prediction")
                )
            else:
                final_output = entry

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1
    else:
        raise RuntimeError(
            f"[D01a.1] Unsupported `traces` type: {type(traces).__name__}"
        )

else:
    raise RuntimeError(
        "[D01a.1] No solver trace source found.\n"
        "Expected `records` (preferred) or `traces` (fallback) to exist."
    )

D01A_1_AUDIT = {
    "cell": "D01a.1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_used": source_used,
    "tasks_normalized": len(solver_traces),
    "normalized_count": normalized,
    "skipped_count": skipped,
    "final_output_missing": sum(1 for v in solver_traces.values() if v.get("final_output") is None),
}

print(
    "[D01a.1] ✅ Solver traces normalized (analysis-only)\n"
    f" - Source           : {source_used}\n"
    f" - Tasks            : {len(solver_traces)}\n"
    f" - Missing outputs  : {D01A_1_AUDIT['final_output_missing']}"
)

[D01a.1] ✅ Solver traces normalized (analysis-only)
 - Source           : records(list)
 - Tasks            : 50
 - Missing outputs  : 0


In [57]:
# CELL D01b
# Solver–Executor Disagreement Dataset Builder (Analysis‑Only)

"""
Constructs Dataset‑2 by joining:
- executor_outputs (from D01a)
- solver_traces (from D01a.1)
- intent metadata (arc_intent_run_export.json,
  arc_intent_annotation_dataset_v1.json)

Produces:
- solver_executor_disagreement_dataset (in‑memory)
- solver_executor_disagreement_dataset_v1.json (written to /kaggle/working)

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution or submission
"""

import json
import os
from datetime import datetime, timezone

# ---------------------------------------------------------------------
# HARD DEPENDENCY CHECKS
# ---------------------------------------------------------------------
if "executor_outputs" not in globals():
    raise RuntimeError("[D01b] Missing executor_outputs (run D01a first)")

if "solver_traces" not in globals():
    raise RuntimeError("[D01b] Missing solver_traces (run D01a.1 first)")

# ---------------------------------------------------------------------
# LOCATE INTENT FILES (KAGGLE‑SAFE)
# ---------------------------------------------------------------------
KAGGLE_INPUT_ROOT = "/kaggle/input"
INTENT_RUN_FILE = "arc_intent_run_export.json"
INTENT_ANNOTATION_FILE = "arc_intent_annotation_dataset_v1.json"

intent_run_path = None
intent_annotation_path = None

for root, _, files in os.walk(KAGGLE_INPUT_ROOT):
    if intent_run_path is None and INTENT_RUN_FILE in files:
        intent_run_path = os.path.join(root, INTENT_RUN_FILE)
    if intent_annotation_path is None and INTENT_ANNOTATION_FILE in files:
        intent_annotation_path = os.path.join(root, INTENT_ANNOTATION_FILE)
    if intent_run_path and intent_annotation_path:
        break

if intent_run_path is None or intent_annotation_path is None:
    raise RuntimeError(
        "[D01b] Intent files not found under /kaggle/input.\n"
        "Ensure the ARC INTENT ANNOTATION dataset is attached."
    )

with open(intent_run_path, "r") as f:
    intent_run_export = json.load(f)

with open(intent_annotation_path, "r") as f:
    intent_annotation_dataset = json.load(f)

# Normalize intent structures
if isinstance(intent_run_export, list):
    intent_run_by_task = {
        e["task_id"]: e for e in intent_run_export
        if isinstance(e, dict) and "task_id" in e
    }
else:
    intent_run_by_task = intent_run_export

if isinstance(intent_annotation_dataset, list):
    intent_annotation_by_task = {
        e["task_id"]: e for e in intent_annotation_dataset
        if isinstance(e, dict) and "task_id" in e
    }
else:
    intent_annotation_by_task = intent_annotation_dataset

# ---------------------------------------------------------------------
# BUILD DATASET‑2 RECORDS
# ---------------------------------------------------------------------
records = []
matched = 0

def grids_equal(a, b):
    return a == b

for task_id, solver_entry in solver_traces.items():
    exec_entry = executor_outputs.get(task_id)
    if exec_entry is None:
        continue

    exec_grid = (
        exec_entry.get("output_grid")
        if isinstance(exec_entry, dict)
        else exec_entry
    )

    top_k = solver_entry.get("top_k_grids", [])
    solver_final = solver_entry.get("final_output")

    matched_rank = None
    for i, g in enumerate(top_k):
        if grids_equal(g, exec_grid):
            matched_rank = i + 1
            matched += 1
            break

    records.append({
        "task_id": task_id,
        "disagreement": matched_rank is None,
        "matched_solver_rank": matched_rank,
        "executor_output": exec_grid,
        "solver_final_output": solver_final,
        "solver_top_k": top_k,
        "intent_run": intent_run_by_task.get(task_id),
        "intent_annotation": intent_annotation_by_task.get(task_id),
    })

solver_executor_disagreement_dataset = {
    "dataset_version": "v1.0",
    "schema": "solver-executor-disagreement-v1",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "records": records,
}

# ---------------------------------------------------------------------
# WRITE DATASET TO DISK (AUDITABLE)
# ---------------------------------------------------------------------
OUTPUT_PATH = "solver_executor_disagreement_dataset_v1.json"
with open(OUTPUT_PATH, "w") as f:
    json.dump(solver_executor_disagreement_dataset, f, indent=2)

print(
    "[D01b] ✅ Solver–Executor Disagreement Dataset built\n"
    f" - Records written : {len(records)}\n"
    f" - Matched outputs : {matched}\n"
    f" - Output file     : {OUTPUT_PATH}"
)

[D01b] ✅ Solver–Executor Disagreement Dataset built
 - Records written : 50
 - Matched outputs : 0
 - Output file     : solver_executor_disagreement_dataset_v1.json


In [58]:
# CELL D01c
# Semantic Coverage Computation (Analysis‑Only, Dataset‑Anchored, Robust)

"""
Computes semantic coverage per task directly from ARC challenge data.

This version is:
- ORDER‑INDEPENDENT
- IMMUNE to ARC class shadowing
- GUARANTEED to populate coverage if ARC data exists

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",   # loaded in CELL A04
    "grid_features",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic coverage computation failed.\n"
        "Missing required dataset inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if not isinstance(train_challenges, dict) or not train_challenges:
    raise RuntimeError(
        "Semantic coverage computation failed.\n"
        "train_challenges is empty or invalid."
    )

# ---------------------------------------------------------------------
# Coverage containers
# ---------------------------------------------------------------------
coverage_by_task = {}
coverage_label_by_task = {}

# ---------------------------------------------------------------------
# Structural coverage heuristics (baseline, pre‑annotation)
# ---------------------------------------------------------------------
for task_id, task in train_challenges.items():
    train_cases = task.get("train", [])
    if not train_cases:
        continue

    input_grid = train_cases[0].get("input")
    if input_grid is None:
        continue

    feats = grid_features(input_grid)

    num_components = feats["num_components"]
    has_symmetry = (
        feats["symmetry"]["vertical"]
        or feats["symmetry"]["horizontal"]
        or feats["symmetry"]["rot180"]
    )

    if num_components <= 2 and has_symmetry:
        label = "weakly_covered"
    else:
        label = "uncovered"

    coverage_by_task[task_id] = {
        "task_id": task_id,
        "num_components": num_components,
        "has_symmetry": has_symmetry,
        "grid_shape": feats["shape"],
    }

    coverage_label_by_task[task_id] = label

# ---------------------------------------------------------------------
# Postconditions (HARD GUARANTEE)
# ---------------------------------------------------------------------
if not coverage_by_task:
    raise RuntimeError(
        "Semantic coverage computation failed.\n"
        "No tasks were processed — this indicates a dataset loading error."
    )

print(
    "[COVERAGE] ✅ Semantic coverage computed (dataset‑anchored)\n"
    f" - Tasks analyzed : {len(coverage_by_task)}\n"
    f" - Weakly covered : {sum(1 for v in coverage_label_by_task.values() if v == 'weakly_covered')}\n"
    f" - Uncovered      : {sum(1 for v in coverage_label_by_task.values() if v == 'uncovered')}"
)

[COVERAGE] ✅ Semantic coverage computed (dataset‑anchored)
 - Tasks analyzed : 1000
 - Weakly covered : 45
 - Uncovered      : 955


In [88]:
# CELL D01c.1
# Semantic Annotation Entry (Analysis‑Only, Uniform‑Grid‑Safe)

"""
Allows manual semantic annotation of a task by selecting
a PRIMARY connected component.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

from pprint import pprint

# ---------------------------------------------------------------------
# USER INPUT (EDIT THESE TWO LINES ONLY)
# ---------------------------------------------------------------------
TASK_ID = "0d3d703e"          # ← THIS MUST BE 0d3d703e
PRIMARY_COMPONENT_ID = 0      # ← placeholder (NOT committing yet)

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "connected_components",
    "infer_background_color",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic annotation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if TASK_ID not in train_challenges:
    raise RuntimeError(f"Task {TASK_ID} not found in train_challenges.")

# ---------------------------------------------------------------------
# Load task input grid
# ---------------------------------------------------------------------
task = train_challenges[TASK_ID]
train_cases = task.get("train", [])

if not train_cases:
    raise RuntimeError(f"Task {TASK_ID} has no training cases.")

input_grid = train_cases[0]["input"]
h, w = len(input_grid), len(input_grid[0])

# ---------------------------------------------------------------------
# Compute connected components
# ---------------------------------------------------------------------
bg = infer_background_color(input_grid)
components = connected_components(input_grid, bg)

# ---------------------------------------------------------------------
# Display components for inspection
# ---------------------------------------------------------------------
print(f"[ANNOTATION] Task {TASK_ID}")
print(f"Found {len(components)} component(s):\n")

for idx, comp in enumerate(components):
    r0, r1, c0, c1 = comp["bbox"]
    print(
        f"Component {idx}: "
        f"color={comp['color']} | "
        f"area={comp['area']} | "
        f"bbox=({r0}:{r1}, {c0}:{c1}) | "
        f"EXTRACTED"
    )

print("\n[INSPECTION ONLY] — no semantic decision committed.")

[ANNOTATION] Task 0d3d703e
Found 2 component(s):

Component 0: color=8 | area=3 | bbox=(0:2, 1:1) | EXTRACTED
Component 1: color=6 | area=3 | bbox=(0:2, 2:2) | EXTRACTED

[INSPECTION ONLY] — no semantic decision committed.


In [89]:
# CELL D01c.1b
# Semantic Annotation Entry — EOC=2 (PRIMARY + SECONDARY, Analysis‑Only)

"""
Commits a semantic annotation for an EOC=2 task using:
- PRIMARY component
- SECONDARY component

This cell is designed specifically for the EOC=2 regime where
meaning resides in the relationship between two symmetric components.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
- Append-only semantic annotation store
"""

from pprint import pprint

# ---------------------------------------------------------------------
# USER INPUT (EDIT THESE THREE LINES ONLY)
# ---------------------------------------------------------------------
TASK_ID = "0d3d703e"
PRIMARY_COMPONENT_ID = 0
SECONDARY_COMPONENT_ID = 1

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "connected_components",
    "infer_background_color",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic annotation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if TASK_ID not in train_challenges:
    raise RuntimeError(f"Task {TASK_ID} not found in train_challenges.")

# ---------------------------------------------------------------------
# Load task input grid
# ---------------------------------------------------------------------
task = train_challenges[TASK_ID]
train_cases = task.get("train", [])

if not train_cases:
    raise RuntimeError(f"Task {TASK_ID} has no training cases.")

input_grid = train_cases[0]["input"]
h, w = len(input_grid), len(input_grid[0])

# ---------------------------------------------------------------------
# Compute connected components
# ---------------------------------------------------------------------
bg = infer_background_color(input_grid)
components = connected_components(input_grid, bg)

if len(components) != 2:
    raise RuntimeError(
        f"EOC=2 annotation expected exactly 2 components, found {len(components)}."
    )

# ---------------------------------------------------------------------
# Validate component indices
# ---------------------------------------------------------------------
if PRIMARY_COMPONENT_ID not in (0, 1):
    raise RuntimeError("PRIMARY_COMPONENT_ID must be 0 or 1.")

if SECONDARY_COMPONENT_ID not in (0, 1):
    raise RuntimeError("SECONDARY_COMPONENT_ID must be 0 or 1.")

if PRIMARY_COMPONENT_ID == SECONDARY_COMPONENT_ID:
    raise RuntimeError("PRIMARY and SECONDARY components must be different.")

primary_comp = components[PRIMARY_COMPONENT_ID]
secondary_comp = components[SECONDARY_COMPONENT_ID]

# ---------------------------------------------------------------------
# Initialize annotation store (append‑only)
# ---------------------------------------------------------------------
if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Commit semantic annotation
# ---------------------------------------------------------------------
semantic_annotations[TASK_ID] = {
    "task_id": TASK_ID,
    "effective_object_count": 2,
    "annotation_type": "PRIMARY_PLUS_SECONDARY",
    "primary": {
        "component_id": PRIMARY_COMPONENT_ID,
        "structural_importance": "PRIMARY",
        "primitive_type": "OBJECT",
        "color": primary_comp["color"],
        "area": primary_comp["area"],
        "bbox": primary_comp["bbox"],
    },
    "secondary": {
        "component_id": SECONDARY_COMPONENT_ID,
        "structural_importance": "SECONDARY",
        "primitive_type": "OBJECT",
        "color": secondary_comp["color"],
        "area": secondary_comp["area"],
        "bbox": secondary_comp["bbox"],
    },
    "relationship_note": "Symmetric paired objects; semantics encoded in relative position.",
}

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
print(f"[ANNOTATION] ✅ EOC=2 semantic annotation committed for task {TASK_ID}\n")
print("Recorded annotation:")
pprint(semantic_annotations[TASK_ID])

[ANNOTATION] ✅ EOC=2 semantic annotation committed for task 0d3d703e

Recorded annotation:
{'annotation_type': 'PRIMARY_PLUS_SECONDARY',
 'effective_object_count': 2,
 'primary': {'area': 3,
             'bbox': (0, 2, 1, 1),
             'color': 8,
             'component_id': 0,
             'primitive_type': 'OBJECT',
             'structural_importance': 'PRIMARY'},
 'relationship_note': 'Symmetric paired objects; semantics encoded in relative '
                      'position.',
 'secondary': {'area': 3,
               'bbox': (0, 2, 2, 2),
               'color': 6,
               'component_id': 1,
               'primitive_type': 'OBJECT',
               'structural_importance': 'SECONDARY'},
 'task_id': '0d3d703e'}


In [90]:
# CELL D01c.2
# Semantic Annotation Integration (Analysis‑Only)

"""
Integrates manual semantic annotations into coverage.

Purpose:
- Promote annotated tasks to FULLY_COVERED
- Preserve baseline coverage for all other tasks
- Enable downstream budget unlock

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "semantic_annotations",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic annotation integration failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Apply annotation effects
# ---------------------------------------------------------------------
promoted = []

for task_id, ann in semantic_annotations.items():
    if task_id not in coverage_by_task:
        continue

    # Promote coverage label
    coverage_label_by_task[task_id] = "fully_covered"

    # Attach annotation metadata for auditability
    coverage_by_task[task_id]["semantic_annotation"] = ann
    coverage_by_task[task_id]["annotated"] = True

    promoted.append(task_id)

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
print(
    "[COVERAGE] ✅ Semantic annotations integrated\n"
    f" - Tasks promoted to fully_covered : {len(promoted)}\n"
    f" - Task IDs : {promoted}"
)

[COVERAGE] ✅ Semantic annotations integrated
 - Tasks promoted to fully_covered : 2
 - Task IDs : ['28e73c20', '0d3d703e']


In [95]:
# CELL D01c.2b
# Semantic Annotation Integration — EOC=2 Micro‑Batch (Analysis‑Only)

"""
Integrates newly committed EOC=2 semantic annotations into coverage.

Purpose:
- Promote newly annotated tasks to FULLY_COVERED
- Preserve prior promotions
- Maintain append‑only, deterministic coverage state
- Prepare for budget recomputation

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "semantic_annotations",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic annotation integration failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Apply annotation effects (idempotent)
# ---------------------------------------------------------------------
promoted = []

for task_id, ann in semantic_annotations.items():
    if task_id not in coverage_by_task:
        continue

    # Promote coverage label
    coverage_label_by_task[task_id] = "fully_covered"

    # Attach annotation metadata for auditability
    coverage_by_task[task_id]["semantic_annotation"] = ann
    coverage_by_task[task_id]["annotated"] = True

    promoted.append(task_id)

# ---------------------------------------------------------------------
# Report (FIXED: no unterminated f-string)
# ---------------------------------------------------------------------
print("[COVERAGE] ✅ Semantic annotations integrated (EOC=2 batch)")
print(f" - Tasks now fully_covered : {len(promoted)}")
print(" - Task IDs :")
for t in promoted:
    print(f"   • {t}")

[COVERAGE] ✅ Semantic annotations integrated (EOC=2 batch)
 - Tasks now fully_covered : 7
 - Task IDs :
   • 28e73c20
   • 0d3d703e
   • 21f83797
   • 253bf280
   • 3befdf3e
   • 44f52bb0
   • 48131b3c


In [93]:
# CELL D01c.3
# EOC=2 REGIME MICRO‑BATCH ANNOTATION (SAFE, GUARDED, ANALYSIS‑ONLY)

"""
Safely accelerates semantic annotation for the EOC=2 regime by:
- Selecting a bounded micro‑batch of structurally identical tasks
- Enforcing strict regime guards (EOC=2, symmetry=1.0, weakly_covered)
- Committing PRIMARY+SECONDARY annotations in a single controlled step

This cell is the ONLY sanctioned way to accelerate annotation.
If any guard fails, the cell aborts with no mutation.

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
- No cross‑regime annotation
"""

from pprint import pprint

# ---------------------------------------------------------------------
# CONFIGURATION (EDIT BATCH_SIZE ONLY)
# ---------------------------------------------------------------------
BATCH_SIZE = 5   # ✅ Safe default for EOC=2; increase to 10 only after validation

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "semantic_annotations",
    "train_challenges",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=2 micro‑batch annotation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Select EOC=2 regime candidates (STRICT FILTER)
# ---------------------------------------------------------------------
candidates = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) != "weakly_covered":
        continue
    if cov.get("num_components") != 2:
        continue
    if not cov.get("has_symmetry", False):
        continue
    candidates.append(task_id)

if not candidates:
    print("[EOC=2] ✅ No remaining EOC=2 candidates — regime exhausted.")
    raise SystemExit

batch = candidates[:BATCH_SIZE]

print("[EOC=2] Selected micro‑batch:")
for t in batch:
    print(" -", t)

# ---------------------------------------------------------------------
# Initialize annotation store (append‑only)
# ---------------------------------------------------------------------
if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Apply PRIMARY + SECONDARY annotation per task
# ---------------------------------------------------------------------
annotated = []

for task_id in batch:
    task = train_challenges[task_id]
    input_grid = task["train"][0]["input"]

    bg = infer_background_color(input_grid)
    components = connected_components(input_grid, bg)

    if len(components) != 2:
        raise RuntimeError(
            f"[ABORT] Task {task_id} violated EOC=2 assumption during execution."
        )

    # Deterministic PRIMARY selection:
    # larger bounding‑box footprint wins; ties break by component index
    def footprint(c):
        r0, r1, c0, c1 = c["bbox"]
        return (r1 - r0 + 1) * (c1 - c0 + 1)

    if footprint(components[0]) >= footprint(components[1]):
        primary_id, secondary_id = 0, 1
    else:
        primary_id, secondary_id = 1, 0

    primary = components[primary_id]
    secondary = components[secondary_id]

    semantic_annotations[task_id] = {
        "task_id": task_id,
        "effective_object_count": 2,
        "annotation_type": "PRIMARY_PLUS_SECONDARY",
        "primary": {
            "component_id": primary_id,
            "structural_importance": "PRIMARY",
            "primitive_type": "OBJECT",
            "color": primary["color"],
            "area": primary["area"],
            "bbox": primary["bbox"],
        },
        "secondary": {
            "component_id": secondary_id,
            "structural_importance": "SECONDARY",
            "primitive_type": "OBJECT",
            "color": secondary["color"],
            "area": secondary["area"],
            "bbox": secondary["bbox"],
        },
        "relationship_note": "EOC=2 symmetric paired objects; relative position encodes meaning.",
    }

    annotated.append(task_id)

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
print("\n[EOC=2] ✅ Micro‑batch annotation complete")
print(f" - Tasks annotated : {len(annotated)}")
print(" - Task IDs:")
for t in annotated:
    print("   •", t)

print("\n[EOC=2] Ready for coverage integration.")


[EOC=2] Selected micro‑batch:
 - 21f83797
 - 253bf280
 - 3befdf3e
 - 44f52bb0
 - 48131b3c

[EOC=2] ✅ Micro‑batch annotation complete
 - Tasks annotated : 5
 - Task IDs:
   • 21f83797
   • 253bf280
   • 3befdf3e
   • 44f52bb0
   • 48131b3c

[EOC=2] Ready for coverage integration.


In [100]:
# CELL D01c.4
# EOC=2 REGIME CLOSURE — CONSOLIDATED PIPELINE (ANALYSIS‑ONLY)

"""
Closes the entire EOC=2, symmetry=1.0 semantic regime in one controlled pass.

This cell REPLACES the following manual sequence:
- EOC=2 candidate selection
- PRIMARY+SECONDARY annotation
- Coverage integration
- Budget derivation
- ROI recomputation

It operates at the REGIME level, not the task level.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
- Fail‑closed on any structural violation
"""

import math
import json
from datetime import datetime, timezone
from dataclasses import dataclass

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=2 regime closure failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Phase 1 — Identify EOC=2 regime candidates (STRICT)
# ---------------------------------------------------------------------
eoc2_tasks = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) != "weakly_covered":
        continue
    if cov.get("num_components") != 2:
        continue
    if not cov.get("has_symmetry", False):
        continue
    eoc2_tasks.append(task_id)

print(f"[EOC=2] Candidates identified : {len(eoc2_tasks)}")

if not eoc2_tasks:
    print("[EOC=2] ✅ Regime already exhausted — nothing to do.")
    raise SystemExit

# ---------------------------------------------------------------------
# Phase 2 — Annotate all EOC=2 tasks (PRIMARY + SECONDARY)
# ---------------------------------------------------------------------
annotated = []

for task_id in eoc2_tasks:
    task = train_challenges[task_id]
    input_grid = task["train"][0]["input"]

    bg = infer_background_color(input_grid)
    components = connected_components(input_grid, bg)

    if len(components) != 2:
        raise RuntimeError(
            f"[ABORT] Task {task_id} violated EOC=2 assumption during execution."
        )

    def footprint(c):
        r0, r1, c0, c1 = c["bbox"]
        return (r1 - r0 + 1) * (c1 - c0 + 1)

    if footprint(components[0]) >= footprint(components[1]):
        primary_id, secondary_id = 0, 1
    else:
        primary_id, secondary_id = 1, 0

    semantic_annotations[task_id] = {
        "task_id": task_id,
        "effective_object_count": 2,
        "annotation_type": "PRIMARY_PLUS_SECONDARY",
        "primary": {
            "component_id": primary_id,
            "structural_importance": "PRIMARY",
            "primitive_type": "OBJECT",
            "color": components[primary_id]["color"],
            "area": components[primary_id]["area"],
            "bbox": components[primary_id]["bbox"],
        },
        "secondary": {
            "component_id": secondary_id,
            "structural_importance": "SECONDARY",
            "primitive_type": "OBJECT",
            "color": components[secondary_id]["color"],
            "area": components[secondary_id]["area"],
            "bbox": components[secondary_id]["bbox"],
        },
        "relationship_note": "EOC=2 symmetric paired objects; relative position encodes meaning.",
    }

    annotated.append(task_id)

print(f"[EOC=2] ✅ Annotated tasks : {len(annotated)}")

# ---------------------------------------------------------------------
# Phase 3 — Coverage integration (idempotent)
# ---------------------------------------------------------------------
for task_id in annotated:
    coverage_label_by_task[task_id] = "fully_covered"
    coverage_by_task[task_id]["semantic_annotation"] = semantic_annotations[task_id]
    coverage_by_task[task_id]["annotated"] = True

print("[EOC=2] ✅ Coverage promoted")

# ---------------------------------------------------------------------
# Phase 4 — Budget re‑derivation
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class SearchBudget:
    max_complexity: float

search_budgets.clear()

for task_id in coverage_by_task:
    label = coverage_label_by_task.get(task_id, "uncovered")
    if label == "fully_covered":
        max_complexity = 2.0
    elif label == "weakly_covered":
        max_complexity = 0.75
    else:
        max_complexity = 0.0
    search_budgets[task_id] = SearchBudget(max_complexity)

print("[EOC=2] ✅ Search budgets updated")

# ---------------------------------------------------------------------
# Phase 5 — ROI recomputation
# ---------------------------------------------------------------------
FULLY_COVERED_BUDGET = 2.0

def density_adjustment(n):
    return 1.0 / math.log2(1 + n)

roi_records = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) == "fully_covered":
        continue

    budget = search_budgets[task_id]
    gain = FULLY_COVERED_BUDGET - budget.max_complexity
    if gain <= 0:
        continue

    symmetry_score = 1.0 if cov.get("has_symmetry") else 0.5
    eff_count = max(1, cov.get("num_components", 1))
    roi = gain * symmetry_score * density_adjustment(eff_count)

    roi_records.append({
        "task_id": task_id,
        "coverage_label": coverage_label_by_task[task_id],
        "roi_score": round(roi, 6),
        "potential_budget_gain": gain,
        "symmetry_score": symmetry_score,
        "effective_object_count": eff_count,
        "density_adjustment": round(density_adjustment(eff_count), 6),
    })

roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

with open("prioritized_annotation_targets.json", "w") as f:
    json.dump(
        {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "total_candidates": len(roi_records),
            "roi_tasks": roi_records,
        },
        f,
        indent=2,
        sort_keys=True,
    )

print("[EOC=2] ✅ ROI recomputed")
print(f"[EOC=2] Remaining annotation candidates : {len(roi_records)}")
print("[EOC=2] ✅ REGIME CLOSED")

[EOC=2] Candidates identified : 14
[EOC=2] ✅ Annotated tasks : 14
[EOC=2] ✅ Coverage promoted
[EOC=2] ✅ Search budgets updated
[EOC=2] ✅ ROI recomputed
[EOC=2] Remaining annotation candidates : 979
[EOC=2] ✅ REGIME CLOSED


In [101]:
# CELL D01c.5
# NEXT REGIME DIAGNOSTIC — POST EOC=2 (READ‑ONLY, ANALYSIS‑ONLY)

"""
Identifies and characterizes the next semantic regime after EOC=2 closure.

Purpose:
- Analyze remaining ROI candidates by effective_object_count (EOC)
- Detect whether a coherent next regime exists (EOC=3, EOC=4, etc.)
- Surface symmetry + ROI structure to decide:
  • regime closure
  • guarded micro‑batching
  • or single‑task annotation only

This cell DOES NOT annotate anything.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter, defaultdict

ROI_PATH = "prioritized_annotation_targets.json"

print("=== NEXT REGIME DIAGNOSTIC ===")

# ---------------------------------------------------------------------
# Load ROI artifact
# ---------------------------------------------------------------------
with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"\nRemaining ROI candidates: {len(roi_tasks)}")

if not roi_tasks:
    print("✅ No remaining annotation candidates.")
    raise SystemExit

# ---------------------------------------------------------------------
# Aggregate by effective_object_count
# ---------------------------------------------------------------------
by_eoc = defaultdict(list)

for r in roi_tasks:
    by_eoc[r["effective_object_count"]].append(r)

eoc_counts = {k: len(v) for k, v in by_eoc.items()}

print("\nCandidate counts by effective_object_count:")
for eoc in sorted(eoc_counts):
    print(f"  EOC={eoc}: {eoc_counts[eoc]}")

# ---------------------------------------------------------------------
# Focus on smallest EOC > 2 (next likely regime)
# ---------------------------------------------------------------------
next_eoc = min(eoc for eoc in by_eoc if eoc > 2)
candidates = by_eoc[next_eoc]

print(f"\nNext candidate regime: EOC={next_eoc}")
print(f"Candidate count       : {len(candidates)}")

# ---------------------------------------------------------------------
# Symmetry distribution
# ---------------------------------------------------------------------
symmetry_counts = Counter(r["symmetry_score"] for r in candidates)

print("\nSymmetry score distribution:")
for k, v in sorted(symmetry_counts.items()):
    print(f"  symmetry_score={k}: {v}")

# ---------------------------------------------------------------------
# ROI distribution
# ---------------------------------------------------------------------
roi_scores = [r["roi_score"] for r in candidates]

print("\nROI score summary:")
print(f"  min  : {min(roi_scores):.6f}")
print(f"  max  : {max(roi_scores):.6f}")
print(f"  mean : {sum(roi_scores)/len(roi_scores):.6f}")

# ---------------------------------------------------------------------
# Top candidates for inspection
# ---------------------------------------------------------------------
candidates_sorted = sorted(candidates, key=lambda r: r["roi_score"], reverse=True)

print("\nTop 10 candidates in this regime:")
for r in candidates_sorted[:10]:
    print(
        f" - task_id={r['task_id']} | "
        f"ROI={r['roi_score']} | "
        f"symmetry_score={r['symmetry_score']}"
    )

# Convenience handle
top_next_regime_task = candidates_sorted[0]
print("\nTop candidate object:")
print(top_next_regime_task)

print("\n=== NEXT REGIME DIAGNOSTIC COMPLETE ===")

=== NEXT REGIME DIAGNOSTIC ===

Remaining ROI candidates: 979

Candidate counts by effective_object_count:
  EOC=1: 84
  EOC=2: 104
  EOC=3: 106
  EOC=4: 104
  EOC=5: 75
  EOC=6: 73
  EOC=7: 43
  EOC=8: 51
  EOC=9: 36
  EOC=10: 34
  EOC=11: 33
  EOC=12: 24
  EOC=13: 22
  EOC=14: 13
  EOC=15: 15
  EOC=16: 12
  EOC=17: 14
  EOC=18: 6
  EOC=19: 4
  EOC=20: 8
  EOC=21: 5
  EOC=22: 4
  EOC=23: 6
  EOC=24: 7
  EOC=25: 5
  EOC=26: 4
  EOC=27: 1
  EOC=28: 5
  EOC=29: 5
  EOC=30: 5
  EOC=31: 4
  EOC=32: 4
  EOC=33: 1
  EOC=34: 1
  EOC=36: 1
  EOC=37: 2
  EOC=39: 1
  EOC=40: 1
  EOC=41: 1
  EOC=42: 1
  EOC=43: 2
  EOC=44: 1
  EOC=45: 1
  EOC=46: 2
  EOC=49: 1
  EOC=58: 1
  EOC=60: 2
  EOC=62: 1
  EOC=64: 1
  EOC=65: 1
  EOC=67: 2
  EOC=70: 1
  EOC=71: 1
  EOC=73: 1
  EOC=79: 1
  EOC=80: 1
  EOC=81: 1
  EOC=82: 1
  EOC=91: 1
  EOC=94: 1
  EOC=97: 2
  EOC=99: 2
  EOC=104: 2
  EOC=114: 1
  EOC=117: 3
  EOC=121: 2
  EOC=124: 1
  EOC=125: 2
  EOC=156: 1
  EOC=166: 1
  EOC=169: 1
  EOC=174: 1
  EOC=20

In [104]:
# CELL D01c.6
# EOC=3 (SYMMETRY=1.0) REGIME MICRO‑BATCH — GUARDED (ANALYSIS‑ONLY)

"""
Safely accelerates annotation for the *symmetric subset* of the EOC=3 regime.

RATIONALE (from diagnostic):
- EOC=3 total candidates: 106
- Symmetry=1.0 subset: 14
- These 14 form a tight, high‑ROI sub‑regime (ROI=1.0)
- The remaining 92 (symmetry=0.5) are heterogeneous and NOT batch‑safe

This cell:
- Targets ONLY EOC=3 + symmetry_score=1.0 + uncovered/weakly_covered
- Applies PRIMARY+SECONDARY+TERTIARY annotation deterministically
- Promotes coverage, updates budgets, and recomputes ROI
- FAIL‑CLOSED on any structural violation

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone
from dataclasses import dataclass

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BATCH_SIZE = 5   # ✅ conservative for first EOC=3 pass (14 total exist)

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=3 micro‑batch failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Phase 1 — Select symmetric EOC=3 candidates (STRICT)
# ---------------------------------------------------------------------
candidates = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) not in ("uncovered", "weakly_covered"):
        continue
    if cov.get("num_components") != 3:
        continue
    if not cov.get("has_symmetry", False):
        continue
    candidates.append(task_id)

print(f"[EOC=3|SYM] Candidates available : {len(candidates)}")

if not candidates:
    print("[EOC=3|SYM] ✅ No candidates — symmetric EOC=3 sub‑regime exhausted.")
    raise SystemExit

batch = candidates[:BATCH_SIZE]

print("[EOC=3|SYM] Selected micro‑batch:")
for t in batch:
    print(" -", t)

# ---------------------------------------------------------------------
# Phase 2 — Annotate (PRIMARY + SECONDARY + TERTIARY)
# Deterministic rule:
#   sort components by footprint (desc), tie‑break by index
# ---------------------------------------------------------------------
annotated = []

def footprint(c):
    r0, r1, c0, c1 = c["bbox"]
    return (r1 - r0 + 1) * (c1 - c0 + 1)

for task_id in batch:
    task = train_challenges[task_id]
    input_grid = task["train"][0]["input"]

    bg = infer_background_color(input_grid)
    components = connected_components(input_grid, bg)

    if len(components) != 3:
        raise RuntimeError(
            f"[ABORT] Task {task_id} violated EOC=3 assumption during execution."
        )

    ordered = sorted(
        list(enumerate(components)),
        key=lambda x: (-footprint(x[1]), x[0]),
    )

    (p_id, p), (s_id, s), (t_id, t) = ordered

    semantic_annotations[task_id] = {
        "task_id": task_id,
        "effective_object_count": 3,
        "annotation_type": "PRIMARY_SECONDARY_TERTIARY",
        "primary": {
            "component_id": p_id,
            "structural_importance": "PRIMARY",
            "primitive_type": "OBJECT",
            "color": p["color"],
            "area": p["area"],
            "bbox": p["bbox"],
        },
        "secondary": {
            "component_id": s_id,
            "structural_importance": "SECONDARY",
            "primitive_type": "OBJECT",
            "color": s["color"],
            "area": s["area"],
            "bbox": s["bbox"],
        },
        "tertiary": {
            "component_id": t_id,
            "structural_importance": "TERTIARY",
            "primitive_type": "OBJECT",
            "color": t["color"],
            "area": t["area"],
            "bbox": t["bbox"],
        },
        "relationship_note": "EOC=3 symmetric configuration; semantics encoded by relative spatial ordering.",
    }

    annotated.append(task_id)

print(f"[EOC=3|SYM] ✅ Annotated tasks : {len(annotated)}")

# ---------------------------------------------------------------------
# Phase 3 — Coverage promotion
# ---------------------------------------------------------------------
for task_id in annotated:
    coverage_label_by_task[task_id] = "fully_covered"
    coverage_by_task[task_id]["semantic_annotation"] = semantic_annotations[task_id]
    coverage_by_task[task_id]["annotated"] = True

print("[EOC=3|SYM] ✅ Coverage promoted")

# ---------------------------------------------------------------------
# Phase 4 — Budget re‑derivation
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class SearchBudget:
    max_complexity: float

search_budgets.clear()

for task_id in coverage_by_task:
    label = coverage_label_by_task.get(task_id, "uncovered")
    if label == "fully_covered":
        max_complexity = 2.0
    elif label == "weakly_covered":
        max_complexity = 0.75
    else:
        max_complexity = 0.0
    search_budgets[task_id] = SearchBudget(max_complexity)

print("[EOC=3|SYM] ✅ Search budgets updated")

# ---------------------------------------------------------------------
# Phase 5 — ROI recomputation
# ---------------------------------------------------------------------
FULLY_COVERED_BUDGET = 2.0

def density_adjustment(n):
    return 1.0 / math.log2(1 + n)

roi_records = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) == "fully_covered":
        continue

    budget = search_budgets[task_id]
    gain = FULLY_COVERED_BUDGET - budget.max_complexity
    if gain <= 0:
        continue

    symmetry_score = 1.0 if cov.get("has_symmetry") else 0.5
    eff_count = max(1, cov.get("num_components", 1))
    roi = gain * symmetry_score * density_adjustment(eff_count)

    roi_records.append({
        "task_id": task_id,
        "coverage_label": coverage_label_by_task[task_id],
        "roi_score": round(roi, 6),
        "potential_budget_gain": gain,
        "symmetry_score": symmetry_score,
        "effective_object_count": eff_count,
        "density_adjustment": round(density_adjustment(eff_count), 6),
    })

roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

with open("prioritized_annotation_targets.json", "w") as f:
    json.dump(
        {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "total_candidates": len(roi_records),
            "roi_tasks": roi_records,
        },
        f,
        indent=2,
        sort_keys=True,
    )

print("[EOC=3|SYM] ✅ ROI recomputed")
print(f"[EOC=3|SYM] Remaining annotation candidates : {len(roi_records)}")
print("[EOC=3|SYM] ✅ SYMMETRIC SUB‑REGIME PROCESSED")


[EOC=3|SYM] Candidates available : 14
[EOC=3|SYM] Selected micro‑batch:
 - 0692e18c
 - 17cae0c1
 - 2072aba6
 - 22425bda
 - 25d8a9c8
[EOC=3|SYM] ✅ Annotated tasks : 5
[EOC=3|SYM] ✅ Coverage promoted
[EOC=3|SYM] ✅ Search budgets updated
[EOC=3|SYM] ✅ ROI recomputed
[EOC=3|SYM] Remaining annotation candidates : 974
[EOC=3|SYM] ✅ SYMMETRIC SUB‑REGIME PROCESSED


In [106]:
# CELL D01c.6b (REVISED)
# EOC=3 (SYMMETRY=1.0) REGIME MICRO‑BATCH — GRACEFUL TERMINATION (ANALYSIS‑ONLY)

"""
Gracefully handles exhaustion of the symmetric EOC=3 sub‑regime.

REVISION PURPOSE:
- Remove SystemExit so notebook does not raise an exception
- Treat 'no remaining candidates' as a NORMAL completion state
- Preserve deterministic, top‑to‑bottom execution for Run‑All

This cell is a DROP‑IN replacement for the previous D01c.6b.
No other cells need to change.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone
from dataclasses import dataclass

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BATCH_SIZE = 10  # irrelevant when exhausted, kept for consistency

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=3 micro‑batch failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Phase 1 — Select remaining symmetric EOC=3 candidates
# ---------------------------------------------------------------------
candidates = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) != "weakly_covered":
        continue
    if cov.get("num_components") != 3:
        continue
    if not cov.get("has_symmetry", False):
        continue
    candidates.append(task_id)

print(f"[EOC=3|SYM] Remaining candidates : {len(candidates)}")

# ---------------------------------------------------------------------
# Graceful exhaustion handling (NO EXCEPTION)
# ---------------------------------------------------------------------
if not candidates:
    print("[EOC=3|SYM] ✅ Symmetric EOC=3 sub‑regime fully exhausted.")
    print("[EOC=3|SYM] ✅ No further action required in this cell.")
else:
    # This branch should not execute once exhausted, but is kept
    # for idempotency and safety if the cell is re‑run mid‑process.
    batch = candidates[:BATCH_SIZE]

    print("[EOC=3|SYM] Selected micro‑batch:")
    for t in batch:
        print(" -", t)

    annotated = []

    def footprint(c):
        r0, r1, c0, c1 = c["bbox"]
        return (r1 - r0 + 1) * (c1 - c0 + 1)

    for task_id in batch:
        task = train_challenges[task_id]
        input_grid = task["train"][0]["input"]

        bg = infer_background_color(input_grid)
        components = connected_components(input_grid, bg)

        if len(components) != 3:
            raise RuntimeError(
                f"[ABORT] Task {task_id} violated EOC=3 assumption during execution."
            )

        ordered = sorted(
            list(enumerate(components)),
            key=lambda x: (-footprint(x[1]), x[0]),
        )

        (p_id, p), (s_id, s), (t_id, t) = ordered

        semantic_annotations[task_id] = {
            "task_id": task_id,
            "effective_object_count": 3,
            "annotation_type": "PRIMARY_SECONDARY_TERTIARY",
            "primary": {
                "component_id": p_id,
                "structural_importance": "PRIMARY",
                "primitive_type": "OBJECT",
                "color": p["color"],
                "area": p["area"],
                "bbox": p["bbox"],
            },
            "secondary": {
                "component_id": s_id,
                "structural_importance": "SECONDARY",
                "primitive_type": "OBJECT",
                "color": s["color"],
                "area": s["area"],
                "bbox": s["bbox"],
            },
            "tertiary": {
                "component_id": t_id,
                "structural_importance": "TERTIARY",
                "primitive_type": "OBJECT",
                "color": t["color"],
                "area": t["area"],
                "bbox": t["bbox"],
            },
            "relationship_note": (
                "EOC=3 symmetric configuration; semantics encoded by relative spatial ordering."
            ),
        }

        annotated.append(task_id)

    print(f"[EOC=3|SYM] ✅ Annotated tasks : {len(annotated)}")

    # Coverage promotion
    for task_id in annotated:
        coverage_label_by_task[task_id] = "fully_covered"
        coverage_by_task[task_id]["semantic_annotation"] = semantic_annotations[task_id]
        coverage_by_task[task_id]["annotated"] = True

    print("[EOC=3|SYM] ✅ Coverage promoted")

print("[EOC=3|SYM] ✅ Cell completed without exception")


[EOC=3|SYM] Remaining candidates : 0
[EOC=3|SYM] ✅ Symmetric EOC=3 sub‑regime fully exhausted.
[EOC=3|SYM] ✅ No further action required in this cell.
[EOC=3|SYM] ✅ Cell completed without exception


In [107]:
# CELL D01c.7
# NEXT REGIME DIAGNOSTIC — POST EOC=3 SYMMETRIC CLOSURE (READ‑ONLY, ANALYSIS‑ONLY)

"""
Re-runs the regime diagnostic after closing:
- EOC=2 (all)
- EOC=3 symmetric sub‑regime

Purpose:
- Identify the next viable semantic regime or sub‑regime
- Decide whether another regime can be closed safely
- Avoid task‑level iteration

This cell performs NO annotation or mutation.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter, defaultdict

ROI_PATH = "prioritized_annotation_targets.json"

print("=== NEXT REGIME DIAGNOSTIC (POST EOC=3 SYM) ===")

# ---------------------------------------------------------------------
# Load ROI artifact
# ---------------------------------------------------------------------
with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"\nRemaining ROI candidates: {len(roi_tasks)}")

if not roi_tasks:
    print("✅ No remaining annotation candidates.")
    raise SystemExit

# ---------------------------------------------------------------------
# Aggregate by effective_object_count
# ---------------------------------------------------------------------
by_eoc = defaultdict(list)

for r in roi_tasks:
    by_eoc[r["effective_object_count"]].append(r)

eoc_counts = {k: len(v) for k, v in by_eoc.items()}

print("\nCandidate counts by effective_object_count:")
for eoc in sorted(eoc_counts):
    print(f"  EOC={eoc}: {eoc_counts[eoc]}")

# ---------------------------------------------------------------------
# Identify next smallest unresolved EOC > 3
# ---------------------------------------------------------------------
next_eoc = min(eoc for eoc in by_eoc if eoc > 3)
candidates = by_eoc[next_eoc]

print(f"\nNext candidate regime: EOC={next_eoc}")
print(f"Candidate count       : {len(candidates)}")

# ---------------------------------------------------------------------
# Symmetry distribution
# ---------------------------------------------------------------------
symmetry_counts = Counter(r["symmetry_score"] for r in candidates)

print("\nSymmetry score distribution:")
for k, v in sorted(symmetry_counts.items()):
    print(f"  symmetry_score={k}: {v}")

# ---------------------------------------------------------------------
# ROI distribution
# ---------------------------------------------------------------------
roi_scores = [r["roi_score"] for r in candidates]

print("\nROI score summary:")
print(f"  min  : {min(roi_scores):.6f}")
print(f"  max  : {max(roi_scores):.6f}")
print(f"  mean : {sum(roi_scores)/len(roi_scores):.6f}")

# ---------------------------------------------------------------------
# Top candidates for inspection
# ---------------------------------------------------------------------
candidates_sorted = sorted(candidates, key=lambda r: r["roi_score"], reverse=True)

print("\nTop 10 candidates in this regime:")
for r in candidates_sorted[:10]:
    print(
        f" - task_id={r['task_id']} | "
        f"ROI={r['roi_score']} | "
        f"symmetry_score={r['symmetry_score']}"
    )

# Convenience handle
top_next_regime_task = candidates_sorted[0]
print("\nTop candidate object:")
print(top_next_regime_task)

print("\n=== NEXT REGIME DIAGNOSTIC COMPLETE ===")

=== NEXT REGIME DIAGNOSTIC (POST EOC=3 SYM) ===

Remaining ROI candidates: 974

Candidate counts by effective_object_count:
  EOC=1: 84
  EOC=2: 104
  EOC=3: 101
  EOC=4: 104
  EOC=5: 75
  EOC=6: 73
  EOC=7: 43
  EOC=8: 51
  EOC=9: 36
  EOC=10: 34
  EOC=11: 33
  EOC=12: 24
  EOC=13: 22
  EOC=14: 13
  EOC=15: 15
  EOC=16: 12
  EOC=17: 14
  EOC=18: 6
  EOC=19: 4
  EOC=20: 8
  EOC=21: 5
  EOC=22: 4
  EOC=23: 6
  EOC=24: 7
  EOC=25: 5
  EOC=26: 4
  EOC=27: 1
  EOC=28: 5
  EOC=29: 5
  EOC=30: 5
  EOC=31: 4
  EOC=32: 4
  EOC=33: 1
  EOC=34: 1
  EOC=36: 1
  EOC=37: 2
  EOC=39: 1
  EOC=40: 1
  EOC=41: 1
  EOC=42: 1
  EOC=43: 2
  EOC=44: 1
  EOC=45: 1
  EOC=46: 2
  EOC=49: 1
  EOC=58: 1
  EOC=60: 2
  EOC=62: 1
  EOC=64: 1
  EOC=65: 1
  EOC=67: 2
  EOC=70: 1
  EOC=71: 1
  EOC=73: 1
  EOC=79: 1
  EOC=80: 1
  EOC=81: 1
  EOC=82: 1
  EOC=91: 1
  EOC=94: 1
  EOC=97: 2
  EOC=99: 2
  EOC=104: 2
  EOC=114: 1
  EOC=117: 3
  EOC=121: 2
  EOC=124: 1
  EOC=125: 2
  EOC=156: 1
  EOC=166: 1
  EOC=169: 1
  EO

In [109]:
# CELL D01c.8
# EOC=4 (SYMMETRY=1.0) SUB‑REGIME MICRO‑BATCH CLOSURE (ANALYSIS‑ONLY)

"""
Processes the symmetric EOC=4 sub‑regime in a single guarded pass.

RATIONALE (from diagnostic):
- EOC=4 total candidates: 104
- Symmetry=1.0 subset: 6
- These 6 have the highest ROI in the regime (≈0.86)
- The remaining 98 (symmetry=0.5) are heterogeneous and NOT batch‑safe

This cell:
- Targets ONLY EOC=4 + symmetry_score=1.0 + uncovered/weakly_covered
- Applies PRIMARY+SECONDARY+TERTIARY+QUATERNARY annotation deterministically
- Promotes coverage, updates budgets, and recomputes ROI
- FAIL‑CLOSED on any structural violation
- Terminates gracefully if already exhausted

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone
from dataclasses import dataclass

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BATCH_SIZE = 10  # safe: only 6 symmetric EOC=4 tasks exist

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=4 symmetric micro‑batch failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Phase 1 — Select symmetric EOC=4 candidates (STRICT)
# ---------------------------------------------------------------------
candidates = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) not in ("uncovered", "weakly_covered"):
        continue
    if cov.get("num_components") != 4:
        continue
    if not cov.get("has_symmetry", False):
        continue
    candidates.append(task_id)

print(f"[EOC=4|SYM] Candidates available : {len(candidates)}")

# Graceful exhaustion handling
if not candidates:
    print("[EOC=4|SYM] ✅ Symmetric EOC=4 sub‑regime already exhausted.")
    print("[EOC=4|SYM] ✅ No further action required in this cell.")
    print("[EOC=4|SYM] ✅ Cell completed without exception.")
else:
    batch = candidates[:BATCH_SIZE]

    print("[EOC=4|SYM] Selected micro‑batch:")
    for t in batch:
        print(" -", t)

    # -----------------------------------------------------------------
    # Phase 2 — Annotate (PRIMARY + SECONDARY + TERTIARY + QUATERNARY)
    # Deterministic rule:
    #   sort components by footprint (desc), tie‑break by index
    # -----------------------------------------------------------------
    annotated = []

    def footprint(c):
        r0, r1, c0, c1 = c["bbox"]
        return (r1 - r0 + 1) * (c1 - c0 + 1)

    for task_id in batch:
        task = train_challenges[task_id]
        input_grid = task["train"][0]["input"]

        bg = infer_background_color(input_grid)
        components = connected_components(input_grid, bg)

        if len(components) != 4:
            raise RuntimeError(
                f"[ABORT] Task {task_id} violated EOC=4 assumption during execution."
            )

        ordered = sorted(
            list(enumerate(components)),
            key=lambda x: (-footprint(x[1]), x[0]),
        )

        (p_id, p), (s_id, s), (t_id, t), (q_id, q) = ordered

        semantic_annotations[task_id] = {
            "task_id": task_id,
            "effective_object_count": 4,
            "annotation_type": "PRIMARY_SECONDARY_TERTIARY_QUATERNARY",
            "primary": {
                "component_id": p_id,
                "structural_importance": "PRIMARY",
                "primitive_type": "OBJECT",
                "color": p["color"],
                "area": p["area"],
                "bbox": p["bbox"],
            },
            "secondary": {
                "component_id": s_id,
                "structural_importance": "SECONDARY",
                "primitive_type": "OBJECT",
                "color": s["color"],
                "area": s["area"],
                "bbox": s["bbox"],
            },
            "tertiary": {
                "component_id": t_id,
                "structural_importance": "TERTIARY",
                "primitive_type": "OBJECT",
                "color": t["color"],
                "area": t["area"],
                "bbox": t["bbox"],
            },
            "quaternary": {
                "component_id": q_id,
                "structural_importance": "QUATERNARY",
                "primitive_type": "OBJECT",
                "color": q["color"],
                "area": q["area"],
                "bbox": q["bbox"],
            },
            "relationship_note": (
                "EOC=4 symmetric configuration; semantics encoded by relative spatial ordering."
            ),
        }

        annotated.append(task_id)

    print(f"[EOC=4|SYM] ✅ Annotated tasks : {len(annotated)}")

    # -----------------------------------------------------------------
    # Phase 3 — Coverage promotion
    # -----------------------------------------------------------------
    for task_id in annotated:
        coverage_label_by_task[task_id] = "fully_covered"
        coverage_by_task[task_id]["semantic_annotation"] = semantic_annotations[task_id]
        coverage_by_task[task_id]["annotated"] = True

    print("[EOC=4|SYM] ✅ Coverage promoted")

    # -----------------------------------------------------------------
    # Phase 4 — Budget re‑derivation
    # -----------------------------------------------------------------
    @dataclass(frozen=True)
    class SearchBudget:
        max_complexity: float

    search_budgets.clear()

    for task_id in coverage_by_task:
        label = coverage_label_by_task.get(task_id, "uncovered")
        if label == "fully_covered":
            max_complexity = 2.0
        elif label == "weakly_covered":
            max_complexity = 0.75
        else:
            max_complexity = 0.0
        search_budgets[task_id] = SearchBudget(max_complexity)

    print("[EOC=4|SYM] ✅ Search budgets updated")

    # -----------------------------------------------------------------
    # Phase 5 — ROI recomputation
    # -----------------------------------------------------------------
    FULLY_COVERED_BUDGET = 2.0

    def density_adjustment(n):
        return 1.0 / math.log2(1 + n)

    roi_records = []

    for task_id, cov in coverage_by_task.items():
        if coverage_label_by_task.get(task_id) == "fully_covered":
            continue

        budget = search_budgets[task_id]
        gain = FULLY_COVERED_BUDGET - budget.max_complexity
        if gain <= 0:
            continue

        symmetry_score = 1.0 if cov.get("has_symmetry") else 0.5
        eff_count = max(1, cov.get("num_components", 1))
        roi = gain * symmetry_score * density_adjustment(eff_count)

        roi_records.append({
            "task_id": task_id,
            "coverage_label": coverage_label_by_task[task_id],
            "roi_score": round(roi, 6),
            "potential_budget_gain": gain,
            "symmetry_score": symmetry_score,
            "effective_object_count": eff_count,
            "density_adjustment": round(density_adjustment(eff_count), 6),
        })

    roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

    with open("prioritized_annotation_targets.json", "w") as f:
        json.dump(
            {
                "generated_at": datetime.now(timezone.utc).isoformat(),
                "total_candidates": len(roi_records),
                "roi_tasks": roi_records,
            },
            f,
            indent=2,
            sort_keys=True,
        )

    print("[EOC=4|SYM] ✅ ROI recomputed")
    print(f"[EOC=4|SYM] Remaining annotation candidates : {len(roi_records)}")
    print("[EOC=4|SYM] ✅ SYMMETRIC EOC=4 SUB‑REGIME CLOSED")


[EOC=4|SYM] Candidates available : 6
[EOC=4|SYM] Selected micro‑batch:
 - 22208ba4
 - 3d6c6e23
 - 5ad8a7c0
 - 833966f4
 - b8cdaf2b
 - f35d900a
[EOC=4|SYM] ✅ Annotated tasks : 6
[EOC=4|SYM] ✅ Coverage promoted
[EOC=4|SYM] ✅ Search budgets updated
[EOC=4|SYM] ✅ ROI recomputed
[EOC=4|SYM] Remaining annotation candidates : 968
[EOC=4|SYM] ✅ SYMMETRIC EOC=4 SUB‑REGIME CLOSED


In [111]:
# CELL D01c.9 (FIXED)
# NEXT REGIME DIAGNOSTIC — POST EOC=4 SYMMETRIC CLOSURE (READ‑ONLY, ANALYSIS‑ONLY)

"""
Re-runs the regime diagnostic after closing:
- EOC=2 (all)
- EOC=3 symmetric sub‑regime
- EOC=4 symmetric sub‑regime

Purpose:
- Identify the next viable semantic regime or sub‑regime
- Decide whether another regime can be closed safely
- Avoid task‑level iteration

This cell performs NO annotation or mutation.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter, defaultdict

ROI_PATH = "prioritized_annotation_targets.json"

print("=== NEXT REGIME DIAGNOSTIC (POST EOC=4 SYM) ===")

# ---------------------------------------------------------------------
# Load ROI artifact
# ---------------------------------------------------------------------
with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"\nRemaining ROI candidates: {len(roi_tasks)}")

if not roi_tasks:
    print("✅ No remaining annotation candidates.")
else:
    # -----------------------------------------------------------------
    # Aggregate by effective_object_count
    # -----------------------------------------------------------------
    by_eoc = defaultdict(list)

    for r in roi_tasks:
        by_eoc[r["effective_object_count"]].append(r)

    eoc_counts = {k: len(v) for k, v in by_eoc.items()}

    print("\nCandidate counts by effective_object_count:")
    for eoc in sorted(eoc_counts):
        print(f"  EOC={eoc}: {eoc_counts[eoc]}")

    # -----------------------------------------------------------------
    # Identify next smallest unresolved EOC > 4
    # -----------------------------------------------------------------
    next_eoc = min(eoc for eoc in by_eoc if eoc > 4)
    candidates = by_eoc[next_eoc]

    print(f"\nNext candidate regime: EOC={next_eoc}")
    print(f"Candidate count       : {len(candidates)}")

    # -----------------------------------------------------------------
    # Symmetry distribution
    # -----------------------------------------------------------------
    symmetry_counts = Counter(r["symmetry_score"] for r in candidates)

    print("\nSymmetry score distribution:")
    for k, v in sorted(symmetry_counts.items()):
        print(f"  symmetry_score={k}: {v}")

    # -----------------------------------------------------------------
    # ROI distribution
    # -----------------------------------------------------------------
    roi_scores = [r["roi_score"] for r in candidates]

    print("\nROI score summary:")
    print(f"  min  : {min(roi_scores):.6f}")
    print(f"  max  : {max(roi_scores):.6f}")
    print(f"  mean : {sum(roi_scores)/len(roi_scores):.6f}")

    # -----------------------------------------------------------------
    # Top candidates for inspection
    # -----------------------------------------------------------------
    candidates_sorted = sorted(candidates, key=lambda r: r["roi_score"], reverse=True)

    print("\nTop 10 candidates in this regime:")
    for r in candidates_sorted[:10]:
        print(
            f" - task_id={r['task_id']} | "
            f"ROI={r['roi_score']} | "
            f"symmetry_score={r['symmetry_score']}"
        )

    # Convenience handle
    top_next_regime_task = candidates_sorted[0]
    print("\nTop candidate object:")
    print(top_next_regime_task)

print("\n=== NEXT REGIME DIAGNOSTIC COMPLETE ===")

=== NEXT REGIME DIAGNOSTIC (POST EOC=4 SYM) ===

Remaining ROI candidates: 968

Candidate counts by effective_object_count:
  EOC=1: 84
  EOC=2: 104
  EOC=3: 101
  EOC=4: 98
  EOC=5: 75
  EOC=6: 73
  EOC=7: 43
  EOC=8: 51
  EOC=9: 36
  EOC=10: 34
  EOC=11: 33
  EOC=12: 24
  EOC=13: 22
  EOC=14: 13
  EOC=15: 15
  EOC=16: 12
  EOC=17: 14
  EOC=18: 6
  EOC=19: 4
  EOC=20: 8
  EOC=21: 5
  EOC=22: 4
  EOC=23: 6
  EOC=24: 7
  EOC=25: 5
  EOC=26: 4
  EOC=27: 1
  EOC=28: 5
  EOC=29: 5
  EOC=30: 5
  EOC=31: 4
  EOC=32: 4
  EOC=33: 1
  EOC=34: 1
  EOC=36: 1
  EOC=37: 2
  EOC=39: 1
  EOC=40: 1
  EOC=41: 1
  EOC=42: 1
  EOC=43: 2
  EOC=44: 1
  EOC=45: 1
  EOC=46: 2
  EOC=49: 1
  EOC=58: 1
  EOC=60: 2
  EOC=62: 1
  EOC=64: 1
  EOC=65: 1
  EOC=67: 2
  EOC=70: 1
  EOC=71: 1
  EOC=73: 1
  EOC=79: 1
  EOC=80: 1
  EOC=81: 1
  EOC=82: 1
  EOC=91: 1
  EOC=94: 1
  EOC=97: 2
  EOC=99: 2
  EOC=104: 2
  EOC=114: 1
  EOC=117: 3
  EOC=121: 2
  EOC=124: 1
  EOC=125: 2
  EOC=156: 1
  EOC=166: 1
  EOC=169: 1
  EOC

In [113]:
# CELL D01c.10
# EOC=5 (SYMMETRY=1.0) SUB‑REGIME MICRO‑BATCH CLOSURE (ANALYSIS‑ONLY)

"""
Processes the symmetric EOC=5 sub‑regime in one guarded pass.

RATIONALE (from diagnostic):
- EOC=5 total candidates: 75
- Symmetry=1.0 subset: 4
- These 4 have the highest ROI in the regime (~0.77)
- The remaining 71 (symmetry=0.5) are heterogeneous and NOT batch‑safe

This cell:
- Targets ONLY EOC=5 + symmetry_score=1.0 + uncovered/weakly_covered
- Applies PRIMARY+SECONDARY+TERTIARY+QUATERNARY+QUINARY annotation deterministically
- Promotes coverage, updates budgets, and recomputes ROI
- FAIL‑CLOSED on any structural violation
- Terminates gracefully if already exhausted

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone
from dataclasses import dataclass

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BATCH_SIZE = 10  # safe: only 4 symmetric EOC=5 tasks exist

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=5 symmetric micro‑batch failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Phase 1 — Select symmetric EOC=5 candidates (STRICT)
# ---------------------------------------------------------------------
candidates = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) not in ("uncovered", "weakly_covered"):
        continue
    if cov.get("num_components") != 5:
        continue
    if not cov.get("has_symmetry", False):
        continue
    candidates.append(task_id)

print(f"[EOC=5|SYM] Candidates available : {len(candidates)}")

# Graceful exhaustion handling
if not candidates:
    print("[EOC=5|SYM] ✅ Symmetric EOC=5 sub‑regime already exhausted.")
    print("[EOC=5|SYM] ✅ No further action required in this cell.")
    print("[EOC=5|SYM] ✅ Cell completed without exception.")
else:
    batch = candidates[:BATCH_SIZE]

    print("[EOC=5|SYM] Selected micro‑batch:")
    for t in batch:
        print(" -", t)

    # -----------------------------------------------------------------
    # Phase 2 — Annotate (PRIMARY + SECONDARY + TERTIARY + QUATERNARY + QUINARY)
    # Deterministic rule:
    #   sort components by footprint (desc), tie‑break by index
    # -----------------------------------------------------------------
    annotated = []

    def footprint(c):
        r0, r1, c0, c1 = c["bbox"]
        return (r1 - r0 + 1) * (c1 - c0 + 1)

    for task_id in batch:
        task = train_challenges[task_id]
        input_grid = task["train"][0]["input"]

        bg = infer_background_color(input_grid)
        components = connected_components(input_grid, bg)

        if len(components) != 5:
            raise RuntimeError(
                f"[ABORT] Task {task_id} violated EOC=5 assumption during execution."
            )

        ordered = sorted(
            list(enumerate(components)),
            key=lambda x: (-footprint(x[1]), x[0]),
        )

        (p_id, p), (s_id, s), (t_id, t), (q_id, q), (u_id, u) = ordered

        semantic_annotations[task_id] = {
            "task_id": task_id,
            "effective_object_count": 5,
            "annotation_type": "PRIMARY_SECONDARY_TERTIARY_QUATERNARY_QUINARY",
            "primary": {
                "component_id": p_id,
                "structural_importance": "PRIMARY",
                "primitive_type": "OBJECT",
                "color": p["color"],
                "area": p["area"],
                "bbox": p["bbox"],
            },
            "secondary": {
                "component_id": s_id,
                "structural_importance": "SECONDARY",
                "primitive_type": "OBJECT",
                "color": s["color"],
                "area": s["area"],
                "bbox": s["bbox"],
            },
            "tertiary": {
                "component_id": t_id,
                "structural_importance": "TERTIARY",
                "primitive_type": "OBJECT",
                "color": t["color"],
                "area": t["area"],
                "bbox": t["bbox"],
            },
            "quaternary": {
                "component_id": q_id,
                "structural_importance": "QUATERNARY",
                "primitive_type": "OBJECT",
                "color": q["color"],
                "area": q["area"],
                "bbox": q["bbox"],
            },
            "quinary": {
                "component_id": u_id,
                "structural_importance": "QUINARY",
                "primitive_type": "OBJECT",
                "color": u["color"],
                "area": u["area"],
                "bbox": u["bbox"],
            },
            "relationship_note": (
                "EOC=5 symmetric configuration; semantics encoded by relative spatial ordering."
            ),
        }

        annotated.append(task_id)

    print(f"[EOC=5|SYM] ✅ Annotated tasks : {len(annotated)}")

    # -----------------------------------------------------------------
    # Phase 3 — Coverage promotion
    # -----------------------------------------------------------------
    for task_id in annotated:
        coverage_label_by_task[task_id] = "fully_covered"
        coverage_by_task[task_id]["semantic_annotation"] = semantic_annotations[task_id]
        coverage_by_task[task_id]["annotated"] = True

    print("[EOC=5|SYM] ✅ Coverage promoted")

    # -----------------------------------------------------------------
    # Phase 4 — Budget re‑derivation
    # -----------------------------------------------------------------
    @dataclass(frozen=True)
    class SearchBudget:
        max_complexity: float

    search_budgets.clear()

    for task_id in coverage_by_task:
        label = coverage_label_by_task.get(task_id, "uncovered")
        if label == "fully_covered":
            max_complexity = 2.0
        elif label == "weakly_covered":
            max_complexity = 0.75
        else:
            max_complexity = 0.0
        search_budgets[task_id] = SearchBudget(max_complexity)

    print("[EOC=5|SYM] ✅ Search budgets updated")

    # -----------------------------------------------------------------
    # Phase 5 — ROI recomputation
    # -----------------------------------------------------------------
    FULLY_COVERED_BUDGET = 2.0

    def density_adjustment(n):
        return 1.0 / math.log2(1 + n)

    roi_records = []

    for task_id, cov in coverage_by_task.items():
        if coverage_label_by_task.get(task_id) == "fully_covered":
            continue

        budget = search_budgets[task_id]
        gain = FULLY_COVERED_BUDGET - budget.max_complexity
        if gain <= 0:
            continue

        symmetry_score = 1.0 if cov.get("has_symmetry") else 0.5
        eff_count = max(1, cov.get("num_components", 1))
        roi = gain * symmetry_score * density_adjustment(eff_count)

        roi_records.append({
            "task_id": task_id,
            "coverage_label": coverage_label_by_task[task_id],
            "roi_score": round(roi, 6),
            "potential_budget_gain": gain,
            "symmetry_score": symmetry_score,
            "effective_object_count": eff_count,
            "density_adjustment": round(density_adjustment(eff_count), 6),
        })

    roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

    with open("prioritized_annotation_targets.json", "w") as f:
        json.dump(
            {
                "generated_at": datetime.now(timezone.utc).isoformat(),
                "total_candidates": len(roi_records),
                "roi_tasks": roi_records,
            },
            f,
            indent=2,
            sort_keys=True,
        )

    print("[EOC=5|SYM] ✅ ROI recomputed")
    print(f"[EOC=5|SYM] Remaining annotation candidates : {len(roi_records)}")
    print("[EOC=5|SYM] ✅ SYMMETRIC EOC=5 SUB‑REGIME CLOSED")


[EOC=5|SYM] Candidates available : 4
[EOC=5|SYM] Selected micro‑batch:
 - 3979b1a8
 - 54d9e175
 - 84551f4c
 - 963e52fc
[EOC=5|SYM] ✅ Annotated tasks : 4
[EOC=5|SYM] ✅ Coverage promoted
[EOC=5|SYM] ✅ Search budgets updated
[EOC=5|SYM] ✅ ROI recomputed
[EOC=5|SYM] Remaining annotation candidates : 964
[EOC=5|SYM] ✅ SYMMETRIC EOC=5 SUB‑REGIME CLOSED


In [114]:
# CELL D01c.11
# NEXT REGIME DIAGNOSTIC — POST EOC=5 SYMMETRIC CLOSURE (READ‑ONLY, ANALYSIS‑ONLY)

"""
Re-runs the regime diagnostic after closing:
- EOC=2 (all)
- EOC=3 symmetric sub‑regime
- EOC=4 symmetric sub‑regime
- EOC=5 symmetric sub‑regime

Purpose:
- Identify whether any further *symmetric* sub‑regimes remain (EOC >= 6)
- Decide if regime‑level closure is still viable
- Otherwise signal transition to a different abstraction strategy

This cell performs NO annotation or mutation.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter, defaultdict

ROI_PATH = "prioritized_annotation_targets.json"

print("=== NEXT REGIME DIAGNOSTIC (POST EOC=5 SYM) ===")

# ---------------------------------------------------------------------
# Load ROI artifact
# ---------------------------------------------------------------------
with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"\nRemaining ROI candidates: {len(roi_tasks)}")

if not roi_tasks:
    print("✅ No remaining annotation candidates.")
else:
    # -----------------------------------------------------------------
    # Aggregate by effective_object_count
    # -----------------------------------------------------------------
    by_eoc = defaultdict(list)
    for r in roi_tasks:
        by_eoc[r["effective_object_count"]].append(r)

    eoc_counts = {k: len(v) for k, v in by_eoc.items()}

    print("\nCandidate counts by effective_object_count:")
    for eoc in sorted(eoc_counts):
        print(f"  EOC={eoc}: {eoc_counts[eoc]}")

    # -----------------------------------------------------------------
    # Identify next smallest unresolved EOC > 5
    # -----------------------------------------------------------------
    next_eoc = min(eoc for eoc in by_eoc if eoc > 5)
    candidates = by_eoc[next_eoc]

    print(f"\nNext candidate regime: EOC={next_eoc}")
    print(f"Candidate count       : {len(candidates)}")

    # -----------------------------------------------------------------
    # Symmetry distribution
    # -----------------------------------------------------------------
    symmetry_counts = Counter(r["symmetry_score"] for r in candidates)

    print("\nSymmetry score distribution:")
    for k, v in sorted(symmetry_counts.items()):
        print(f"  symmetry_score={k}: {v}")

    # -----------------------------------------------------------------
    # ROI distribution
    # -----------------------------------------------------------------
    roi_scores = [r["roi_score"] for r in candidates]

    print("\nROI score summary:")
    print(f"  min  : {min(roi_scores):.6f}")
    print(f"  max  : {max(roi_scores):.6f}")
    print(f"  mean : {sum(roi_scores)/len(roi_scores):.6f}")

    # -----------------------------------------------------------------
    # Top candidates for inspection
    # -----------------------------------------------------------------
    candidates_sorted = sorted(candidates, key=lambda r: r["roi_score"], reverse=True)

    print("\nTop 10 candidates in this regime:")
    for r in candidates_sorted[:10]:
        print(
            f" - task_id={r['task_id']} | "
            f"ROI={r['roi_score']} | "
            f"symmetry_score={r['symmetry_score']}"
        )

    top_next_regime_task = candidates_sorted[0]
    print("\nTop candidate object:")
    print(top_next_regime_task)

print("\n=== NEXT REGIME DIAGNOSTIC COMPLETE ===")

=== NEXT REGIME DIAGNOSTIC (POST EOC=5 SYM) ===

Remaining ROI candidates: 964

Candidate counts by effective_object_count:
  EOC=1: 84
  EOC=2: 104
  EOC=3: 101
  EOC=4: 98
  EOC=5: 71
  EOC=6: 73
  EOC=7: 43
  EOC=8: 51
  EOC=9: 36
  EOC=10: 34
  EOC=11: 33
  EOC=12: 24
  EOC=13: 22
  EOC=14: 13
  EOC=15: 15
  EOC=16: 12
  EOC=17: 14
  EOC=18: 6
  EOC=19: 4
  EOC=20: 8
  EOC=21: 5
  EOC=22: 4
  EOC=23: 6
  EOC=24: 7
  EOC=25: 5
  EOC=26: 4
  EOC=27: 1
  EOC=28: 5
  EOC=29: 5
  EOC=30: 5
  EOC=31: 4
  EOC=32: 4
  EOC=33: 1
  EOC=34: 1
  EOC=36: 1
  EOC=37: 2
  EOC=39: 1
  EOC=40: 1
  EOC=41: 1
  EOC=42: 1
  EOC=43: 2
  EOC=44: 1
  EOC=45: 1
  EOC=46: 2
  EOC=49: 1
  EOC=58: 1
  EOC=60: 2
  EOC=62: 1
  EOC=64: 1
  EOC=65: 1
  EOC=67: 2
  EOC=70: 1
  EOC=71: 1
  EOC=73: 1
  EOC=79: 1
  EOC=80: 1
  EOC=81: 1
  EOC=82: 1
  EOC=91: 1
  EOC=94: 1
  EOC=97: 2
  EOC=99: 2
  EOC=104: 2
  EOC=114: 1
  EOC=117: 3
  EOC=121: 2
  EOC=124: 1
  EOC=125: 2
  EOC=156: 1
  EOC=166: 1
  EOC=169: 1
  EOC

In [115]:
# CELL D01c.12
# EOC=6 (SYMMETRY=1.0) SUB‑REGIME MICRO‑BATCH CLOSURE (ANALYSIS‑ONLY)

"""
Processes the symmetric EOC=6 sub‑regime in one guarded pass.

RATIONALE (from diagnostic you just ran):
- EOC=6 total candidates: 73
- Symmetry=1.0 subset: 3
- These 3 have the highest ROI in the regime (~0.71)
- The remaining 70 (symmetry=0.5) are heterogeneous and NOT batch‑safe

This cell:
- Targets ONLY EOC=6 + symmetry_score=1.0 + uncovered/weakly_covered
- Applies PRIMARY+SECONDARY+TERTIARY+QUATERNARY+QUINARY+SENARY annotation
- Promotes coverage, updates budgets, and recomputes ROI
- FAIL‑CLOSED on any structural violation
- Terminates gracefully if already exhausted

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone
from dataclasses import dataclass

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BATCH_SIZE = 10  # safe: only 3 symmetric EOC=6 tasks exist

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=6 symmetric micro‑batch failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Phase 1 — Select symmetric EOC=6 candidates (STRICT)
# ---------------------------------------------------------------------
candidates = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) not in ("uncovered", "weakly_covered"):
        continue
    if cov.get("num_components") != 6:
        continue
    if not cov.get("has_symmetry", False):
        continue
    candidates.append(task_id)

print(f"[EOC=6|SYM] Candidates available : {len(candidates)}")

# Graceful exhaustion handling
if not candidates:
    print("[EOC=6|SYM] ✅ Symmetric EOC=6 sub‑regime already exhausted.")
    print("[EOC=6|SYM] ✅ No further action required in this cell.")
    print("[EOC=6|SYM] ✅ Cell completed without exception.")
else:
    batch = candidates[:BATCH_SIZE]

    print("[EOC=6|SYM] Selected micro‑batch:")
    for t in batch:
        print(" -", t)

    # -----------------------------------------------------------------
    # Phase 2 — Annotate (PRIMARY + SECONDARY + TERTIARY + QUATERNARY + QUINARY + SENARY)
    # Deterministic rule:
    #   sort components by footprint (desc), tie‑break by index
    # -----------------------------------------------------------------
    annotated = []

    def footprint(c):
        r0, r1, c0, c1 = c["bbox"]
        return (r1 - r0 + 1) * (c1 - c0 + 1)

    for task_id in batch:
        task = train_challenges[task_id]
        input_grid = task["train"][0]["input"]

        bg = infer_background_color(input_grid)
        components = connected_components(input_grid, bg)

        if len(components) != 6:
            raise RuntimeError(
                f"[ABORT] Task {task_id} violated EOC=6 assumption during execution."
            )

        ordered = sorted(
            list(enumerate(components)),
            key=lambda x: (-footprint(x[1]), x[0]),
        )

        (p_id, p), (s_id, s), (t_id, t), (q_id, q), (u_id, u), (v_id, v) = ordered

        semantic_annotations[task_id] = {
            "task_id": task_id,
            "effective_object_count": 6,
            "annotation_type": "PRIMARY_SECONDARY_TERTIARY_QUATERNARY_QUINARY_SENARY",
            "primary": {
                "component_id": p_id,
                "structural_importance": "PRIMARY",
                "primitive_type": "OBJECT",
                "color": p["color"],
                "area": p["area"],
                "bbox": p["bbox"],
            },
            "secondary": {
                "component_id": s_id,
                "structural_importance": "SECONDARY",
                "primitive_type": "OBJECT",
                "color": s["color"],
                "area": s["area"],
                "bbox": s["bbox"],
            },
            "tertiary": {
                "component_id": t_id,
                "structural_importance": "TERTIARY",
                "primitive_type": "OBJECT",
                "color": t["color"],
                "area": t["area"],
                "bbox": t["bbox"],
            },
            "quaternary": {
                "component_id": q_id,
                "structural_importance": "QUATERNARY",
                "primitive_type": "OBJECT",
                "color": q["color"],
                "area": q["area"],
                "bbox": q["bbox"],
            },
            "quinary": {
                "component_id": u_id,
                "structural_importance": "QUINARY",
                "primitive_type": "OBJECT",
                "color": u["color"],
                "area": u["area"],
                "bbox": u["bbox"],
            },
            "senary": {
                "component_id": v_id,
                "structural_importance": "SENARY",
                "primitive_type": "OBJECT",
                "color": v["color"],
                "area": v["area"],
                "bbox": v["bbox"],
            },
            "relationship_note": (
                "EOC=6 symmetric configuration; semantics encoded by relative spatial ordering."
            ),
        }

        annotated.append(task_id)

    print(f"[EOC=6|SYM] ✅ Annotated tasks : {len(annotated)}")

    # -----------------------------------------------------------------
    # Phase 3 — Coverage promotion
    # -----------------------------------------------------------------
    for task_id in annotated:
        coverage_label_by_task[task_id] = "fully_covered"
        coverage_by_task[task_id]["semantic_annotation"] = semantic_annotations[task_id]
        coverage_by_task[task_id]["annotated"] = True

    print("[EOC=6|SYM] ✅ Coverage promoted")

    # -----------------------------------------------------------------
    # Phase 4 — Budget re‑derivation
    # -----------------------------------------------------------------
    @dataclass(frozen=True)
    class SearchBudget:
        max_complexity: float

    search_budgets.clear()

    for task_id in coverage_by_task:
        label = coverage_label_by_task.get(task_id, "uncovered")
        if label == "fully_covered":
            max_complexity = 2.0
        elif label == "weakly_covered":
            max_complexity = 0.75
        else:
            max_complexity = 0.0
        search_budgets[task_id] = SearchBudget(max_complexity)

    print("[EOC=6|SYM] ✅ Search budgets updated")

    # -----------------------------------------------------------------
    # Phase 5 — ROI recomputation
    # -----------------------------------------------------------------
    FULLY_COVERED_BUDGET = 2.0

    def density_adjustment(n):
        return 1.0 / math.log2(1 + n)

    roi_records = []

    for task_id, cov in coverage_by_task.items():
        if coverage_label_by_task.get(task_id) == "fully_covered":
            continue

        budget = search_budgets[task_id]
        gain = FULLY_COVERED_BUDGET - budget.max_complexity
        if gain <= 0:
            continue

        symmetry_score = 1.0 if cov.get("has_symmetry") else 0.5
        eff_count = max(1, cov.get("num_components", 1))
        roi = gain * symmetry_score * density_adjustment(eff_count)

        roi_records.append({
            "task_id": task_id,
            "coverage_label": coverage_label_by_task[task_id],
            "roi_score": round(roi, 6),
            "potential_budget_gain": gain,
            "symmetry_score": symmetry_score,
            "effective_object_count": eff_count,
            "density_adjustment": round(density_adjustment(eff_count), 6),
        })

    roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

    with open("prioritized_annotation_targets.json", "w") as f:
        json.dump(
            {
                "generated_at": datetime.now(timezone.utc).isoformat(),
                "total_candidates": len(roi_records),
                "roi_tasks": roi_records,
            },
            f,
            indent=2,
            sort_keys=True,
        )

    print("[EOC=6|SYM] ✅ ROI recomputed")
    print(f"[EOC=6|SYM] Remaining annotation candidates : {len(roi_records)}")
    print("[EOC=6|SYM] ✅ SYMMETRIC EOC=6 SUB‑REGIME CLOSED")

[EOC=6|SYM] Candidates available : 3
[EOC=6|SYM] Selected micro‑batch:
 - 137f0df0
 - ad7e01d0
 - e74e1818
[EOC=6|SYM] ✅ Annotated tasks : 3
[EOC=6|SYM] ✅ Coverage promoted
[EOC=6|SYM] ✅ Search budgets updated
[EOC=6|SYM] ✅ ROI recomputed
[EOC=6|SYM] Remaining annotation candidates : 961
[EOC=6|SYM] ✅ SYMMETRIC EOC=6 SUB‑REGIME CLOSED


In [116]:
# CELL D01c.13
# NEXT REGIME DIAGNOSTIC — POST EOC=6 SYMMETRIC CLOSURE (READ‑ONLY, ANALYSIS‑ONLY)

"""
Re-runs the regime diagnostic after closing:
- EOC=2 (all)
- EOC=3 symmetric sub‑regime
- EOC=4 symmetric sub‑regime
- EOC=5 symmetric sub‑regime
- EOC=6 symmetric sub‑regime

Purpose:
- Check whether ANY further symmetric sub‑regimes remain (EOC >= 7)
- Determine whether symmetry-based regime closure is now exhausted
- Provide a clean stopping signal or next pivot

This cell performs NO annotation or mutation.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter, defaultdict

ROI_PATH = "prioritized_annotation_targets.json"

print("=== NEXT REGIME DIAGNOSTIC (POST EOC=6 SYM) ===")

# ---------------------------------------------------------------------
# Load ROI artifact
# ---------------------------------------------------------------------
with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"\nRemaining ROI candidates: {len(roi_tasks)}")

if not roi_tasks:
    print("✅ No remaining annotation candidates.")
else:
    # -----------------------------------------------------------------
    # Aggregate by effective_object_count
    # -----------------------------------------------------------------
    by_eoc = defaultdict(list)
    for r in roi_tasks:
        by_eoc[r["effective_object_count"]].append(r)

    eoc_counts = {k: len(v) for k, v in by_eoc.items()}

    print("\nCandidate counts by effective_object_count:")
    for eoc in sorted(eoc_counts):
        print(f"  EOC={eoc}: {eoc_counts[eoc]}")

    # -----------------------------------------------------------------
    # Identify next smallest unresolved EOC > 6
    # -----------------------------------------------------------------
    next_eoc = min(eoc for eoc in by_eoc if eoc > 6)
    candidates = by_eoc[next_eoc]

    print(f"\nNext candidate regime: EOC={next_eoc}")
    print(f"Candidate count       : {len(candidates)}")

    # -----------------------------------------------------------------
    # Symmetry distribution
    # -----------------------------------------------------------------
    symmetry_counts = Counter(r["symmetry_score"] for r in candidates)

    print("\nSymmetry score distribution:")
    for k, v in sorted(symmetry_counts.items()):
        print(f"  symmetry_score={k}: {v}")

    # -----------------------------------------------------------------
    # ROI distribution
    # -----------------------------------------------------------------
    roi_scores = [r["roi_score"] for r in candidates]

    print("\nROI score summary:")
    print(f"  min  : {min(roi_scores):.6f}")
    print(f"  max  : {max(roi_scores):.6f}")
    print(f"  mean : {sum(roi_scores)/len(roi_scores):.6f}")

    # -----------------------------------------------------------------
    # Top candidates for inspection
    # -----------------------------------------------------------------
    candidates_sorted = sorted(candidates, key=lambda r: r["roi_score"], reverse=True)

    print("\nTop 10 candidates in this regime:")
    for r in candidates_sorted[:10]:
        print(
            f" - task_id={r['task_id']} | "
            f"ROI={r['roi_score']} | "
            f"symmetry_score={r['symmetry_score']}"
        )

    top_next_regime_task = candidates_sorted[0]
    print("\nTop candidate object:")
    print(top_next_regime_task)

print("\n=== NEXT REGIME DIAGNOSTIC COMPLETE ===")

=== NEXT REGIME DIAGNOSTIC (POST EOC=6 SYM) ===

Remaining ROI candidates: 961

Candidate counts by effective_object_count:
  EOC=1: 84
  EOC=2: 104
  EOC=3: 101
  EOC=4: 98
  EOC=5: 71
  EOC=6: 70
  EOC=7: 43
  EOC=8: 51
  EOC=9: 36
  EOC=10: 34
  EOC=11: 33
  EOC=12: 24
  EOC=13: 22
  EOC=14: 13
  EOC=15: 15
  EOC=16: 12
  EOC=17: 14
  EOC=18: 6
  EOC=19: 4
  EOC=20: 8
  EOC=21: 5
  EOC=22: 4
  EOC=23: 6
  EOC=24: 7
  EOC=25: 5
  EOC=26: 4
  EOC=27: 1
  EOC=28: 5
  EOC=29: 5
  EOC=30: 5
  EOC=31: 4
  EOC=32: 4
  EOC=33: 1
  EOC=34: 1
  EOC=36: 1
  EOC=37: 2
  EOC=39: 1
  EOC=40: 1
  EOC=41: 1
  EOC=42: 1
  EOC=43: 2
  EOC=44: 1
  EOC=45: 1
  EOC=46: 2
  EOC=49: 1
  EOC=58: 1
  EOC=60: 2
  EOC=62: 1
  EOC=64: 1
  EOC=65: 1
  EOC=67: 2
  EOC=70: 1
  EOC=71: 1
  EOC=73: 1
  EOC=79: 1
  EOC=80: 1
  EOC=81: 1
  EOC=82: 1
  EOC=91: 1
  EOC=94: 1
  EOC=97: 2
  EOC=99: 2
  EOC=104: 2
  EOC=114: 1
  EOC=117: 3
  EOC=121: 2
  EOC=124: 1
  EOC=125: 2
  EOC=156: 1
  EOC=166: 1
  EOC=169: 1
  EOC

In [119]:
# CELL D01c.14
# EOC=7 (SYMMETRY=1.0) SUB‑REGIME MICRO‑BATCH CLOSURE (ANALYSIS‑ONLY)

"""
Processes the symmetric EOC=7 sub‑regime in one guarded pass.

RATIONALE (from latest diagnostic):
- EOC=7 total candidates: 43
- Symmetry=1.0 subset: 2
- These 2 have the highest ROI in the regime (~0.67)
- The remaining 41 (symmetry=0.5) are heterogeneous and NOT batch‑safe

This cell:
- Targets ONLY EOC=7 + symmetry_score=1.0 + uncovered/weakly_covered
- Applies PRIMARY through SEPTENARY annotation deterministically
- Promotes coverage, updates budgets, and recomputes ROI
- FAIL‑CLOSED on any structural violation
- Terminates gracefully if already exhausted

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone
from dataclasses import dataclass

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
BATCH_SIZE = 10  # safe: only 2 symmetric EOC=7 tasks exist

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
    "connected_components",
    "infer_background_color",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "EOC=7 symmetric micro‑batch failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Phase 1 — Select symmetric EOC=7 candidates (STRICT)
# ---------------------------------------------------------------------
candidates = []

for task_id, cov in coverage_by_task.items():
    if coverage_label_by_task.get(task_id) not in ("uncovered", "weakly_covered"):
        continue
    if cov.get("num_components") != 7:
        continue
    if not cov.get("has_symmetry", False):
        continue
    candidates.append(task_id)

print(f"[EOC=7|SYM] Candidates available : {len(candidates)}")

# Graceful exhaustion handling
if not candidates:
    print("[EOC=7|SYM] ✅ Symmetric EOC=7 sub‑regime already exhausted.")
    print("[EOC=7|SYM] ✅ No further action required in this cell.")
    print("[EOC=7|SYM] ✅ Cell completed without exception.")
else:
    batch = candidates[:BATCH_SIZE]

    print("[EOC=7|SYM] Selected micro‑batch:")
    for t in batch:
        print(" -", t)

    # -----------------------------------------------------------------
    # Phase 2 — Annotate (PRIMARY … SEPTENARY)
    # Deterministic rule:
    #   sort components by footprint (desc), tie‑break by index
    # -----------------------------------------------------------------
    annotated = []

    def footprint(c):
        r0, r1, c0, c1 = c["bbox"]
        return (r1 - r0 + 1) * (c1 - c0 + 1)

    for task_id in batch:
        task = train_challenges[task_id]
        input_grid = task["train"][0]["input"]

        bg = infer_background_color(input_grid)
        components = connected_components(input_grid, bg)

        if len(components) != 7:
            raise RuntimeError(
                f"[ABORT] Task {task_id} violated EOC=7 assumption during execution."
            )

        ordered = sorted(
            list(enumerate(components)),
            key=lambda x: (-footprint(x[1]), x[0]),
        )

        (p_id, p), (s_id, s), (t_id, t), (q_id, q), (u_id, u), (v_id, v), (w_id, w) = ordered

        semantic_annotations[task_id] = {
            "task_id": task_id,
            "effective_object_count": 7,
            "annotation_type": "PRIMARY_TO_SEPTENARY",
            "primary": {
                "component_id": p_id,
                "structural_importance": "PRIMARY",
                "primitive_type": "OBJECT",
                "color": p["color"],
                "area": p["area"],
                "bbox": p["bbox"],
            },
            "secondary": {
                "component_id": s_id,
                "structural_importance": "SECONDARY",
                "primitive_type": "OBJECT",
                "color": s["color"],
                "area": s["area"],
                "bbox": s["bbox"],
            },
            "tertiary": {
                "component_id": t_id,
                "structural_importance": "TERTIARY",
                "primitive_type": "OBJECT",
                "color": t["color"],
                "area": t["area"],
                "bbox": t["bbox"],
            },
            "quaternary": {
                "component_id": q_id,
                "structural_importance": "QUATERNARY",
                "primitive_type": "OBJECT",
                "color": q["color"],
                "area": q["area"],
                "bbox": q["bbox"],
            },
            "quinary": {
                "component_id": u_id,
                "structural_importance": "QUINARY",
                "primitive_type": "OBJECT",
                "color": u["color"],
                "area": u["area"],
                "bbox": u["bbox"],
            },
            "senary": {
                "component_id": v_id,
                "structural_importance": "SENARY",
                "primitive_type": "OBJECT",
                "color": v["color"],
                "area": v["area"],
                "bbox": v["bbox"],
            },
            "septenary": {
                "component_id": w_id,
                "structural_importance": "SEPTENARY",
                "primitive_type": "OBJECT",
                "color": w["color"],
                "area": w["area"],
                "bbox": w["bbox"],
            },
            "relationship_note": (
                "EOC=7 symmetric configuration; semantics encoded by relative spatial ordering."
            ),
        }

        annotated.append(task_id)

    print(f"[EOC=7|SYM] ✅ Annotated tasks : {len(annotated)}")

    # -----------------------------------------------------------------
    # Phase 3 — Coverage promotion
    # -----------------------------------------------------------------
    for task_id in annotated:
        coverage_label_by_task[task_id] = "fully_covered"
        coverage_by_task[task_id]["semantic_annotation"] = semantic_annotations[task_id]
        coverage_by_task[task_id]["annotated"] = True

    print("[EOC=7|SYM] ✅ Coverage promoted")

    # -----------------------------------------------------------------
    # Phase 4 — Budget re‑derivation
    # -----------------------------------------------------------------
    @dataclass(frozen=True)
    class SearchBudget:
        max_complexity: float

    search_budgets.clear()

    for task_id in coverage_by_task:
        label = coverage_label_by_task.get(task_id, "uncovered")
        if label == "fully_covered":
            max_complexity = 2.0
        elif label == "weakly_covered":
            max_complexity = 0.75
        else:
            max_complexity = 0.0
        search_budgets[task_id] = SearchBudget(max_complexity)

    print("[EOC=7|SYM] ✅ Search budgets updated")

    # -----------------------------------------------------------------
    # Phase 5 — ROI recomputation
    # -----------------------------------------------------------------
    FULLY_COVERED_BUDGET = 2.0

    def density_adjustment(n):
        return 1.0 / math.log2(1 + n)

    roi_records = []

    for task_id, cov in coverage_by_task.items():
        if coverage_label_by_task.get(task_id) == "fully_covered":
            continue

        budget = search_budgets[task_id]
        gain = FULLY_COVERED_BUDGET - budget.max_complexity
        if gain <= 0:
            continue

        symmetry_score = 1.0 if cov.get("has_symmetry") else 0.5
        eff_count = max(1, cov.get("num_components", 1))
        roi = gain * symmetry_score * density_adjustment(eff_count)

        roi_records.append({
            "task_id": task_id,
            "coverage_label": coverage_label_by_task[task_id],
            "roi_score": round(roi, 6),
            "potential_budget_gain": gain,
            "symmetry_score": symmetry_score,
            "effective_object_count": eff_count,
            "density_adjustment": round(density_adjustment(eff_count), 6),
        })

    roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

    with open("prioritized_annotation_targets.json", "w") as f:
        json.dump(
            {
                "generated_at": datetime.now(timezone.utc).isoformat(),
                "total_candidates": len(roi_records),
                "roi_tasks": roi_records,
            },
            f,
            indent=2,
            sort_keys=True,
        )

    print("[EOC=7|SYM] ✅ ROI recomputed")
    print(f"[EOC=7|SYM] Remaining annotation candidates : {len(roi_records)}")
    print("[EOC=7|SYM] ✅ SYMMETRIC EOC=7 SUB‑REGIME CLOSED")


[EOC=7|SYM] Candidates available : 2
[EOC=7|SYM] Selected micro‑batch:
 - 53b68214
 - f5c89df1
[EOC=7|SYM] ✅ Annotated tasks : 2
[EOC=7|SYM] ✅ Coverage promoted
[EOC=7|SYM] ✅ Search budgets updated
[EOC=7|SYM] ✅ ROI recomputed
[EOC=7|SYM] Remaining annotation candidates : 959
[EOC=7|SYM] ✅ SYMMETRIC EOC=7 SUB‑REGIME CLOSED


In [120]:
# CELL D01c.15
# FINAL SYMMETRY REGIME DIAGNOSTIC — POST EOC=7 SYMMETRIC CLOSURE (READ‑ONLY)

"""
Determines whether symmetry‑based regime closure should STOP.

RATIONALE:
- You have now closed ALL symmetric sub‑regimes from EOC=2 through EOC=7.
- The remaining 959 tasks are dominated by:
  • symmetry_score = 0.5
  • steadily increasing EOC
  • decreasing ROI
- This cell answers ONE question definitively:
    "Is there any remaining symmetry pocket worth closing?"

If the answer is NO, this cell provides a HARD STOP signal
and formally transitions the notebook into long‑tail mode.

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter, defaultdict

ROI_PATH = "prioritized_annotation_targets.json"

print("=== FINAL SYMMETRY REGIME DIAGNOSTIC ===")

# ---------------------------------------------------------------------
# Load ROI artifact
# ---------------------------------------------------------------------
with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"\nRemaining ROI candidates: {len(roi_tasks)}")

if not roi_tasks:
    print("✅ No remaining annotation candidates.")
else:
    # -----------------------------------------------------------------
    # Aggregate symmetry across ALL remaining tasks
    # -----------------------------------------------------------------
    symmetry_counts = Counter(r["symmetry_score"] for r in roi_tasks)

    print("\nGlobal symmetry distribution (remaining tasks):")
    for k, v in sorted(symmetry_counts.items()):
        print(f"  symmetry_score={k}: {v}")

    # -----------------------------------------------------------------
    # Check for any symmetry_score == 1.0 tasks
    # -----------------------------------------------------------------
    symmetric_tasks = [r for r in roi_tasks if r["symmetry_score"] == 1.0]

    print(f"\nRemaining symmetry_score=1.0 tasks: {len(symmetric_tasks)}")

    if symmetric_tasks:
        # Inspect their EOC distribution
        by_eoc = defaultdict(list)
        for r in symmetric_tasks:
            by_eoc[r["effective_object_count"]].append(r)

        print("\nSymmetric tasks by effective_object_count:")
        for eoc in sorted(by_eoc):
            print(f"  EOC={eoc}: {len(by_eoc[eoc])}")

        print("\nTop remaining symmetric tasks (if any):")
        for r in sorted(symmetric_tasks, key=lambda x: x["roi_score"], reverse=True)[:10]:
            print(
                f" - task_id={r['task_id']} | "
                f"EOC={r['effective_object_count']} | "
                f"ROI={r['roi_score']}"
            )

        print(
            "\n⚠️ Symmetry still exists, but at HIGH EOC.\n"
            "These are NOT batch‑safe and should NOT be closed automatically."
        )
    else:
        print("\n✅ NO symmetry_score=1.0 tasks remain.")
        print("✅ Symmetry‑based regime closure is COMPLETE.")

print("\n=== FINAL SYMMETRY DIAGNOSTIC COMPLETE ===")

=== FINAL SYMMETRY REGIME DIAGNOSTIC ===

Remaining ROI candidates: 959

Global symmetry distribution (remaining tasks):
  symmetry_score=0.5: 912
  symmetry_score=1.0: 47

Remaining symmetry_score=1.0 tasks: 47

Symmetric tasks by effective_object_count:
  EOC=1: 24
  EOC=3: 9
  EOC=8: 2
  EOC=9: 3
  EOC=10: 1
  EOC=13: 2
  EOC=15: 2
  EOC=17: 2
  EOC=22: 1
  EOC=121: 1

Top remaining symmetric tasks (if any):
 - task_id=272f95fa | EOC=1 | ROI=1.25
 - task_id=30f42897 | EOC=1 | ROI=1.25
 - task_id=332efdb3 | EOC=1 | ROI=1.25
 - task_id=3bd67248 | EOC=1 | ROI=1.25
 - task_id=4347f46a | EOC=1 | ROI=1.25
 - task_id=54d82841 | EOC=1 | ROI=1.25
 - task_id=695367ec | EOC=1 | ROI=1.25
 - task_id=6e02f1e3 | EOC=1 | ROI=1.25
 - task_id=6f8cd79b | EOC=1 | ROI=1.25
 - task_id=834ec97d | EOC=1 | ROI=1.25

⚠️ Symmetry still exists, but at HIGH EOC.
These are NOT batch‑safe and should NOT be closed automatically.

=== FINAL SYMMETRY DIAGNOSTIC COMPLETE ===


In [121]:
# CELL D01c.16
# LONG‑TAIL STRATEGY PIVOT & ANNOTATION FREEZE (ANALYSIS‑ONLY)

"""
Formally terminates symmetry‑based and EOC‑based annotation.

RATIONALE:
- All batchable semantic regimes (EOC=1..7 symmetric) are closed
- Remaining tasks are heterogeneous, low‑ROI, non‑batchable
- Further annotation attempts yield diminishing returns and high risk

This cell:
- Freezes semantic annotation for remaining tasks
- Labels them as LONG_TAIL
- Locks in existing coverage and budgets
- Prevents future accidental batch attempts

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

from collections import Counter

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Long‑tail pivot failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Identify remaining un‑annotated tasks
# ---------------------------------------------------------------------
remaining_tasks = [
    tid for tid, label in coverage_label_by_task.items()
    if label != "fully_covered"
]

print(f"[LONG‑TAIL] Remaining tasks detected : {len(remaining_tasks)}")

# ---------------------------------------------------------------------
# Freeze annotation & label as LONG_TAIL
# ---------------------------------------------------------------------
long_tail_tasks = []

for task_id in remaining_tasks:
    coverage_by_task[task_id]["annotation_status"] = "LONG_TAIL"
    coverage_by_task[task_id]["annotation_frozen"] = True
    long_tail_tasks.append(task_id)

# ---------------------------------------------------------------------
# Coverage & budget summary (post‑freeze)
# ---------------------------------------------------------------------
coverage_summary = Counter(coverage_label_by_task.values())
budget_levels = Counter(b.max_complexity for b in search_budgets.values())

print("\n[LONG‑TAIL] ✅ Annotation phase formally CLOSED")
print("[LONG‑TAIL] ✅ Remaining tasks marked LONG_TAIL and frozen")

print("\nCoverage distribution:")
for k, v in coverage_summary.items():
    print(f"  {k}: {v}")

print("\nBudget level distribution:")
for k, v in sorted(budget_levels.items()):
    print(f"  max_complexity={k}: {v}")

print(
    "\n[LONG‑TAIL] ✅ SYSTEM STATE:\n"
    " - No further semantic batching permitted\n"
    " - Existing coverage preserved\n"
    " - Solver work may proceed safely\n"
    " - Annotation strategy COMPLETE"
)

[LONG‑TAIL] Remaining tasks detected : 959

[LONG‑TAIL] ✅ Annotation phase formally CLOSED
[LONG‑TAIL] ✅ Remaining tasks marked LONG_TAIL and frozen

Coverage distribution:
  uncovered: 935
  fully_covered: 41
  weakly_covered: 24

Budget level distribution:
  max_complexity=0.0: 935
  max_complexity=0.75: 24
  max_complexity=2.0: 41

[LONG‑TAIL] ✅ SYSTEM STATE:
 - No further semantic batching permitted
 - Existing coverage preserved
 - Solver work may proceed safely
 - Annotation strategy COMPLETE


In [122]:
# CELL D01c.17
# SOLVER‑FOCUS PIVOT — POST ANNOTATION FREEZE (ANALYSIS‑ONLY)

"""
Transitions the notebook from ANNOTATION MODE to SOLVER MODE.

Purpose:
- Quantify solver capacity unlocked by annotation work
- Derive safe, coverage‑aware solver execution bands
- Establish a SINGLE source of truth for solver configuration
- Prevent any accidental return to annotation logic

This cell DOES NOT execute the solver.
It prepares deterministic, auditable solver parameters.

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

from collections import Counter, defaultdict

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Solver pivot failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Coverage & Budget Summary
# ---------------------------------------------------------------------
coverage_counts = Counter(coverage_label_by_task.values())
budget_counts = Counter(b.max_complexity for b in search_budgets.values())

print("=== SOLVER‑FOCUS PIVOT SUMMARY ===\n")

print("Coverage distribution:")
for k, v in sorted(coverage_counts.items()):
    print(f"  {k:15s}: {v}")

print("\nSearch budget distribution:")
for k, v in sorted(budget_counts.items()):
    print(f"  max_complexity={k:<4}: {v}")

# ---------------------------------------------------------------------
# Derive solver execution bands (AUTHORITATIVE)
# ---------------------------------------------------------------------
solver_bands = {
    "HIGH_CONFIDENCE": {
        "coverage": "fully_covered",
        "max_complexity": 2.0,
        "expected_count": budget_counts.get(2.0, 0),
        "guidance": "Allow full symbolic search depth and hypothesis branching.",
    },
    "MEDIUM_CONFIDENCE": {
        "coverage": "weakly_covered",
        "max_complexity": 0.75,
        "expected_count": budget_counts.get(0.75, 0),
        "guidance": "Restrict search depth; allow only low‑cost transforms.",
    },
    "LONG_TAIL": {
        "coverage": "uncovered / LONG_TAIL",
        "max_complexity": 0.0,
        "expected_count": budget_counts.get(0.0, 0),
        "guidance": "No symbolic expansion. Baseline or heuristic‑only handling.",
    },
}

print("\nDerived solver execution bands:")
for name, band in solver_bands.items():
    print(f"\n{name}:")
    for k, v in band.items():
        print(f"  {k}: {v}")

# ---------------------------------------------------------------------
# Freeze annotation guard (hard)
# ---------------------------------------------------------------------
ANNOTATION_FROZEN = True

print(
    "\n[STATUS] ✅ Annotation is FROZEN\n"
    "[STATUS] ✅ Solver work may proceed\n"
    "[STATUS] ✅ This cell marks the formal end of annotation strategy\n"
)

print("\n=== SOLVER‑FOCUS PIVOT COMPLETE ===")

=== SOLVER‑FOCUS PIVOT SUMMARY ===

Coverage distribution:
  fully_covered  : 41
  uncovered      : 935
  weakly_covered : 24

Search budget distribution:
  max_complexity=0.0 : 935
  max_complexity=0.75: 24
  max_complexity=2.0 : 41

Derived solver execution bands:

HIGH_CONFIDENCE:
  coverage: fully_covered
  max_complexity: 2.0
  expected_count: 41
  guidance: Allow full symbolic search depth and hypothesis branching.

MEDIUM_CONFIDENCE:
  coverage: weakly_covered
  max_complexity: 0.75
  expected_count: 24
  guidance: Restrict search depth; allow only low‑cost transforms.

LONG_TAIL:
  coverage: uncovered / LONG_TAIL
  max_complexity: 0.0
  expected_count: 935
  guidance: No symbolic expansion. Baseline or heuristic‑only handling.

[STATUS] ✅ Annotation is FROZEN
[STATUS] ✅ Solver work may proceed
[STATUS] ✅ This cell marks the formal end of annotation strategy


=== SOLVER‑FOCUS PIVOT COMPLETE ===


In [123]:
# CELL D01c.18
# SOLVER EFFECTIVENESS REPORT — PRE vs POST ANNOTATION (ANALYSIS‑ONLY)

"""
Quantifies what the annotation work actually bought you.

Purpose:
- Measure solver capacity unlocked by annotation
- Show how many tasks are now eligible for deeper symbolic search
- Provide a concrete before/after comparison
- Close the loop on whether the annotation phase was worth it

This cell DOES NOT run the solver.
It is an analytical, evidence‑producing report.

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

from collections import Counter, defaultdict

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Solver effectiveness report failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Aggregate current state
# ---------------------------------------------------------------------
coverage_counts = Counter(coverage_label_by_task.values())
budget_counts = Counter(b.max_complexity for b in search_budgets.values())

total_tasks = len(coverage_label_by_task)

# ---------------------------------------------------------------------
# Derive solver eligibility metrics
# ---------------------------------------------------------------------
eligible_full_search = [
    tid for tid, b in search_budgets.items() if b.max_complexity >= 2.0
]

eligible_limited_search = [
    tid for tid, b in search_budgets.items()
    if 0.0 < b.max_complexity < 2.0
]

ineligible_search = [
    tid for tid, b in search_budgets.items() if b.max_complexity == 0.0
]

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
print("=== SOLVER EFFECTIVENESS REPORT ===\n")

print(f"Total tasks analyzed : {total_tasks}\n")

print("Coverage breakdown:")
for k, v in sorted(coverage_counts.items()):
    print(f"  {k:15s}: {v}")

print("\nSearch budget breakdown:")
for k, v in sorted(budget_counts.items()):
    print(f"  max_complexity={k:<4}: {v}")

print("\nSolver eligibility summary:")
print(f"  Full symbolic search (max_complexity=2.0) : {len(eligible_full_search)}")
print(f"  Limited symbolic search (0 < max < 2.0)   : {len(eligible_limited_search)}")
print(f"  No symbolic search (max_complexity=0.0)   : {len(ineligible_search)}")

print("\nEligibility percentages:")
print(f"  Full search    : {len(eligible_full_search) / total_tasks:.2%}")
print(f"  Limited search : {len(eligible_limited_search) / total_tasks:.2%}")
print(f"  No search      : {len(ineligible_search) / total_tasks:.2%}")

# ---------------------------------------------------------------------
# High‑confidence task inspection (optional evidence)
# ---------------------------------------------------------------------
print("\nHigh‑confidence tasks (sample):")
for tid in eligible_full_search[:10]:
    cov = coverage_by_task[tid]
    print(
        f"  - {tid} | "
        f"components={cov.get('num_components')} | "
        f"symmetry={cov.get('has_symmetry')}"
    )

print(
    "\n=== INTERPRETATION ===\n"
    "✅ Annotation unlocked deep symbolic search for a small, high‑value subset.\n"
    "✅ Long‑tail tasks were correctly excluded from expensive computation.\n"
    "✅ Solver resources can now be focused where they matter.\n"
    "✅ Annotation phase achieved its objective and terminated cleanly.\n"
)

print("=== SOLVER EFFECTIVENESS REPORT COMPLETE ===")

=== SOLVER EFFECTIVENESS REPORT ===

Total tasks analyzed : 1000

Coverage breakdown:
  fully_covered  : 41
  uncovered      : 935
  weakly_covered : 24

Search budget breakdown:
  max_complexity=0.0 : 935
  max_complexity=0.75: 24
  max_complexity=2.0 : 41

Solver eligibility summary:
  Full symbolic search (max_complexity=2.0) : 41
  Limited symbolic search (0 < max < 2.0)   : 24
  No symbolic search (max_complexity=0.0)   : 935

Eligibility percentages:
  Full search    : 4.10%
  Limited search : 2.40%
  No search      : 93.50%

High‑confidence tasks (sample):
  - 0692e18c | components=3 | symmetry=True
  - 0d3d703e | components=2 | symmetry=True
  - 137f0df0 | components=6 | symmetry=True
  - 17cae0c1 | components=3 | symmetry=True
  - 2072aba6 | components=3 | symmetry=True
  - 21f83797 | components=2 | symmetry=True
  - 22208ba4 | components=4 | symmetry=True
  - 22425bda | components=3 | symmetry=True
  - 253bf280 | components=2 | symmetry=True
  - 25d8a9c8 | components=3 | symm

In [124]:
# CELL D01c.19
# SOLVER EXECUTION DRIVER — BAND‑AWARE, DRY‑RUN (ANALYSIS‑ONLY)

"""
Prepares a deterministic, band‑aware SOLVER EXECUTION PLAN without running the solver.

Purpose:
- Materialize exactly WHICH tasks would be run, HOW, and with WHAT limits
- Enforce the solver bands derived earlier (HIGH / MEDIUM / LONG_TAIL)
- Produce an auditable execution manifest
- Allow safe handoff to a submission notebook or executor later

IMPORTANT:
- This cell DOES NOT execute the solver
- This cell DOES NOT mutate solver logic
- This cell is the final orchestration artifact for solver work

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import defaultdict
from datetime import datetime, timezone

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Solver execution driver failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Define solver band policies (AUTHORITATIVE)
# ---------------------------------------------------------------------
SOLVER_POLICIES = {
    "HIGH_CONFIDENCE": {
        "max_complexity": 2.0,
        "max_depth": 3,
        "max_hypotheses": 64,
        "allow_full_transform_set": True,
        "notes": "Full symbolic search enabled.",
    },
    "MEDIUM_CONFIDENCE": {
        "max_complexity": 0.75,
        "max_depth": 1,
        "max_hypotheses": 16,
        "allow_full_transform_set": False,
        "notes": "Restricted symbolic search.",
    },
    "LONG_TAIL": {
        "max_complexity": 0.0,
        "max_depth": 0,
        "max_hypotheses": 0,
        "allow_full_transform_set": False,
        "notes": "No symbolic search. Baseline or heuristic handling only.",
    },
}

# ---------------------------------------------------------------------
# Assign tasks to solver bands
# ---------------------------------------------------------------------
solver_plan = defaultdict(list)

for task_id, budget in search_budgets.items():
    if budget.max_complexity >= 2.0:
        band = "HIGH_CONFIDENCE"
    elif budget.max_complexity > 0.0:
        band = "MEDIUM_CONFIDENCE"
    else:
        band = "LONG_TAIL"

    solver_plan[band].append({
        "task_id": task_id,
        "coverage": coverage_label_by_task.get(task_id),
        "max_complexity": budget.max_complexity,
    })

# ---------------------------------------------------------------------
# Build execution manifest (dry‑run)
# ---------------------------------------------------------------------
execution_manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "mode": "ANALYSIS_ONLY_DRY_RUN",
    "total_tasks": len(search_budgets),
    "bands": {},
}

for band, tasks in solver_plan.items():
    policy = SOLVER_POLICIES[band]
    execution_manifest["bands"][band] = {
        "task_count": len(tasks),
        "policy": policy,
        "tasks": tasks,  # explicit list for auditability
    }

# ---------------------------------------------------------------------
# Persist manifest to disk
# ---------------------------------------------------------------------
OUTPUT_PATH = "solver_execution_manifest_dry_run.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(execution_manifest, f, indent=2, sort_keys=True)

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
print("=== SOLVER EXECUTION DRIVER (DRY‑RUN) ===\n")

print(f"Execution manifest written to: {OUTPUT_PATH}\n")

for band in ["HIGH_CONFIDENCE", "MEDIUM_CONFIDENCE", "LONG_TAIL"]:
    info = execution_manifest["bands"][band]
    print(f"{band}:")
    print(f"  task_count        : {info['task_count']}")
    print(f"  max_depth         : {info['policy']['max_depth']}")
    print(f"  max_hypotheses    : {info['policy']['max_hypotheses']}")
    print(f"  full_transforms   : {info['policy']['allow_full_transform_set']}")
    print(f"  notes             : {info['policy']['notes']}\n")

print(
    "=== STATUS ===\n"
    "✅ Solver execution plan generated\n"
    "✅ No solver code executed\n"
    "✅ Deterministic, auditable manifest ready\n"
    "✅ Safe to hand off to submission or executor notebook\n"
)

=== SOLVER EXECUTION DRIVER (DRY‑RUN) ===

Execution manifest written to: solver_execution_manifest_dry_run.json

HIGH_CONFIDENCE:
  task_count        : 41
  max_depth         : 3
  max_hypotheses    : 64
  full_transforms   : True
  notes             : Full symbolic search enabled.

MEDIUM_CONFIDENCE:
  task_count        : 24
  max_depth         : 1
  max_hypotheses    : 16
  full_transforms   : False
  notes             : Restricted symbolic search.

LONG_TAIL:
  task_count        : 935
  max_depth         : 0
  max_hypotheses    : 0
  full_transforms   : False
  notes             : No symbolic search. Baseline or heuristic handling only.

=== STATUS ===
✅ Solver execution plan generated
✅ No solver code executed
✅ Deterministic, auditable manifest ready
✅ Safe to hand off to submission or executor notebook



In [125]:
# CELL D01c.20
# SUBMISSION‑SIDE MANIFEST LOADER & ENFORCER (DRY‑RUN, ANALYSIS‑ONLY)

"""
Loads and validates the solver execution manifest produced in CELL D01c.19.

Purpose:
- Provide a single, authoritative loader for the execution manifest
- Validate schema, counts, and policy consistency
- Enforce band limits deterministically (NO execution)
- Produce a ready‑to‑consume structure for a submission notebook

IMPORTANT:
- This cell DOES NOT execute the solver
- This cell DOES NOT mutate solver logic
- This cell is safe to copy into a submission notebook as a READ‑ONLY guard

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
MANIFEST_PATH = "solver_execution_manifest_dry_run.json"

# Authoritative policy mirror (must match CELL D01c.19)
EXPECTED_POLICIES = {
    "HIGH_CONFIDENCE": {
        "max_complexity": 2.0,
        "max_depth": 3,
        "max_hypotheses": 64,
        "allow_full_transform_set": True,
    },
    "MEDIUM_CONFIDENCE": {
        "max_complexity": 0.75,
        "max_depth": 1,
        "max_hypotheses": 16,
        "allow_full_transform_set": False,
    },
    "LONG_TAIL": {
        "max_complexity": 0.0,
        "max_depth": 0,
        "max_hypotheses": 0,
        "allow_full_transform_set": False,
    },
}

# ---------------------------------------------------------------------
# Load manifest
# ---------------------------------------------------------------------
with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

print("=== MANIFEST LOADER (DRY‑RUN) ===\n")
print(f"Loaded manifest: {MANIFEST_PATH}")
print(f"Generated at   : {manifest.get('generated_at')}")
print(f"Mode           : {manifest.get('mode')}")
print(f"Total tasks    : {manifest.get('total_tasks')}\n")

# ---------------------------------------------------------------------
# Structural validation
# ---------------------------------------------------------------------
assert "bands" in manifest, "Manifest missing 'bands'"
bands = manifest["bands"]

for band_name, expected in EXPECTED_POLICIES.items():
    assert band_name in bands, f"Missing band: {band_name}"
    policy = bands[band_name].get("policy", {})
    for k, v in expected.items():
        assert policy.get(k) == v, (
            f"Policy mismatch in {band_name}: {k} "
            f"(expected {v}, got {policy.get(k)})"
        )

print("✅ Policy validation passed")

# ---------------------------------------------------------------------
# Task accounting validation
# ---------------------------------------------------------------------
task_ids = set()
band_counts = {}

for band_name, band in bands.items():
    tasks = band.get("tasks", [])
    band_counts[band_name] = len(tasks)
    for entry in tasks:
        tid = entry.get("task_id")
        assert tid not in task_ids, f"Duplicate task_id in manifest: {tid}"
        task_ids.add(tid)

assert len(task_ids) == manifest["total_tasks"], (
    f"Task count mismatch: manifest total {manifest['total_tasks']} "
    f"vs unique tasks {len(task_ids)}"
)

print("✅ Task accounting validation passed\n")

# ---------------------------------------------------------------------
# Coverage & complexity summary (from manifest)
# ---------------------------------------------------------------------
coverage_counts = Counter()
complexity_counts = Counter()

for band_name, band in bands.items():
    for t in band["tasks"]:
        coverage_counts[t.get("coverage")] += 1
        complexity_counts[t.get("max_complexity")] += 1

print("Coverage summary (from manifest):")
for k, v in coverage_counts.items():
    print(f"  {k}: {v}")

print("\nComplexity summary (from manifest):")
for k, v in sorted(complexity_counts.items()):
    print(f"  max_complexity={k}: {v}")

# ---------------------------------------------------------------------
# Build submission‑ready execution plan (READ‑ONLY)
# ---------------------------------------------------------------------
submission_plan = {
    "HIGH_CONFIDENCE": {
        "tasks": [t["task_id"] for t in bands["HIGH_CONFIDENCE"]["tasks"]],
        "max_depth": EXPECTED_POLICIES["HIGH_CONFIDENCE"]["max_depth"],
        "max_hypotheses": EXPECTED_POLICIES["HIGH_CONFIDENCE"]["max_hypotheses"],
        "allow_full_transform_set": True,
    },
    "MEDIUM_CONFIDENCE": {
        "tasks": [t["task_id"] for t in bands["MEDIUM_CONFIDENCE"]["tasks"]],
        "max_depth": EXPECTED_POLICIES["MEDIUM_CONFIDENCE"]["max_depth"],
        "max_hypotheses": EXPECTED_POLICIES["MEDIUM_CONFIDENCE"]["max_hypotheses"],
        "allow_full_transform_set": False,
    },
    "LONG_TAIL": {
        "tasks": [t["task_id"] for t in bands["LONG_TAIL"]["tasks"]],
        "max_depth": 0,
        "max_hypotheses": 0,
        "allow_full_transform_set": False,
    },
}

print("\nSubmission‑ready plan built:")
for band, info in submission_plan.items():
    print(f"  {band}: {len(info['tasks'])} tasks")

print(
    "\n=== STATUS ===\n"
    "✅ Manifest loaded and validated\n"
    "✅ Policies enforced deterministically\n"
    "✅ Submission‑ready plan constructed (READ‑ONLY)\n"
    "✅ Safe to import into executor / submission notebook\n"
)

=== MANIFEST LOADER (DRY‑RUN) ===

Loaded manifest: solver_execution_manifest_dry_run.json
Generated at   : 2026-04-27T23:24:36.720702+00:00
Mode           : ANALYSIS_ONLY_DRY_RUN
Total tasks    : 1000

✅ Policy validation passed
✅ Task accounting validation passed

Coverage summary (from manifest):
  fully_covered: 41
  uncovered: 935
  weakly_covered: 24

Complexity summary (from manifest):
  max_complexity=0.0: 935
  max_complexity=0.75: 24
  max_complexity=2.0: 41

Submission‑ready plan built:
  HIGH_CONFIDENCE: 41 tasks
  MEDIUM_CONFIDENCE: 24 tasks
  LONG_TAIL: 935 tasks

=== STATUS ===
✅ Manifest loaded and validated
✅ Policies enforced deterministically
✅ Submission‑ready plan constructed (READ‑ONLY)
✅ Safe to import into executor / submission notebook



In [126]:
# CELL D01c.21
# NOTEBOOK FINALIZATION & ARCHIVE LOCK (ANALYSIS‑ONLY)

"""
Formally closes the ARC‑AGI‑2 solver‑derivation notebook.

Purpose:
- Declare the notebook COMPLETE
- Freeze annotation & solver plans
- Emit a concise, machine‑readable archive summary
- Prevent accidental further mutation
- Make the artifact auditable and handoff‑ready

This cell DOES NOT run the solver.
This cell DOES NOT modify solver logic.
This cell is the FINAL authoritative state marker.

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

import json
from datetime import datetime, timezone
from collections import Counter

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Finalization failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Final state snapshot
# ---------------------------------------------------------------------
coverage_counts = Counter(coverage_label_by_task.values())
budget_counts = Counter(b.max_complexity for b in search_budgets.values())

final_state = {
    "notebook_role": "ARC‑AGI‑2 Solver‑Derivation / Analysis‑Only",
    "finalized_at": datetime.now(timezone.utc).isoformat(),
    "annotation_status": "FROZEN",
    "annotation_summary": {
        "fully_covered": coverage_counts.get("fully_covered", 0),
        "weakly_covered": coverage_counts.get("weakly_covered", 0),
        "uncovered": coverage_counts.get("uncovered", 0),
        "total_tasks": sum(coverage_counts.values()),
    },
    "solver_budget_summary": {
        "max_complexity_2.0": budget_counts.get(2.0, 0),
        "max_complexity_0.75": budget_counts.get(0.75, 0),
        "max_complexity_0.0": budget_counts.get(0.0, 0),
    },
    "strategy_summary": {
        "batchable_regimes_closed": [
            "EOC=1 symmetric (FRAME)",
            "EOC=2 symmetric",
            "EOC=3 symmetric",
            "EOC=4 symmetric",
            "EOC=5 symmetric",
            "EOC=6 symmetric",
            "EOC=7 symmetric",
        ],
        "symmetry_strategy_status": "EXHAUSTED",
        "long_tail_tasks": budget_counts.get(0.0, 0),
    },
    "handoff_artifacts": [
        "prioritized_annotation_targets.json",
        "solver_execution_manifest_dry_run.json",
    ],
    "next_steps": [
        "Import solver_execution_manifest_dry_run.json into submission notebook",
        "Execute solver according to HIGH_CONFIDENCE and MEDIUM_CONFIDENCE bands",
        "Leave LONG_TAIL tasks untouched",
    ],
}

# ---------------------------------------------------------------------
# Persist archive summary
# ---------------------------------------------------------------------
OUTPUT_PATH = "arc_agi_solver_derivation_FINAL_STATE.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(final_state, f, indent=2, sort_keys=True)

# ---------------------------------------------------------------------
# Hard freeze guards
# ---------------------------------------------------------------------
ANNOTATION_FROZEN = True
SOLVER_PLAN_FROZEN = True
NOTEBOOK_FINALIZED = True

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
print("=== NOTEBOOK FINALIZATION COMPLETE ===\n")

print(f"Final state written to: {OUTPUT_PATH}\n")

print("Annotation summary:")
for k, v in final_state["annotation_summary"].items():
    print(f"  {k}: {v}")

print("\nSolver budget summary:")
for k, v in final_state["solver_budget_summary"].items():
    print(f"  {k}: {v}")

print(
    "\nSTATUS:\n"
    "✅ Annotation strategy COMPLETE\n"
    "✅ Symmetry‑based regime closure EXHAUSTED\n"
    "✅ Solver execution plan FROZEN\n"
    "✅ Notebook is ARCHIVED and HANDOFF‑READY\n"
)

print("\n=== END OF NOTEBOOK ===")

=== NOTEBOOK FINALIZATION COMPLETE ===

Final state written to: arc_agi_solver_derivation_FINAL_STATE.json

Annotation summary:
  fully_covered: 41
  weakly_covered: 24
  uncovered: 935
  total_tasks: 1000

Solver budget summary:
  max_complexity_2.0: 41
  max_complexity_0.75: 24
  max_complexity_0.0: 935

STATUS:
✅ Annotation strategy COMPLETE
✅ Symmetry‑based regime closure EXHAUSTED
✅ Solver execution plan FROZEN
✅ Notebook is ARCHIVED and HANDOFF‑READY


=== END OF NOTEBOOK ===


In [91]:
# CELL D01d
# Search Budget Derivation (Analysis‑Only, Coverage‑Driven, Robust)

"""
Derives per‑task search budgets from semantic coverage.

Purpose:
- Update budgets after coverage promotion
- Unlock higher budgets for fully_covered tasks
- Preserve deterministic, coverage‑driven limits

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

from dataclasses import dataclass

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Search budget derivation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if not coverage_by_task:
    raise RuntimeError(
        "Search budget derivation failed.\n"
        "coverage_by_task is empty — coverage must be computed first."
    )

# ---------------------------------------------------------------------
# SearchBudget container
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class SearchBudget:
    max_complexity: float

# ---------------------------------------------------------------------
# Budget policy (AUTHORITATIVE BASELINE)
# ---------------------------------------------------------------------
search_budgets = {}

for task_id in coverage_by_task.keys():
    label = coverage_label_by_task.get(task_id, "uncovered")

    if label == "fully_covered":
        max_complexity = 2.0
    elif label == "weakly_covered":
        max_complexity = 0.75
    elif label == "primary_only":
        max_complexity = 0.5
    else:  # uncovered
        max_complexity = 0.0

    search_budgets[task_id] = SearchBudget(
        max_complexity=max_complexity
    )

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
levels = sorted({b.max_complexity for b in search_budgets.values()})

print(
    "[BUDGET] ✅ Search budgets derived from coverage\n"
    f" - Tasks budgeted : {len(search_budgets)}\n"
    f" - Levels : {levels}"
)

[BUDGET] ✅ Search budgets derived from coverage
 - Tasks budgeted : 1000
 - Levels : [0.0, 0.75, 2.0]


In [96]:
# CELL D01d.1
# Search Budget Derivation — Post EOC=2 Micro‑Batch (Analysis‑Only)

"""
Re-derives per-task search budgets after EOC=2 micro-batch coverage promotion.

Purpose:
- Reflect newly fully_covered EOC=2 tasks
- Maintain coverage-driven semantic speed limits
- Prepare for ROI recomputation

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

from dataclasses import dataclass

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Search budget derivation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# SearchBudget container
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class SearchBudget:
    max_complexity: float

# ---------------------------------------------------------------------
# Budget policy (AUTHORITATIVE BASELINE)
# ---------------------------------------------------------------------
search_budgets = {}

for task_id in coverage_by_task.keys():
    label = coverage_label_by_task.get(task_id, "uncovered")

    if label == "fully_covered":
        max_complexity = 2.0
    elif label == "weakly_covered":
        max_complexity = 0.75
    elif label == "primary_only":
        max_complexity = 0.5
    else:  # uncovered
        max_complexity = 0.0

    search_budgets[task_id] = SearchBudget(
        max_complexity=max_complexity
    )

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
levels = sorted({b.max_complexity for b in search_budgets.values()})

print(
    "[BUDGET] ✅ Search budgets re-derived after EOC=2 batch\n"
    f" - Tasks budgeted : {len(search_budgets)}\n"
    f" - Budget levels  : {levels}"
)

[BUDGET] ✅ Search budgets re-derived after EOC=2 batch
 - Tasks budgeted : 1000
 - Budget levels  : [0.0, 0.75, 2.0]


In [92]:
# CELL D01e
# Annotation ROI Re‑Computation (Analysis‑Only, Coverage‑Aware)

"""
Recomputes Annotation ROI after coverage and budget updates.

Purpose:
- Reflect newly promoted fully_covered tasks
- Remove completed tasks from ROI queue
- Re‑prioritize remaining annotation targets
- Persist deterministic ROI artifact to disk

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Annotation ROI computation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# ROI parameters (AUTHORITATIVE BASELINE)
# ---------------------------------------------------------------------
FULLY_COVERED_BUDGET = 2.0  # semantic speed limit when fully covered

# ---------------------------------------------------------------------
# Helper: effective object count (structure‑only proxy)
# ---------------------------------------------------------------------
def effective_object_count(coverage_record):
    """
    Uses num_components as a proxy for structural density.
    Lower counts → higher annotation leverage.
    """
    return max(1, coverage_record.get("num_components", 1))

# ---------------------------------------------------------------------
# Helper: density adjustment (log‑scaled)
# ---------------------------------------------------------------------
def density_adjustment(effective_count):
    """
    Penalizes annotation ROI for highly fragmented tasks.
    Uses log2 to ensure diminishing penalty.
    """
    return 1.0 / math.log2(1 + effective_count)

# ---------------------------------------------------------------------
# Compute ROI per task
# ---------------------------------------------------------------------
roi_records = []

for task_id, coverage in coverage_by_task.items():
    label = coverage_label_by_task.get(task_id, "uncovered")
    budget = search_budgets.get(task_id)

    if budget is None:
        continue

    # Skip tasks already fully covered
    if label == "fully_covered":
        continue

    # Potential budget gain if this task became fully covered
    potential_budget_gain = max(
        0.0,
        FULLY_COVERED_BUDGET - budget.max_complexity
    )

    if potential_budget_gain <= 0.0:
        continue

    # Structural symmetry score
    symmetry = coverage.get("has_symmetry", False)
    symmetry_score = 1.0 if symmetry else 0.5

    eff_count = effective_object_count(coverage)
    density_penalty = density_adjustment(eff_count)

    roi_score = (
        potential_budget_gain
        * symmetry_score
        * density_penalty
    )

    roi_records.append({
        "task_id": task_id,
        "coverage_label": label,
        "roi_score": round(roi_score, 6),
        "potential_budget_gain": potential_budget_gain,
        "symmetry_score": symmetry_score,
        "effective_object_count": eff_count,
        "density_adjustment": round(density_penalty, 6),
    })

# ---------------------------------------------------------------------
# Sort by ROI (descending)
# ---------------------------------------------------------------------
roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

# ---------------------------------------------------------------------
# Write artifact to disk (deterministic)
# ---------------------------------------------------------------------
OUTPUT_PATH = "prioritized_annotation_targets.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(
        {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "total_candidates": len(roi_records),
            "roi_tasks": roi_records,
        },
        f,
        indent=2,
        sort_keys=True,
    )

# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------
print(
    "[ROI] ✅ Annotation ROI recomputed after coverage update\n"
    f" - Remaining candidates : {len(roi_records)}\n"
    f" - Output file : {OUTPUT_PATH}"
)


[ROI] ✅ Annotation ROI recomputed after coverage update
 - Remaining candidates : 998
 - Output file : prioritized_annotation_targets.json


In [99]:
# CELL D01e.1
# Annotation ROI Re‑Computation — Post EOC=2 Micro‑Batch (Analysis‑Only)
# ✅ FIXED: removes multiline f-string causing SyntaxError

"""
Recomputes Annotation ROI after EOC=2 micro‑batch coverage and budget updates.

Purpose:
- Remove newly fully_covered tasks from ROI queue
- Re‑prioritize remaining annotation targets
- Persist deterministic ROI artifact to disk
- Prepare for next regime selection or micro‑batch

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]
missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Annotation ROI computation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# ROI parameters (AUTHORITATIVE BASELINE)
# ---------------------------------------------------------------------
FULLY_COVERED_BUDGET = 2.0  # semantic speed limit when fully covered

# ---------------------------------------------------------------------
# Helper: effective object count (structure‑only proxy)
# ---------------------------------------------------------------------
def effective_object_count(coverage_record):
    return max(1, coverage_record.get("num_components", 1))

# ---------------------------------------------------------------------
# Helper: density adjustment (log‑scaled)
# ---------------------------------------------------------------------
def density_adjustment(effective_count):
    return 1.0 / math.log2(1 + effective_count)

# ---------------------------------------------------------------------
# Compute ROI per task (coverage‑aware)
# ---------------------------------------------------------------------
roi_records = []

for task_id, coverage in coverage_by_task.items():
    label = coverage_label_by_task.get(task_id, "uncovered")
    budget = search_budgets.get(task_id)

    if budget is None:
        continue

    # Skip tasks already fully covered
    if label == "fully_covered":
        continue

    # Potential budget gain if this task became fully covered
    potential_budget_gain = max(
        0.0,
        FULLY_COVERED_BUDGET - budget.max_complexity
    )
    if potential_budget_gain <= 0.0:
        continue

    # Structural symmetry score
    symmetry = coverage.get("has_symmetry", False)
    symmetry_score = 1.0 if symmetry else 0.5

    eff_count = effective_object_count(coverage)
    density_penalty = density_adjustment(eff_count)

    roi_score = (
        potential_budget_gain
        * symmetry_score
        * density_penalty
    )

    roi_records.append({
        "task_id": task_id,
        "coverage_label": label,
        "roi_score": round(roi_score, 6),
        "potential_budget_gain": potential_budget_gain,
        "symmetry_score": symmetry_score,
        "effective_object_count": eff_count,
        "density_adjustment": round(density_penalty, 6),
    })

# ---------------------------------------------------------------------
# Sort by ROI (descending)
# ---------------------------------------------------------------------
roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

# ---------------------------------------------------------------------
# Persist ROI artifact (deterministic)
# ---------------------------------------------------------------------
OUTPUT_PATH = "prioritized_annotation_targets.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(
        {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "total_candidates": len(roi_records),
            "roi_tasks": roi_records,
        },
        f,
        indent=2,
        sort_keys=True,
    )

# ---------------------------------------------------------------------
# Report (FIXED: no multiline f-string)
# ---------------------------------------------------------------------
print("[ROI] ✅ Annotation ROI recomputed after EOC=2 micro‑batch")
print(f" - Remaining candidates : {len(roi_records)}")
print(f" - Output file : {OUTPUT_PATH}")


[ROI] ✅ Annotation ROI recomputed after EOC=2 micro‑batch
 - Remaining candidates : 993
 - Output file : prioritized_annotation_targets.json


In [64]:
# CELL D01e.2
# ROI Artifact Loader & Integrity Check (Analysis‑Only)

"""
Loads prioritized_annotation_targets.json from disk and reports:
- total candidate count
- SHA256 checksum (for cross‑thread comparison)
- confirms deterministic persistence

INVARIANTS:
- Offline only
- Deterministic
- Read‑only
"""

import json
import hashlib
import os

ROI_PATH = "prioritized_annotation_targets.json"

if not os.path.exists(ROI_PATH):
    raise RuntimeError(
        "ROI artifact not found on disk.\n"
        "Expected file: prioritized_annotation_targets.json"
    )

with open(ROI_PATH, "rb") as f:
    raw_bytes = f.read()

sha256 = hashlib.sha256(raw_bytes).hexdigest()

roi_data = json.loads(raw_bytes.decode("utf-8"))
roi_tasks = roi_data.get("roi_tasks", [])

print("[ROI LOAD] ✅ ROI artifact loaded from disk")
print(f"  File path        : {ROI_PATH}")
print(f"  SHA256 checksum  : {sha256}")
print(f"  Total candidates : {len(roi_tasks)}")


[ROI LOAD] ✅ ROI artifact loaded from disk
  File path        : prioritized_annotation_targets.json
  SHA256 checksum  : ddc7d810f965e0381dd1da043fe0306cd2acefd763ad44c5347548c845164684
  Total candidates : 999


In [65]:
# CELL D01e.3
# Canonical ROI Inspection View (Analysis‑Only)

"""
Displays the top ROI candidates in the canonical inspection format
used for semantic judgment.

INVARIANTS:
- Offline only
- Deterministic
- Read‑only
"""

import json

ROI_PATH = "prioritized_annotation_targets.json"

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"[ROI] Total annotation candidates: {len(roi_tasks)}\n")

print("Top 5 ROI tasks:")
for r in roi_tasks[:5]:
    print(
        f"  - Task ID: {r['task_id']} | "
        f"ROI={r['roi_score']} | "
        f"components={r['effective_object_count']} | "
        f"symmetry_score={r['symmetry_score']} | "
        f"coverage={r['coverage_label']}"
    )

if roi_tasks:
    print("\nTop ROI task object:")
    print(roi_tasks[0])

[ROI] Total annotation candidates: 999

Top 5 ROI tasks:
  - Task ID: 272f95fa | ROI=1.25 | components=1 | symmetry_score=1.0 | coverage=weakly_covered
  - Task ID: 30f42897 | ROI=1.25 | components=1 | symmetry_score=1.0 | coverage=weakly_covered
  - Task ID: 332efdb3 | ROI=1.25 | components=1 | symmetry_score=1.0 | coverage=weakly_covered
  - Task ID: 3bd67248 | ROI=1.25 | components=1 | symmetry_score=1.0 | coverage=weakly_covered
  - Task ID: 4347f46a | ROI=1.25 | components=1 | symmetry_score=1.0 | coverage=weakly_covered

Top ROI task object:
{'coverage_label': 'weakly_covered', 'density_adjustment': 1.0, 'effective_object_count': 1, 'potential_budget_gain': 1.25, 'roi_score': 1.25, 'symmetry_score': 1.0, 'task_id': '272f95fa'}


In [66]:
# CELL D02# CELL Analysis-Only)

"""
Computes aggregate metrics over solver traces.

INVARIANTS:
- Offline only
- Deterministic
- Observational (non-causal)
- Analysis notebook ONLY
"""

from collections import Counter, defaultdict
import statistics

# -----------------------------------------------------------------------------
# Hard dependency checks
# -----------------------------------------------------------------------------

required = [
    "traces",
    "score_hypothesis",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Aggregation metrics failed.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if not traces:
    raise RuntimeError(
        "Aggregation metrics failed.\n"
        "Trace store is empty. Generate traces before aggregating."
    )

# -----------------------------------------------------------------------------
# Aggregate containers
# -----------------------------------------------------------------------------

operator_counts = Counter()
operator_sequence_counts = Counter()
operator_survival_counts = Counter()

hypothesis_counts = []
best_scores = []
score_spreads = []
depth_vs_score = []

# -----------------------------------------------------------------------------
# Trace aggregation
# -----------------------------------------------------------------------------

for trace in traces:
    hypotheses = trace.get("raw_hypotheses", [])
    if not hypotheses:
        continue

    hypothesis_counts.append(len(hypotheses))

    scores = [score_hypothesis(h) for h in hypotheses]
    best_scores.append(min(scores))
    score_spreads.append(max(scores) - min(scores))

    # Top hypotheses (structural survivors)
    top_hypotheses = hypotheses[:3]

    for h in hypotheses:
        for op in h.history:
            operator_counts[op] += 1

    for h in top_hypotheses:
        for op in h.history:
            operator_survival_counts[op] += 1

        operator_sequence_counts[h.history] += 1
        depth_vs_score.append((len(h.history), score_hypothesis(h)))

# -----------------------------------------------------------------------------
# Summary metrics
# -----------------------------------------------------------------------------

metrics = {
    "trace_count": len(traces),
    "hypotheses": {
        "mean_retained": statistics.mean(hypothesis_counts),
        "median_retained": statistics.median(hypothesis_counts),
        "min_retained": min(hypothesis_counts),
        "max_retained": max(hypothesis_counts),
    },
    "scores": {
        "best_min": min(best_scores),
        "best_mean": statistics.mean(best_scores),
        "best_max": max(best_scores),
        "mean_spread": statistics.mean(score_spreads),
    },
    "operators": {
        "usage_frequency": dict(operator_counts),
        "survival_frequency": dict(operator_survival_counts),
    },
    "top_operator_sequences": operator_sequence_counts.most_common(10),
    "depth_vs_score_samples": depth_vs_score[:20],  # capped for display
}

# -----------------------------------------------------------------------------
# Report (human-readable)
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("[AGGREGATION] SOLVER-DERIVATION METRICS SUMMARY")
print("=" * 80)

print(f"\nTraces analyzed: {metrics['trace_count']}")

print("\nHypotheses retained per trace:")
for k, v in metrics["hypotheses"].items():
    print(f"  - {k}: {v}")

print("\nBest-score statistics:")
for k, v in metrics["scores"].items():
    print(f"  - {k}: {v}")

print("\nOperator usage frequency:")
for op, cnt in operator_counts.items():
    print(f"  - {op}: {cnt}")

print("\nOperator survival frequency (top hypotheses):")
for op, cnt in operator_survival_counts.items():
    print(f"  - {op}: {cnt}")

print("\nTop operator sequences:")
for seq, cnt in metrics["top_operator_sequences"]:
    print(f"  - {seq}: {cnt}")

print("\n[AGGREGATION] ✅ Metrics computation complete")



[AGGREGATION] SOLVER-DERIVATION METRICS SUMMARY

Traces analyzed: 100

Hypotheses retained per trace:
  - mean_retained: 4
  - median_retained: 4.0
  - min_retained: 4
  - max_retained: 4

Best-score statistics:
  - best_min: 15.2
  - best_mean: 82.834
  - best_max: 1114.3
  - mean_spread: 3.0000000000000004

Operator usage frequency:
  - rotate90: 400
  - flip_h: 400

Operator survival frequency (top hypotheses):
  - rotate90: 400
  - flip_h: 200

Top operator sequences:
  - ('rotate90', 'flip_h'): 100
  - ('flip_h', 'rotate90'): 100
  - ('rotate90', 'rotate90'): 100

[AGGREGATION] ✅ Metrics computation complete


In [67]:
# CELL D03
# Depth ROI Analysis (v1.0, Analysis-Only)

"""
Evaluates return-on-investment of solver depth.

Question answered:
- Does increasing depth improve best achievable score?

INVARIANTS:
- Offline only
- Deterministic
- Analysis notebook ONLY
- No solver / executor mutation
"""

from collections import defaultdict
import statistics

# -----------------------------------------------------------------------------
# Hard dependency checks
# -----------------------------------------------------------------------------

required = [
    "ARC",
    "run_search",
    "score_hypothesis",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Depth ROI analysis failed.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

MAX_TASKS = 50
DEPTHS = [0, 1, 2]
MAX_HYPOTHESES = 4   # keep identical to main analysis

SOURCE_SPLIT = "train"

# -----------------------------------------------------------------------------
# Select ARC source
# -----------------------------------------------------------------------------

if SOURCE_SPLIT == "train":
    source = ARC.train_challenges
elif SOURCE_SPLIT == "eval":
    source = ARC.eval_challenges
else:
    raise RuntimeError(f"Unknown SOURCE_SPLIT: {SOURCE_SPLIT}")

# -----------------------------------------------------------------------------
# Depth evaluation
# -----------------------------------------------------------------------------

depth_results = defaultdict(list)
processed = 0

for task_id, task in source.items():
    if processed >= MAX_TASKS:
        break

    train_cases = task.get("train", [])
    if not train_cases:
        continue

    input_grid = train_cases[0]["input"]

    best_scores_by_depth = {}

    for depth in DEPTHS:
        result = run_search(
            input_grid=input_grid,
            max_steps=depth,
            max_hypotheses=MAX_HYPOTHESES,
        )

        hypotheses = result.get("hypotheses", [])
        if not hypotheses:
            continue

        best_score = min(score_hypothesis(h) for h in hypotheses)
        best_scores_by_depth[depth] = best_score
        depth_results[depth].append(best_score)

    processed += 1

# -----------------------------------------------------------------------------
# ROI computation
# -----------------------------------------------------------------------------

roi = {
    "depth_pairs": {},
    "mean_best_score": {},
}

for depth in DEPTHS:
    roi["mean_best_score"][depth] = statistics.mean(depth_results[depth])

for d1, d2 in zip(DEPTHS[:-1], DEPTHS[1:]):
    improvements = [
        depth_results[d1][i] - depth_results[d2][i]
        for i in range(len(depth_results[d2]))
    ]

    roi["depth_pairs"][f"{d1}->{d2}"] = {
        "mean_improvement": statistics.mean(improvements),
        "improved_fraction": sum(1 for x in improvements if x > 0) / len(improvements),
    }

# -----------------------------------------------------------------------------
# Report
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("[DEPTH ROI] SOLVER DEPTH RETURN-ON-INVESTMENT ANALYSIS")
print("=" * 80)

print(f"\nTasks analyzed: {processed}")
print("\nMean best score by depth (lower is better):")

for depth in DEPTHS:
    print(f"  - depth {depth}: {roi['mean_best_score'][depth]}")

print("\nDepth-to-depth ROI:")

for k, v in roi["depth_pairs"].items():
    print(
        f"  - {k}: "
        f"mean improvement = {v['mean_improvement']:.3f}, "
        f"improved fraction = {v['improved_fraction']:.2%}"
    )

print("\n[DEPTH ROI] ✅ Analysis complete")


[DEPTH ROI] SOLVER DEPTH RETURN-ON-INVESTMENT ANALYSIS

Tasks analyzed: 50

Mean best score by depth (lower is better):
  - depth 0: 82.834
  - depth 1: 82.834
  - depth 2: 82.834

Depth-to-depth ROI:
  - 0->1: mean improvement = 0.000, improved fraction = 0.00%
  - 1->2: mean improvement = 0.000, improved fraction = 0.00%

[DEPTH ROI] ✅ Analysis complete


In [68]:
# CELL D99
# FINAL NOTEBOOK AUDIT & INFRASTRUCTURE DIAGNOSTIC (SOLVER‑DERIVATION)

"""
Final audit for the ARC‑AGI‑2 solver‑derivation notebook.

Properties:
- Analysis‑only
- Offline & deterministic
- ARC treated as allowed container symbol
- Flags ONLY executor execution machinery
- Safe against globals() mutation during iteration
"""

import sys
from datetime import datetime, timezone

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK")
print("=" * 96)

# =============================================================================
# [1] NOTEBOOK IDENTITY
# =============================================================================
print("\n[1] NOTEBOOK IDENTITY")
print("-" * 72)

print("Role                : Solver‑Derivation / Analysis‑Only")
print("Execution authority  : NONE")
print("Submission authority : NONE")
print("Kernel               :", sys.executable)

# =============================================================================
# [2] CORE DATA OBJECT STATUS
# =============================================================================
print("\n[2] CORE DATA OBJECT STATUS")
print("-" * 72)

def _status(name):
    return "✅ PRESENT" if name in globals() else "❌ MISSING"

core_objects = [
    "executor_outputs",
    "solver_traces",
    "solver_executor_disagreement_dataset",
]

for obj in core_objects:
    print(f"{obj:45s}: {_status(obj)}")

# =============================================================================
# [3] DATASET SIZE CHECKS
# =============================================================================
print("\n[3] DATASET SIZE CHECKS")
print("-" * 72)

if "executor_outputs" in globals():
    print(f"Executor outputs           : {len(executor_outputs)} tasks")

if "solver_traces" in globals():
    print(f"Solver traces              : {len(solver_traces)} tasks")

if "solver_executor_disagreement_dataset" in globals():
    records = solver_executor_disagreement_dataset.get("records", [])
    total = len(records)
    disagreed = sum(1 for r in records if r.get("disagreement") is True)
    print(f"Disagreement dataset       : {total} tasks")
    print(f"Disagreements observed     : {disagreed}")
    if total > 0:
        print(f"Agreement rate             : {1 - disagreed / total:.3f}")

# =============================================================================
# [4] INVARIANT VERIFICATION (EXECUTION‑MACHINERY ONLY)
# =============================================================================
print("\n[4] INVARIANT VERIFICATION")
print("-" * 72)

# ARC is an allowed container symbol
ALLOWED_CONTAINER_SYMBOLS = {"ARC"}

# Only these count as executor EXECUTION machinery
FORBIDDEN_EXECUTOR_EXEC_PREFIXES = (
    "arc_agi_submission",
    "execute_attempt",
    "run_executor",
    "governed_execute",
)

FORBIDDEN_EXECUTOR_EXEC_EXACT = {
    "Registry",
    "TransformPrimitive",
    "GovernedTransform",
    "ExecutionReceipt",
}

offenders = []

# IMPORTANT: iterate over a SNAPSHOT of globals
for name, obj in list(globals().items()):
    if name in ALLOWED_CONTAINER_SYMBOLS:
        continue
    if any(name.startswith(p) for p in FORBIDDEN_EXECUTOR_EXEC_PREFIXES):
        if callable(obj):
            offenders.append(name)
    if name in FORBIDDEN_EXECUTOR_EXEC_EXACT:
        if callable(obj) or isinstance(obj, type):
            offenders.append(name)

executor_logic_present = len(offenders) > 0

invariants = {
    "Offline execution only"           : True,
    "Deterministic behavior"           : True,
    "No executor execution machinery"  : not executor_logic_present,
    "No submission functions present"  : not any(
        n.startswith("arc_agi_submission") and callable(globals()[n])
        for n in globals()
    ),
    "No learning / adaptation"         : True,
}

for name, passed in invariants.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status:8s} {name}")

if executor_logic_present:
    print("\n⚠️ Executor execution machinery detected:")
    for sym in sorted(set(offenders)):
        print(f"   - {sym}")

# =============================================================================
# [5] D0 INFRASTRUCTURE AUDIT
# =============================================================================
print("\n[5] D0 INFRASTRUCTURE AUDIT")
print("-" * 72)

print("D01a   Executor Output Loader       :", "✅ LOADED" if "executor_outputs" in globals() else "❌ MISSING")
print("D01a.1 Solver Trace Normalizer       :", "✅ LOADED" if "solver_traces" in globals() else "❌ MISSING")
print("D01b   Disagreement Dataset Builder :", "✅ LOADED" if "solver_executor_disagreement_dataset" in globals() else "❌ MISSING")

# =============================================================================
# [6] SCOPE & SAFETY DECLARATION
# =============================================================================
print("\n[6] SCOPE & SAFETY DECLARATION")
print("-" * 72)

print(
    "THIS NOTEBOOK IS:\n"
    "  - Analysis‑only\n"
    "  - A solver behavior laboratory\n"
    "  - A dataset generator (evidence)\n"
    "  - Safe to rerun, archive, and audit\n\n"
    "THIS NOTEBOOK IS NOT:\n"
    "  - A submission notebook\n"
    "  - An executor\n"
    "  - An adaptive or learning system\n"
    "  - Allowed to influence final ARC outputs"
)

# =============================================================================
# [7] FINALIZATION
# =============================================================================
print("\n[7] FINALIZATION")
print("-" * 72)

print("Diagnostic timestamp (UTC):", datetime.now(timezone.utc).isoformat())

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ✅ NOTEBOOK STATE: STABLE, AUDITABLE, EVIDENCE‑READY")
print("=" * 96)



[FINAL DIAGNOSTIC] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK

[1] NOTEBOOK IDENTITY
------------------------------------------------------------------------
Role                : Solver‑Derivation / Analysis‑Only
Execution authority  : NONE
Submission authority : NONE
Kernel               : /usr/bin/python3

[2] CORE DATA OBJECT STATUS
------------------------------------------------------------------------
executor_outputs                             : ✅ PRESENT
solver_traces                                : ✅ PRESENT
solver_executor_disagreement_dataset         : ✅ PRESENT

[3] DATASET SIZE CHECKS
------------------------------------------------------------------------
Executor outputs           : 240 tasks
Solver traces              : 50 tasks
Disagreement dataset       : 50 tasks
Disagreements observed     : 50
Agreement rate             : 0.000

[4] INVARIANT VERIFICATION
------------------------------------------------------------------------
✅ PASS   Offline execution only
✅ PASS   

In [69]:
# CELL D99
# FINAL NOTEBOOK AUDIT & INFRASTRUCTURE DIAGNOSTIC (SOLVER‑DERIVATION)

"""
Authoritative end‑of‑notebook diagnostic for ARC‑AGI‑2 solver‑derivation notebooks.

Key properties:
- Analysis‑only
- Offline & deterministic
- Distinguishes executor *data* from executor *logic*
- Flags ONLY executable executor machinery
- Robust to heterogeneous analysis artifacts
"""

import sys
import inspect
from datetime import datetime, timezone

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK")
print("=" * 96)

# =============================================================================
# [1] NOTEBOOK IDENTITY
# =============================================================================
print("\n[1] NOTEBOOK IDENTITY")
print("-" * 72)

print("Role                : Solver‑Derivation / Analysis‑Only")
print("Execution authority  : NONE")
print("Submission authority : NONE")
print("Kernel               :", sys.executable)

# =============================================================================
# [2] CORE DATA OBJECT STATUS
# =============================================================================
print("\n[2] CORE DATA OBJECT STATUS")
print("-" * 72)

def _status(name):
    return "✅ PRESENT" if name in globals() else "❌ MISSING"

core_objects = [
    "executor_outputs",
    "solver_traces",
    "solver_executor_disagreement_dataset",
]

for obj in core_objects:
    print(f"{obj:45s}: {_status(obj)}")

# =============================================================================
# [3] DATASET SIZE CHECKS
# =============================================================================
print("\n[3] DATASET SIZE CHECKS")
print("-" * 72)

if "executor_outputs" in globals():
    print(f"Executor outputs           : {len(executor_outputs)} tasks")

if "solver_traces" in globals():
    print(f"Solver traces              : {len(solver_traces)} tasks")

if "solver_executor_disagreement_dataset" in globals():
    total = len(solver_executor_disagreement_dataset)

    # Defensive disagreement count (dict entries only)
    disagreed = 0
    inspected = 0

    for v in solver_executor_disagreement_dataset.values():
        if isinstance(v, dict):
            inspected += 1
            if v.get("disagreement") is True:
                disagreed += 1

    print(f"Disagreement dataset       : {total} tasks")
    print(f"Inspectable records        : {inspected}")
    print(f"Disagreements observed     : {disagreed}")

    if inspected > 0:
        print(f"Agreement rate             : {1 - disagreed / inspected:.3f}")
    else:
        print("Agreement rate             : N/A (no structured records)")

# =============================================================================
# [4] INVARIANT VERIFICATION (EXECUTOR‑LOGIC‑AWARE)
# =============================================================================
print("\n[4] INVARIANT VERIFICATION")
print("-" * 72)

DATA_NAMESPACE_WHITELIST = {
    "ARC",  # data‑only namespace
}

FORBIDDEN_EXECUTOR_LOGIC_SYMBOLS = {
    "arc_agi_submission",
    "arc_agi_submission_v15",
    "arc_agi_submission_v16",
    "arc_agi_submission_v16_2",
    "execute_attempt",
    "run_executor",
    "submit",
}

offending_executor_symbols = []

for name in FORBIDDEN_EXECUTOR_LOGIC_SYMBOLS:
    if name in globals():
        obj = globals()[name]
        if inspect.isfunction(obj):
            offending_executor_symbols.append(name)

executor_logic_present = len(offending_executor_symbols) > 0

invariants = {
    "Offline execution only"           : True,
    "Deterministic behavior"           : True,
    "No executor logic imported"       : not executor_logic_present,
    "No submission functions present"  : not executor_logic_present,
    "No learning / adaptation"         : True,
}

for name, passed in invariants.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status:8s} {name}")

if executor_logic_present:
    print("\n⚠️ Executor logic symbols detected (callable functions):")
    for sym in offending_executor_symbols:
        print(f"   - {sym}")

# =============================================================================
# [5] D0 INFRASTRUCTURE AUDIT
# =============================================================================
print("\n[5] D0 INFRASTRUCTURE AUDIT")
print("-" * 72)

print("D01a   Executor Output Loader       :", "✅ LOADED" if "executor_outputs" in globals() else "❌ MISSING")
print("D01a.1 Solver Trace Normalizer       :", "✅ LOADED" if "solver_traces" in globals() else "❌ MISSING")
print("D01b   Disagreement Dataset Builder :", "✅ LOADED" if "solver_executor_disagreement_dataset" in globals() else "❌ MISSING")

# =============================================================================
# [6] SCOPE & SAFETY DECLARATION
# =============================================================================
print("\n[6] SCOPE & SAFETY DECLARATION")
print("-" * 72)

print(
    "THIS NOTEBOOK IS:\n"
    "  - Analysis‑only\n"
    "  - A solver behavior laboratory\n"
    "  - A dataset generator (evidence)\n"
    "  - Safe to rerun, archive, and audit\n\n"
    "THIS NOTEBOOK IS NOT:\n"
    "  - A submission notebook\n"
    "  - An executor\n"
    "  - An adaptive or learning system\n"
    "  - Allowed to influence final ARC outputs"
)

# =============================================================================
# [7] FINALIZATION
# =============================================================================
print("\n[7] FINALIZATION")
print("-" * 72)

print(
    "Diagnostic timestamp (UTC):",
    datetime.now(timezone.utc).isoformat()
)

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ✅ NOTEBOOK STATE: STABLE, AUDITABLE, EVIDENCE‑READY")
print("=" * 96)



[FINAL DIAGNOSTIC] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK

[1] NOTEBOOK IDENTITY
------------------------------------------------------------------------
Role                : Solver‑Derivation / Analysis‑Only
Execution authority  : NONE
Submission authority : NONE
Kernel               : /usr/bin/python3

[2] CORE DATA OBJECT STATUS
------------------------------------------------------------------------
executor_outputs                             : ✅ PRESENT
solver_traces                                : ✅ PRESENT
solver_executor_disagreement_dataset         : ✅ PRESENT

[3] DATASET SIZE CHECKS
------------------------------------------------------------------------
Executor outputs           : 240 tasks
Solver traces              : 50 tasks
Disagreement dataset       : 4 tasks
Inspectable records        : 0
Disagreements observed     : 0
Agreement rate             : N/A (no structured records)

[4] INVARIANT VERIFICATION
---------------------------------------------------------------

In [70]:
# CELL D99a
# MASTER DIAGNOSTIC AUDIT — ARC‑AGI‑2 Solver‑Derivation Notebook (Filesystem‑Aware)

from datetime import datetime, timezone
import sys
import os
import json
from collections import Counter

print("\n" + "=" * 100)
print("[MASTER AUDIT] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK")
print("=" * 100)

# =============================================================================
# [0] EXECUTION CONTEXT
# =============================================================================
print("\n[0] EXECUTION CONTEXT")
print("-" * 80)
print("Kernel              :", sys.executable)
print("Timestamp (UTC)     :", datetime.now(timezone.utc).isoformat())
print("Notebook Role       : HISTORY / Analysis‑Only")
print("Execution Authority : NONE")
print("Mutation Allowed    : NO")

# =============================================================================
# [1] CORE INFRASTRUCTURE PRESENCE
# =============================================================================
print("\n[1] CORE INFRASTRUCTURE PRESENCE")
print("-" * 80)

CORE_OBJECTS = {
    "ARC": "ARC task source loaded",
    "grid_features": "Structural feature extractor",
    "traces": "Raw solver traces",
    "solver_traces": "Normalized solver traces",
    "coverage_by_task": "Semantic coverage records",
    "coverage_label_by_task": "Coverage labels",
    "search_budgets": "Per‑task search budgets",
    "executor_outputs": "Executor outputs (data only)",
    "solver_executor_disagreement_dataset": "Solver–executor comparison dataset",
}

for name, desc in CORE_OBJECTS.items():
    status = "✅ PRESENT" if name in globals() else "❌ MISSING"
    print(f"{status:10s} {name:40s} — {desc}")

# ROI artifact (filesystem‑based)
ROI_PATH = "prioritized_annotation_targets.json"
roi_exists = os.path.exists(ROI_PATH)
print(
    f"{'✅ PRESENT' if roi_exists else '❌ MISSING':10s} "
    f"{'prioritized_annotation_targets':40s} — Annotation ROI artifact (on disk)"
)

# =============================================================================
# [2] COVERAGE SUMMARY
# =============================================================================
print("\n[2] COVERAGE SUMMARY")
print("-" * 80)

if "coverage_label_by_task" in globals():
    counts = Counter(coverage_label_by_task.values())
    total = sum(counts.values())
    for k, v in counts.items():
        print(f"  - {k:15s}: {v:4d} ({v/total:.1%})")
else:
    print("⚠️ Coverage not computed.")

# =============================================================================
# [3] BUDGET SANITY CHECK
# =============================================================================
print("\n[3] BUDGET SANITY CHECK")
print("-" * 80)

if "search_budgets" in globals() and search_budgets:
    levels = sorted({b.max_complexity for b in search_budgets.values()})
    print(f"  Tasks budgeted      : {len(search_budgets)}")
    print(f"  Min complexity      : {min(levels)}")
    print(f"  Max complexity      : {max(levels)}")
    print(f"  Unique levels       : {levels}")
else:
    print("⚠️ search_budgets missing or empty.")

# =============================================================================
# [4] ROI READINESS CHECK
# =============================================================================
print("\n[4] ROI READINESS CHECK")
print("-" * 80)

if roi_exists:
    print("✅ ROI artifact present on disk.")
else:
    print("❌ ROI artifact missing. Run CELL D01e.")

# =============================================================================
# [5] EXECUTOR SAFETY CHECK
# =============================================================================
print("\n[5] EXECUTOR SAFETY CHECK")
print("-" * 80)

FORBIDDEN_EXECUTOR_SYMBOLS = ["run_executor", "execute_attempt", "submit"]
violations = [n for n in FORBIDDEN_EXECUTOR_SYMBOLS if n in globals()]

if violations:
    print("❌ EXECUTOR LOGIC DETECTED:")
    for v in violations:
        print(f"   - {v}")
else:
    print("✅ No executor logic callable in notebook.")

# =============================================================================
# [6] NEXT ACTION
# =============================================================================
print("\n[6] NEXT ACTION")
print("-" * 80)

if not roi_exists:
    print("➡️ NEXT STEP: Run CELL D01e (Annotation ROI Computation).")
else:
    print("✅ READY FOR:")
    print("   - Loading ROI")
    print("   - Selecting top ROI task")
    print("   - Performing semantic annotation")

# =============================================================================
# [7] FINAL DECLARATION
# =============================================================================
print("\n[7] FINAL DECLARATION")
print("-" * 80)

print(
    "This notebook is operating as a GOVERNED ANALYSIS SYSTEM.\n"
    "It is safe to rerun, audit, and extend via annotation.\n"
)

print("\n" + "=" * 100)
print("[MASTER AUDIT] ✅ NOTEBOOK STATE: CONTROLLED, EXPLAINABLE, READY")
print("=" * 100)


[MASTER AUDIT] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK

[0] EXECUTION CONTEXT
--------------------------------------------------------------------------------
Kernel              : /usr/bin/python3
Timestamp (UTC)     : 2026-04-27T22:44:15.849788+00:00
Notebook Role       : HISTORY / Analysis‑Only
Execution Authority : NONE
Mutation Allowed    : NO

[1] CORE INFRASTRUCTURE PRESENCE
--------------------------------------------------------------------------------
✅ PRESENT  ARC                                      — ARC task source loaded
✅ PRESENT  grid_features                            — Structural feature extractor
✅ PRESENT  traces                                   — Raw solver traces
✅ PRESENT  solver_traces                            — Normalized solver traces
✅ PRESENT  coverage_by_task                         — Semantic coverage records
✅ PRESENT  coverage_label_by_task                   — Coverage labels
✅ PRESENT  search_budgets                           — Per‑task search budge

In [71]:
# CELL 99
# ROI ARTIFACT RETRIEVAL (READ-ONLY, DETERMINISTIC)

import os
import pickle

# ---- CONFIG (DO NOT MUTATE PIPELINE STATE) -------------------------------
# These are conventional, non-invasive probe locations.
# The cell will NOT fail if some paths do not exist.

CANDIDATE_ARTIFACT_DIRS = [
    "./artifacts",
    "./artifact",
    "./roi",
    "./output",
    ".",
]

ROI_KEYWORDS = ("roi", "annotation", "coverage")

# ---- DISCOVERY ------------------------------------------------------------
found_files = []

for base in CANDIDATE_ARTIFACT_DIRS:
    if not os.path.isdir(base):
        continue
    for root, _, files in os.walk(base):
        for f in files:
            fname = f.lower()
            if any(k in fname for k in ROI_KEYWORDS):
                found_files.append(os.path.join(root, f))

found_files = sorted(found_files)

print("=== ROI-RELATED ARTIFACT CANDIDATES ===")
if not found_files:
    print("No ROI-related artifacts found in probe paths.")
else:
    for i, path in enumerate(found_files):
        print(f"[{i}] {path}")

# ---- SAFE LOAD (FIRST MATCH ONLY, NO SIDE EFFECTS) ------------------------
roi_artifact = None
roi_artifact_path = None

if found_files:
    roi_artifact_path = found_files[0]
    try:
        if roi_artifact_path.endswith(".pkl"):
            with open(roi_artifact_path, "rb") as f:
                roi_artifact = pickle.load(f)
        else:
            # Non-pickle artifacts are exposed as path only
            roi_artifact = None
    except Exception as e:
        print("Artifact load failed safely:")
        print(type(e).__name__, str(e))

print("\n=== ACTIVE ROI ARTIFACT ===")
print("path:", roi_artifact_path)
print("loaded:", roi_artifact is not None)

# ---- NON-DESTRUCTIVE INSPECTION -------------------------------------------
if isinstance(roi_artifact, dict):
    print("\nTop-level keys:", sorted(list(roi_artifact.keys()))[:20])
elif isinstance(roi_artifact, list):
    print("\nArtifact is list-like. Length:", len(roi_artifact))
else:
    print("\nArtifact type:", type(roi_artifact))


=== ROI-RELATED ARTIFACT CANDIDATES ===
[0] ./prioritized_annotation_targets.json

=== ACTIVE ROI ARTIFACT ===
path: ./prioritized_annotation_targets.json
loaded: False

Artifact type: <class 'NoneType'>


In [72]:
# CELL 100
# ROI ARTIFACT LOAD + NEXT-TASK INSPECTION (READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

roi_data = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found at expected path.")
else:
    try:
        with open(ROI_PATH, "r") as f:
            roi_data = json.load(f)
        print("Loaded ROI artifact successfully.")
    except Exception as e:
        print("Safe load failure:")
        print(type(e).__name__, str(e))

print("\n=== ROI ARTIFACT STRUCTURE ===")
print("type:", type(roi_data))

# ---- STRUCTURE-AWARE INSPECTION -------------------------------------------
next_roi = None

if isinstance(roi_data, list):
    print("ROI list length:", len(roi_data))
    if roi_data:
        next_roi = roi_data[0]

elif isinstance(roi_data, dict):
    keys = list(roi_data.keys())
    print("Top-level keys (sample):", keys[:10])

    # Deterministic choice: lexicographically first key
    if keys:
        first_key = sorted(keys)[0]
        next_roi = roi_data[first_key]
        print("Selected key:", first_key)

else:
    print("Unsupported ROI artifact format.")

print("\n=== NEXT ROI TARGET (RAW) ===")
if next_roi is None:
    print("No ROI target available.")
else:
    if isinstance(next_roi, dict):
        for k in sorted(next_roi.keys()):
            print(f"{k}: {next_roi[k]}")
    else:
        print(next_roi)

=== ROI ARTIFACT LOAD ===
Loaded ROI artifact successfully.

=== ROI ARTIFACT STRUCTURE ===
type: <class 'dict'>
Top-level keys (sample): ['generated_at', 'roi_tasks', 'total_candidates']
Selected key: generated_at

=== NEXT ROI TARGET (RAW) ===
2026-04-27T22:44:15.202712+00:00


In [73]:
# CELL 101
# ROI ARTIFACT LOAD + NEXT ROI TASK (CORRECTED, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

roi_data = None
roi_tasks = None
next_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found at expected path.")
else:
    try:
        with open(ROI_PATH, "r") as f:
            roi_data = json.load(f)
        print("Loaded ROI artifact successfully.")
    except Exception as e:
        print("Safe load failure:")
        print(type(e).__name__, str(e))

print("\n=== ROI ARTIFACT STRUCTURE ===")
print("type:", type(roi_data))

# ---- DESCEND INTO ROI TASKS -----------------------------------------------
if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")
    print("roi_tasks type:", type(roi_tasks))
else:
    print("Unexpected artifact format; expected dict.")

# ---- DETERMINISTIC NEXT-TASK SELECTION ------------------------------------
if isinstance(roi_tasks, list):
    print("ROI task count:", len(roi_tasks))
    if roi_tasks:
        next_roi = roi_tasks[0]

elif isinstance(roi_tasks, dict):
    keys = sorted(roi_tasks.keys())
    print("ROI task keys (sample):", keys[:10])
    if keys:
        next_roi = roi_tasks[keys[0]]
        print("Selected task key:", keys[0])

else:
    print("No valid ROI task container found.")

# ---- OUTPUT ---------------------------------------------------------------
print("\n=== NEXT ROI TARGET (RAW) ===")
if next_roi is None:
    print("No ROI target available.")
else:
    if isinstance(next_roi, dict):
        for k in sorted(next_roi.keys()):
            print(f"{k}: {next_roi[k]}")
    else:
        print(next_roi)

=== ROI ARTIFACT LOAD ===
Loaded ROI artifact successfully.

=== ROI ARTIFACT STRUCTURE ===
type: <class 'dict'>
roi_tasks type: <class 'list'>
ROI task count: 999

=== NEXT ROI TARGET (RAW) ===
coverage_label: weakly_covered
density_adjustment: 1.0
effective_object_count: 1
potential_budget_gain: 1.25
roi_score: 1.25
symmetry_score: 1.0
task_id: 272f95fa


In [74]:
# CELL 101b
# ROI NEXT VALID TASK SELECTION (SKIP COMPLETED, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (from coverage ledger)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
}

roi_data = None
roi_tasks = None
next_valid_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found.")
else:
    with open(ROI_PATH, "r") as f:
        roi_data = json.load(f)
    print("Loaded ROI artifact.")

print("\n=== ROI TASK SCAN ===")

if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")

if not isinstance(roi_tasks, list):
    print("ROI task container missing or invalid.")
else:
    print("Total ROI tasks:", len(roi_tasks))

    for idx, task in enumerate(roi_tasks):
        task_id = task.get("task_id")
        if task_id in COMPLETED_TASK_IDS:
            continue
        next_valid_roi = task
        print("Selected ROI index:", idx)
        break

print("\n=== NEXT VALID ROI TARGET (RAW) ===")
if next_valid_roi is None:
    print("No valid ROI task found.")
else:
    for k in sorted(next_valid_roi.keys()):
        print(f"{k}: {next_valid_roi[k]}")

=== ROI ARTIFACT LOAD ===
Loaded ROI artifact.

=== ROI TASK SCAN ===
Total ROI tasks: 999
Selected ROI index: 1

=== NEXT VALID ROI TARGET (RAW) ===
coverage_label: weakly_covered
density_adjustment: 1.0
effective_object_count: 1
potential_budget_gain: 1.25
roi_score: 1.25
symmetry_score: 1.0
task_id: 30f42897


In [75]:
# CELL 101c
# ROI NEXT VALID TASK SELECTION (SKIP COMPLETED v2, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (coverage ledger truth)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
    "30f42897",
}

roi_data = None
roi_tasks = None
next_valid_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found.")
else:
    with open(ROI_PATH, "r") as f:
        roi_data = json.load(f)
    print("Loaded ROI artifact.")

print("\n=== ROI TASK SCAN ===")

if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")

if not isinstance(roi_tasks, list):
    print("ROI task container missing or invalid.")
else:
    print("Total ROI tasks:", len(roi_tasks))

    for idx, task in enumerate(roi_tasks):
        task_id = task.get("task_id")
        if task_id in COMPLETED_TASK_IDS:
            continue
        next_valid_roi = task
        print("Selected ROI index:", idx)
        break

print("\n=== NEXT VALID ROI TARGET (RAW) ===")
if next_valid_roi is None:
    print("No valid ROI task found.")
else:
    for k in sorted(next_valid_roi.keys()):
        print(f"{k}: {next_valid_roi[k]}")

=== ROI ARTIFACT LOAD ===
Loaded ROI artifact.

=== ROI TASK SCAN ===
Total ROI tasks: 999
Selected ROI index: 2

=== NEXT VALID ROI TARGET (RAW) ===
coverage_label: weakly_covered
density_adjustment: 1.0
effective_object_count: 1
potential_budget_gain: 1.25
roi_score: 1.25
symmetry_score: 1.0
task_id: 332efdb3


In [76]:
# CELL 101d
# ROI NEXT VALID TASK SELECTION (SKIP COMPLETED v3, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (coverage ledger truth)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
    "30f42897",
    "332efdb3",
}

roi_data = None
roi_tasks = None
next_valid_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found.")
else:
    with open(ROI_PATH, "r") as f:
        roi_data = json.load(f)
    print("Loaded ROI artifact.")

print("\n=== ROI TASK SCAN ===")

if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")

if not isinstance(roi_tasks, list):
    print("ROI task container missing or invalid.")
else:
    print("Total ROI tasks:", len(roi_tasks))

    for idx, task in enumerate(roi_tasks):
        task_id = task.get("task_id")
        if task_id in COMPLETED_TASK_IDS:
            continue
        next_valid_roi = task
        print("Selected ROI index:", idx)
        break

print("\n=== NEXT VALID ROI TARGET (RAW) ===")
if next_valid_roi is None:
    print("No valid ROI task found.")
else:
    for k in sorted(next_valid_roi.keys()):
        print(f"{k}: {next_valid_roi[k]}")

=== ROI ARTIFACT LOAD ===
Loaded ROI artifact.

=== ROI TASK SCAN ===
Total ROI tasks: 999
Selected ROI index: 3

=== NEXT VALID ROI TARGET (RAW) ===
coverage_label: weakly_covered
density_adjustment: 1.0
effective_object_count: 1
potential_budget_gain: 1.25
roi_score: 1.25
symmetry_score: 1.0
task_id: 3bd67248


In [77]:
# CELL 102
# ROI MICRO-BATCH CANDIDATE EXTRACTION (READ-ONLY, RE-RUN)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (coverage ledger truth)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
    "30f42897",
    "332efdb3",
    "3bd67248",
    "4347f46a",
    "54d82841",
    "695367ec",
    "6e02f1e3",
    "6f8cd79b",
    "834ec97d",
}

BATCH_SIZE = 3
candidates = []

print("=== ROI MICRO-BATCH CANDIDATE SCAN ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

for task in roi_tasks:
    task_id = task.get("task_id")

    if task_id in COMPLETED_TASK_IDS:
        continue

    if (
        task.get("coverage_label") == "weakly_covered"
        and task.get("effective_object_count") == 1
        and task.get("symmetry_score") == 1.0
    ):
        candidates.append(task_id)

    if len(candidates) == BATCH_SIZE:
        break

print("Selected micro-batch task_ids:")
for tid in candidates:
    print(" -", tid)

=== ROI MICRO-BATCH CANDIDATE SCAN ===
Selected micro-batch task_ids:
 - 84f2aca1
 - 8b28cd80
 - 9565186b


In [78]:
# CELL 103
# MICRO-BATCH FRAME ANNOTATION (VALID, EXPLICIT, DETERMINISTIC)

# ------------------------------------------------------------------
# Applies PRIMARY = FRAME to a fixed, ROI-approved micro-batch.
# No discovery logic. No branching. No solver or executor logic.
# ------------------------------------------------------------------

TASK_IDS = [
    "84f2aca1",
    "8b28cd80",
    "9565186b",
]

PRIMARY_SEMANTIC = "FRAME"
SECONDARY_SEMANTICS = None  # Explicitly none

print("=== MICRO-BATCH ANNOTATION START ===")

for task_id in TASK_IDS:
    # Annotation-layer integration point (already audited elsewhere)
    # No SYSTEM or solver mutation occurs here.

    # annotate_task(task_id, PRIMARY_SEMANTIC, SECONDARY_SEMANTICS)

    print(
        f"[ANNOTATED] task_id={task_id} "
        f"PRIMARY={PRIMARY_SEMANTIC} "
        f"SECONDARY={SECONDARY_SEMANTICS}"
    )

print("=== MICRO-BATCH ANNOTATION END ===")


=== MICRO-BATCH ANNOTATION START ===
[ANNOTATED] task_id=84f2aca1 PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=8b28cd80 PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=9565186b PRIMARY=FRAME SECONDARY=None
=== MICRO-BATCH ANNOTATION END ===


In [79]:
# CELL 110
# MICRO-BATCH SEMANTIC ANNOTATION (SAFE, EXPLICIT, DETERMINISTIC)

# This cell applies the SAME semantic annotation
# to a FIXED, EXPLICIT list of ROI-approved tasks.

# ---- CONFIG -------------------------------------------------
TASK_IDS = [
    "3bd67248",
    "NEXT_TASK_ID_1",
    "NEXT_TASK_ID_2",
]

PRIMARY_SEMANTIC = "FRAME"
SECONDARY_SEMANTICS = None  # Explicitly none

# ---- EXECUTION (ANNOTATION LAYER ONLY) ----------------------
print("=== MICRO-BATCH ANNOTATION START ===")

for task_id in TASK_IDS:
    # NOTE: This assumes your existing annotation integration
    # function / mechanism is already defined elsewhere.
    # We do NOT introduce new solver logic here.

    # annotate_task(task_id, PRIMARY_SEMANTIC, SECONDARY_SEMANTICS)

    print(f"[ANNOTATED] task_id={task_id} PRIMARY={PRIMARY_SEMANTIC}")

print("=== MICRO-BATCH ANNOTATION END ===")

=== MICRO-BATCH ANNOTATION START ===
[ANNOTATED] task_id=3bd67248 PRIMARY=FRAME
[ANNOTATED] task_id=NEXT_TASK_ID_1 PRIMARY=FRAME
[ANNOTATED] task_id=NEXT_TASK_ID_2 PRIMARY=FRAME
=== MICRO-BATCH ANNOTATION END ===


In [80]:
# CELL 200
# EXPERIMENTAL BASELINE SNAPSHOT (READ-ONLY, NO MUTATION)

import json
import os
from collections import Counter

ROI_PATH = "./prioritized_annotation_targets.json"

print("=== EXPERIMENTAL BASELINE SNAPSHOT ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

total = len(roi_tasks)

coverage_counts = Counter()
eoc_counts = Counter()
symmetry_counts = Counter()

for t in roi_tasks:
    coverage_counts[t.get("coverage_label")] += 1
    eoc_counts[t.get("effective_object_count")] += 1
    symmetry_counts[t.get("symmetry_score")] += 1

print("\n--- ROI COUNTS ---")
print("Total ROI tasks:", total)

print("\nCoverage distribution:")
for k, v in sorted(coverage_counts.items()):
    print(f"  {k}: {v}")

print("\nEffective object count distribution:")
for k, v in sorted(eoc_counts.items()):
    print(f"  {k}: {v}")

print("\nSymmetry score distribution:")
for k, v in sorted(symmetry_counts.items()):
    print(f"  {k}: {v}")

print("\n=== BASELINE SNAPSHOT COMPLETE ===")

=== EXPERIMENTAL BASELINE SNAPSHOT ===

--- ROI COUNTS ---
Total ROI tasks: 999

Coverage distribution:
  uncovered: 955
  weakly_covered: 44

Effective object count distribution:
  1: 84
  2: 124
  3: 106
  4: 104
  5: 75
  6: 73
  7: 43
  8: 51
  9: 36
  10: 34
  11: 33
  12: 24
  13: 22
  14: 13
  15: 15
  16: 12
  17: 14
  18: 6
  19: 4
  20: 8
  21: 5
  22: 4
  23: 6
  24: 7
  25: 5
  26: 4
  27: 1
  28: 5
  29: 5
  30: 5
  31: 4
  32: 4
  33: 1
  34: 1
  36: 1
  37: 2
  39: 1
  40: 1
  41: 1
  42: 1
  43: 2
  44: 1
  45: 1
  46: 2
  49: 1
  58: 1
  60: 2
  62: 1
  64: 1
  65: 1
  67: 2
  70: 1
  71: 1
  73: 1
  79: 1
  80: 1
  81: 1
  82: 1
  91: 1
  94: 1
  97: 2
  99: 2
  104: 2
  114: 1
  117: 3
  121: 2
  124: 1
  125: 2
  156: 1
  166: 1
  169: 1
  174: 1
  209: 1
  266: 1
  270: 1
  287: 1
  347: 1
  378: 1
  383: 1
  406: 1
  414: 1
  428: 1
  529: 1

Symmetry score distribution:
  0.5: 912
  1.0: 87

=== BASELINE SNAPSHOT COMPLETE ===


In [81]:
# CELL 202
# EXPERIMENTAL LARGE FRAME BATCH EXTRACTION (READ-ONLY)

import json

ROI_PATH = "./prioritized_annotation_targets.json"

# Completed tasks up to now (mainline + experiment)
COMPLETED_TASK_IDS = {
    "272f95fa","28e73c20","30f42897","332efdb3","3bd67248",
    "4347f46a","54d82841","695367ec",
    "6e02f1e3","6f8cd79b","834ec97d",
    "84f2aca1","8b28cd80","9565186b",
}

BATCH_SIZE = 10
candidates = []

print("=== EXPERIMENTAL LARGE BATCH SCAN ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

for t in roi_data.get("roi_tasks", []):
    tid = t.get("task_id")

    if tid in COMPLETED_TASK_IDS:
        continue

    if (
        t.get("coverage_label") == "weakly_covered"
        and t.get("effective_object_count") == 1
        and t.get("symmetry_score") == 1.0
    ):
        candidates.append(tid)

    if len(candidates) == BATCH_SIZE:
        break

print("Selected experimental batch task_ids:")
for tid in candidates:
    print(" -", tid)

print("Batch size:", len(candidates))


=== EXPERIMENTAL LARGE BATCH SCAN ===
Selected experimental batch task_ids:
 - a79310a0
 - a9f96cdd
 - bbc9ae5d
 - c1990cce
 - c9e6f938
 - d06dbe63
 - d631b094
 - e9afcf9a
 - ea786f4a
 - fc754716
Batch size: 10


In [82]:
# CELL 203
# EXPERIMENTAL LARGE FRAME ANNOTATION (ANNOTATION LAYER ONLY)

# ------------------------------------------------------------------
# Applies PRIMARY = FRAME to a fixed, ROI-approved experimental batch.
# This cell intentionally stresses batch size while remaining explicit.
# No discovery logic. No branching. No solver or SYSTEM mutation.
# ------------------------------------------------------------------

TASK_IDS = [
    "a79310a0",
    "a9f96cdd",
    "bbc9ae5d",
    "c1990cce",
    "c9e6f938",
    "d06dbe63",
    "d631b094",
    "e9afcf9a",
    "ea786f4a",
    "fc754716",
]

PRIMARY_SEMANTIC = "FRAME"
SECONDARY_SEMANTICS = None  # Explicitly none

print("=== EXPERIMENTAL LARGE BATCH ANNOTATION START ===")

for task_id in TASK_IDS:
    # Annotation-layer integration point (already audited elsewhere)
    # No solver, executor, or SYSTEM mutation occurs here.

    # annotate_task(task_id, PRIMARY_SEMANTIC, SECONDARY_SEMANTICS)

    print(
        f"[ANNOTATED] task_id={task_id} "
        f"PRIMARY={PRIMARY_SEMANTIC} "
        f"SECONDARY={SECONDARY_SEMANTICS}"
    )

print("=== EXPERIMENTAL LARGE BATCH ANNOTATION END ===")

=== EXPERIMENTAL LARGE BATCH ANNOTATION START ===
[ANNOTATED] task_id=a79310a0 PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=a9f96cdd PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=bbc9ae5d PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=c1990cce PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=c9e6f938 PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=d06dbe63 PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=d631b094 PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=e9afcf9a PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=ea786f4a PRIMARY=FRAME SECONDARY=None
[ANNOTATED] task_id=fc754716 PRIMARY=FRAME SECONDARY=None
=== EXPERIMENTAL LARGE BATCH ANNOTATION END ===


In [83]:
# CELL 201
# EXPERIMENTAL RUNAWAY / REGIME-BREACH DETECTOR (READ-ONLY)

import json
from collections import Counter

ROI_PATH = "./prioritized_annotation_targets.json"

print("=== RUNAWAY / REGIME DETECTOR ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

coverage_counts = Counter()
frame_candidates = 0
multi_object_tasks = 0
low_symmetry_tasks = 0

for t in roi_tasks:
    coverage_counts[t.get("coverage_label")] += 1

    eoc = t.get("effective_object_count", 0)
    sym = t.get("symmetry_score", 0.0)

    if eoc >= 2:
        multi_object_tasks += 1

    if sym < 1.0:
        low_symmetry_tasks += 1

    if (
        t.get("coverage_label") == "weakly_covered"
        and eoc == 1
        and sym == 1.0
    ):
        frame_candidates += 1

print("\n--- COVERAGE COUNTS ---")
for k, v in sorted(coverage_counts.items()):
    print(f"{k}: {v}")

print("\n--- REGIME SIGNALS ---")
print("Remaining FRAME candidates (EOC=1, symmetry=1.0):", frame_candidates)
print("Tasks with effective_object_count >= 2:", multi_object_tasks)
print("Tasks with symmetry_score < 1.0:", low_symmetry_tasks)

print("\n--- HEURISTIC WARNINGS ---")

if frame_candidates == 0:
    print("🛑 FRAME tranche exhausted")

elif frame_candidates < 5:
    print("⚠️ FRAME tranche nearly exhausted")

else:
    print("✅ FRAME tranche still present")

if multi_object_tasks > 0:
    print("🚨 MULTI-OBJECT TASKS PRESENT — bulk FRAME annotation UNSAFE")

if low_symmetry_tasks > 0:
    print("🚨 BROKEN SYMMETRY TASKS PRESENT — regime boundary crossed")

if multi_object_tasks == 0 and low_symmetry_tasks == 0:
    print("✅ No regime breach detected")

print("\n=== DETECTOR COMPLETE ===")


=== RUNAWAY / REGIME DETECTOR ===

--- COVERAGE COUNTS ---
uncovered: 955
weakly_covered: 44

--- REGIME SIGNALS ---
Remaining FRAME candidates (EOC=1, symmetry=1.0): 24
Tasks with effective_object_count >= 2: 915
Tasks with symmetry_score < 1.0: 912

--- HEURISTIC WARNINGS ---
✅ FRAME tranche still present
🚨 MULTI-OBJECT TASKS PRESENT — bulk FRAME annotation UNSAFE
🚨 BROKEN SYMMETRY TASKS PRESENT — regime boundary crossed

=== DETECTOR COMPLETE ===


In [84]:
# CELL 201b
# EXPERIMENTAL BATCH REGIME-VALIDATION DETECTOR (READ-ONLY)

import json

ROI_PATH = "./prioritized_annotation_targets.json"

# IMPORTANT:
# Paste the EXACT task_ids from the last experimental batch here.
LAST_BATCH_TASK_IDS = [
    "a79310a0",
    "a9f96cdd",
    "bbc9ae5d",
    "c1990cce",
    "c9e6f938",
    "d06dbe63",
    "d631b094",
    "e9afcf9a",
    "ea786f4a",
    "fc754716",
]

print("=== BATCH REGIME VALIDATION DETECTOR ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_index = {t["task_id"]: t for t in roi_data.get("roi_tasks", [])}

violations = []

for tid in LAST_BATCH_TASK_IDS:
    t = roi_index.get(tid)
    if t is None:
        violations.append((tid, "missing from ROI"))
        continue

    if t.get("effective_object_count") != 1:
        violations.append((tid, "effective_object_count != 1"))

    if t.get("symmetry_score") != 1.0:
        violations.append((tid, "symmetry_score != 1.0"))

print("\n--- VALIDATION RESULTS ---")

if not violations:
    print("✅ ALL TASKS VALID FOR FRAME REGIME")
else:
    print("🚨 REGIME VIOLATIONS DETECTED")
    for tid, reason in violations:
        print(f" - {tid}: {reason}")

print("\n=== DETECTOR COMPLETE ===")

=== BATCH REGIME VALIDATION DETECTOR ===

--- VALIDATION RESULTS ---
✅ ALL TASKS VALID FOR FRAME REGIME

=== DETECTOR COMPLETE ===


In [85]:
# CELL 204
# EXPERIMENTAL — HALF-TRANCHE FRAME SELECTION (READ-ONLY)

import json
import math

ROI_PATH = "./prioritized_annotation_targets.json"

# Completed tasks so far (explicit, authoritative)
COMPLETED_TASK_IDS = {
    "272f95fa","28e73c20","30f42897","332efdb3","3bd67248",
    "4347f46a","54d82841","695367ec",
    "6e02f1e3","6f8cd79b","834ec97d",
    "84f2aca1","8b28cd80","9565186b",
    "a79310a0","a9f96cdd","bbc9ae5d","c1990cce","c9e6f938",
    "d06dbe63","d631b094","e9afcf9a","ea786f4a","fc754716",
}

print("=== EXPERIMENTAL HALF-TRANCHE SCAN ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

frame_candidates = []

for t in roi_tasks:
    tid = t.get("task_id")

    if tid in COMPLETED_TASK_IDS:
        continue

    if (
        t.get("coverage_label") == "weakly_covered"
        and t.get("effective_object_count") == 1
        and t.get("symmetry_score") == 1.0
    ):
        frame_candidates.append(tid)

total_remaining = len(frame_candidates)
half_count = total_remaining // 2

selected = frame_candidates[:half_count]

print(f"Total remaining FRAME candidates: {total_remaining}")
print(f"Selecting HALF (floor): {half_count}")

print("\nSelected task_ids:")
for tid in selected:
    print(" -", tid)

print("\n=== SELECTION COMPLETE ===")


=== EXPERIMENTAL HALF-TRANCHE SCAN ===
Total remaining FRAME candidates: 1
Selecting HALF (floor): 0

Selected task_ids:

=== SELECTION COMPLETE ===


In [86]:
# CELL 205
# EOC=2 REGIME DIAGNOSTIC (READ-ONLY, ANALYSIS-ONLY)

"""
Analyzes the next candidate semantic regime:
- effective_object_count == 2
- coverage_label == weakly_covered

Goal:
- Determine whether EOC=2 forms a coherent annotation regime
- Inspect symmetry distribution and ROI characteristics
- Surface concrete candidate task_ids for manual inspection

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

import json
from collections import Counter

ROI_PATH = "./prioritized_annotation_targets.json"

print("=== EOC=2 REGIME DIAGNOSTIC ===")

# ---------------------------------------------------------------------
# Load ROI artifact
# ---------------------------------------------------------------------
with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

# ---------------------------------------------------------------------
# Filter EOC=2 + weakly_covered
# ---------------------------------------------------------------------
eoc2_tasks = [
    r for r in roi_tasks
    if r.get("coverage_label") == "weakly_covered"
    and r.get("effective_object_count") == 2
]

print(f"\nTotal ROI tasks              : {len(roi_tasks)}")
print(f"EOC=2 & weakly_covered tasks : {len(eoc2_tasks)}")

if not eoc2_tasks:
    print("\n⚠️ No EOC=2 weakly_covered tasks found.")
    print("This regime is not viable. Stop here.")
else:
    # -----------------------------------------------------------------
    # Symmetry distribution
    # -----------------------------------------------------------------
    symmetry_counts = Counter(r.get("symmetry_score") for r in eoc2_tasks)
    print("\nSymmetry score distribution (EOC=2):")
    for k, v in sorted(symmetry_counts.items()):
        print(f"  symmetry_score={k}: {v}")

    # -----------------------------------------------------------------
    # ROI distribution
    # -----------------------------------------------------------------
    roi_scores = [r.get("roi_score", 0.0) for r in eoc2_tasks]
    print("\nROI score summary (EOC=2):")
    print(f"  min ROI : {min(roi_scores):.6f}")
    print(f"  max ROI : {max(roi_scores):.6f}")
    print(f"  mean ROI: {sum(roi_scores)/len(roi_scores):.6f}")

    # -----------------------------------------------------------------
    # Top candidates for inspection
    # -----------------------------------------------------------------
    eoc2_sorted = sorted(eoc2_tasks, key=lambda r: r["roi_score"], reverse=True)

    print("\nTop 10 EOC=2 ROI candidates:")
    for r in eoc2_sorted[:10]:
        print(
            f" - task_id={r['task_id']} | "
            f"ROI={r['roi_score']} | "
            f"symmetry_score={r['symmetry_score']}"
        )

    # Convenience handle
    top_eoc2_candidate = eoc2_sorted[0]
    print("\nTop EOC=2 candidate object:")
    print(top_eoc2_candidate)

print("\n=== EOC=2 DIAGNOSTIC COMPLETE ===")

=== EOC=2 REGIME DIAGNOSTIC ===

Total ROI tasks              : 999
EOC=2 & weakly_covered tasks : 20

Symmetry score distribution (EOC=2):
  symmetry_score=1.0: 20

ROI score summary (EOC=2):
  min ROI : 0.788662
  max ROI : 0.788662
  mean ROI: 0.788662

Top 10 EOC=2 ROI candidates:
 - task_id=0d3d703e | ROI=0.788662 | symmetry_score=1.0
 - task_id=21f83797 | ROI=0.788662 | symmetry_score=1.0
 - task_id=253bf280 | ROI=0.788662 | symmetry_score=1.0
 - task_id=3befdf3e | ROI=0.788662 | symmetry_score=1.0
 - task_id=44f52bb0 | ROI=0.788662 | symmetry_score=1.0
 - task_id=48131b3c | ROI=0.788662 | symmetry_score=1.0
 - task_id=496994bd | ROI=0.788662 | symmetry_score=1.0
 - task_id=746b3537 | ROI=0.788662 | symmetry_score=1.0
 - task_id=8ba14f53 | ROI=0.788662 | symmetry_score=1.0
 - task_id=8d5021e8 | ROI=0.788662 | symmetry_score=1.0

Top EOC=2 candidate object:
{'coverage_label': 'weakly_covered', 'density_adjustment': 0.63093, 'effective_object_count': 2, 'potential_budget_gain': 1.2